# QTrust-PPO: Confidence-Aware Finite-Shot Policy Optimization for Variational Quantum Reinforcement Learning

This notebook contains the core implementation and primary experimental workflow for QTrust-PPO. It evaluates finite-shot variational quantum policies under PPO and applies an exact confidence-aware certification rule to the clipped-versus-unclipped surrogate decision.

## Experiment scope

- Environments: `CartPole-v1`, `Acrobot-v1`, `LunarLander-v3`
- Primary VQC: 4 qubits, 3 variational layers, data re-uploading, entanglement, finite-shot measurement
- PPO: clipped surrogate with generalized advantage estimation
- Fixed-shot baselines: 128 and 256 shots in the primary benchmark
- Controlled fixed-shot reliability: 32, 64, 128, 256, 512, and 1024 shots
- QTrust schedule: 32, 64, 128, 256, 512, 1024, 2048, and 4096 shots
- Per-path confidence level: $\delta=0.05$
- Independent training seeds: 10
- Resource accounting: action-selection, parameter-shift gradient, and QTrust certification shots

The statistical unit is the independent training seed. Evaluation episodes associated with the same trained seed are treated as repeated measurements rather than independent replicates.

## Confidence allocation

At each stage of the finite schedule, the confidence budget is divided across four one-sided Clopper-Pearson tails: lower and upper limits for both the old and candidate action probabilities. With $K$ stages, each tail uses $\delta/(4K)$.

In [ ]:
# Internet must be ON in Kaggle.
# This cell installs only dependencies that are actually missing.


import sys
import shutil
import subprocess
import importlib.util


def pip_install(package):
    print(f"Installing missing dependency: {package}")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            package,
        ]
    )


# SWIG is required by Box2D builds on some Kaggle images.
if shutil.which("swig") is None:
    pip_install("swig")
else:
    print("✓ swig available")


# Gymnasium + Box2D
gym_missing = (
    importlib.util.find_spec("gymnasium") is None
)
box2d_missing = (
    importlib.util.find_spec("Box2D") is None
)

if gym_missing or box2d_missing:
    pip_install("gymnasium[box2d]")
else:
    print("✓ gymnasium + Box2D available")


# PennyLane core
if importlib.util.find_spec("pennylane") is None:
    pip_install("pennylane")
else:
    print("✓ PennyLane available")


# PennyLane Lightning GPU plugin.
# Import presence alone does not guarantee lightning.gpu availability,
# so probe the device and install the GPU plugin only if needed.
try:
    import pennylane as _qml_probe
    _probe_dev = _qml_probe.device(
        "lightning.gpu",
        wires=2,
        shots=None,
    )
    del _probe_dev
    print("✓ PennyLane lightning.gpu available")
except Exception:
    pip_install("pennylane-lightning-gpu")


# Remaining scientific stack
for module_name, package_name in [
    ("scipy", "scipy"),
    ("statsmodels", "statsmodels"),
    ("tqdm", "tqdm"),
    ("matplotlib", "matplotlib"),
]:
    if importlib.util.find_spec(module_name) is None:
        pip_install(package_name)
    else:
        print(f"✓ {module_name} available")


print()
print("=" * 78)
print("QTrust-PPO dependency setup : PASS")
print("=" * 78)


In [ ]:
# Imports, Version Check, and Hardware Verification


import os
import random
import platform
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import gymnasium as gym
import pennylane as qml

from tqdm.auto import tqdm

print("=" * 78)
print("QTrust-PPO Environment Verification")
print("=" * 78)

print(f"Python version     : {platform.python_version()}")
print(f"PyTorch version    : {torch.__version__}")
print(f"PennyLane version  : {qml.__version__}")
print(f"Gymnasium version  : {gym.__version__}")

print("-" * 78)

CUDA_AVAILABLE = torch.cuda.is_available()
GPU_COUNT = torch.cuda.device_count()

print(f"CUDA available     : {CUDA_AVAILABLE}")
print(f"Detected GPUs      : {GPU_COUNT}")

if CUDA_AVAILABLE:
    for i in range(GPU_COUNT):
        print(f"GPU {i}             : {torch.cuda.get_device_name(i)}")
    
    DEVICE = torch.device("cuda:0")
else:
    DEVICE = torch.device("cpu")

print(f"Primary device     : {DEVICE}")

print("-" * 78)

# Quick PennyLane sanity check
try:
    test_dev = qml.device("default.qubit", wires=2)

    @qml.qnode(test_dev)
    def test_circuit(x):
        qml.RY(x, wires=0)
        qml.CNOT(wires=[0, 1])
        return qml.expval(qml.PauliZ(0))

    test_value = test_circuit(0.5)

    print(f"PennyLane test     : PASS")
    print(f"Test expectation   : {float(test_value):.6f}")

except Exception as e:
    print(f"PennyLane test     : FAIL")
    print(f"Error              : {e}")

print("=" * 78)

In [ ]:
import os
import random
import numpy as np
import torch

MASTER_SEED = 42

def set_global_seed(seed: int = MASTER_SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

set_global_seed(MASTER_SEED)

DEV_SEEDS = [11, 23, 37, 53, 71]
FINAL_SEEDS = [11, 23, 37, 53, 71, 89, 107, 131, 157, 181]

ENV_CONFIG = {
    "CartPole-v1": {"state_dim": 4, "action_dim": 2},
    "Acrobot-v1": {"state_dim": 6, "action_dim": 3},
    "LunarLander-v3": {"state_dim": 8, "action_dim": 4},
}

QUANTUM_CONFIG = {
    "main_qubits": 4,
    "main_layers": 3,
    "ablation_qubits": [4, 6],
    "ablation_layers": [2, 3, 4],
    "fixed_shots": [32, 64, 128, 256, 512, 1024],
    "adaptive_shot_schedule": [32, 64, 128, 256, 512, 1024, 2048, 4096],
    "min_shots": 32,
    "max_shots": 4096,
    "delta": 0.05,
}


PPO_CONFIG = {
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "clip_epsilon": 0.20,
    "actor_lr": 3e-4,
    "critic_lr": 1e-3,
    "update_epochs": 1,
    "minibatch_size": 64,
    "entropy_coef": 0.01,
    "value_coef": 0.50,
    "max_grad_norm": 0.50,
}

COMPUTE_CONFIG = {
    "primary_device": str(DEVICE),
    "gpu_count": torch.cuda.device_count(),
    "mixed_precision": False,
}

assert QUANTUM_CONFIG["main_qubits"] == 4
assert QUANTUM_CONFIG["adaptive_shot_schedule"][-1] == 4096
assert PPO_CONFIG["update_epochs"] == 1
assert 0 < QUANTUM_CONFIG["delta"] < 1

print("=" * 78)
print("QTrust-PPO FINAL CONFIGURATION")
print("=" * 78)
print("Final statistical seeds :", len(FINAL_SEEDS))
print("Environments            :", list(ENV_CONFIG))
print("Main VQC                : 4 qubits × 3 layers = 24 trainable angles")
print("Adaptive shot schedule  :", QUANTUM_CONFIG["adaptive_shot_schedule"])
print("PPO update epochs       :", PPO_CONFIG["update_epochs"])
print("Primary device          :", DEVICE)
print("=" * 78)


## Environment, trajectory, and GAE foundations

The following components define environment construction, state preprocessing, rollout storage, generalized advantage estimation, and the classical value function used throughout the experiments.

In [ ]:
from dataclasses import dataclass, field
from typing import List
from torch.distributions import Categorical

def make_env(env_name: str, seed: int = MASTER_SEED):
    """
    Create and seed a Gymnasium environment reproducibly.
    """
    if env_name not in ENV_CONFIG:
        raise ValueError(f"Unknown environment: {env_name}")

    env = gym.make(env_name)

    obs, info = env.reset(seed=seed)

    env.action_space.seed(seed)
    env.observation_space.seed(seed)

    return env

def preprocess_state(state, device=DEVICE):
    """
    Convert an environment observation into a finite float32 tensor.

    Step 1: replace possible NaN/Inf values safely
    Step 2: softly bound values using tanh
    Step 3: map bounded features to quantum rotation angles [-pi, pi]

    No state feature is discarded.
    """

    state = np.asarray(state, dtype=np.float32)

    # Safety against unexpected numerical values
    state = np.nan_to_num(
        state,
        nan=0.0,
        posinf=10.0,
        neginf=-10.0
    )

    # Soft normalization to [-1, 1]
    bounded_state = np.tanh(state)

    # Quantum-friendly rotation angles [-pi, pi]
    angle_state = np.pi * bounded_state

    return torch.tensor(
        angle_state,
        dtype=torch.float32,
        device=device
    )

@dataclass
class RolloutBuffer:
    """
    Shared trajectory buffer for all PPO variants.
    """

    states: List[np.ndarray] = field(default_factory=list)
    actions: List[int] = field(default_factory=list)
    log_probs: List[float] = field(default_factory=list)

    rewards: List[float] = field(default_factory=list)

    values: List[float] = field(default_factory=list)
    next_values: List[float] = field(default_factory=list)

    terminateds: List[bool] = field(default_factory=list)
    truncateds: List[bool] = field(default_factory=list)

    # Used later for finite-shot QPPO / QTrust-PPO
    shot_counts: List[int] = field(default_factory=list)

    def add(
        self,
        state,
        action,
        log_prob,
        reward,
        value,
        next_value,
        terminated,
        truncated,
        shots=0,
    ):
        self.states.append(
            np.asarray(state, dtype=np.float32).copy()
        )

        self.actions.append(int(action))
        self.log_probs.append(float(log_prob))
        self.rewards.append(float(reward))

        self.values.append(float(value))
        self.next_values.append(float(next_value))

        self.terminateds.append(bool(terminated))
        self.truncateds.append(bool(truncated))

        self.shot_counts.append(int(shots))

    def clear(self):
        self.states.clear()
        self.actions.clear()
        self.log_probs.clear()
        self.rewards.clear()

        self.values.clear()
        self.next_values.clear()

        self.terminateds.clear()
        self.truncateds.clear()

        self.shot_counts.clear()

    def __len__(self):
        return len(self.rewards)

def compute_gae(
    rewards,
    values,
    next_values,
    terminateds,
    truncateds,
    gamma=PPO_CONFIG["gamma"],
    gae_lambda=PPO_CONFIG["gae_lambda"],
):
    """
    Compute Generalized Advantage Estimation.

    Important Gymnasium distinction:

    terminated=True:
        true terminal state -> no value bootstrap.

    truncated=True:
        time-limit/artificial ending -> bootstrap remains valid,
        but recursive GAE must stop before a reset state.
    """

    rewards = np.asarray(rewards, dtype=np.float64)
    values = np.asarray(values, dtype=np.float64)
    next_values = np.asarray(next_values, dtype=np.float64)

    terminateds = np.asarray(
        terminateds,
        dtype=np.float64
    )

    truncateds = np.asarray(
        truncateds,
        dtype=np.float64
    )

    n_steps = len(rewards)

    if not (
        len(values)
        == len(next_values)
        == len(terminateds)
        == len(truncateds)
        == n_steps
    ):
        raise ValueError(
            "All GAE input arrays must have identical lengths."
        )

    advantages = np.zeros(
        n_steps,
        dtype=np.float64
    )

    gae = 0.0

    for t in reversed(range(n_steps)):

        # True termination removes value bootstrap.
        bootstrap_mask = 1.0 - terminateds[t]

        # Both termination and truncation stop recursive GAE,
        # because the next stored transition may belong to a reset episode.
        episode_end = max(
            terminateds[t],
            truncateds[t]
        )

        recursion_mask = 1.0 - episode_end

        delta = (
            rewards[t]
            + gamma
            * next_values[t]
            * bootstrap_mask
            - values[t]
        )

        gae = (
            delta
            + gamma
            * gae_lambda
            * recursion_mask
            * gae
        )

        advantages[t] = gae

    returns = advantages + values

    return advantages, returns

def normalize_advantages(
    advantages,
    eps=1e-8
):
    """
    Normalize PPO advantages for more stable optimization.
    """

    advantages = np.asarray(
        advantages,
        dtype=np.float32
    )

    if len(advantages) <= 1:
        return advantages

    mean = advantages.mean()
    std = advantages.std()

    return (
        advantages - mean
    ) / (std + eps)

class ClassicalActor(nn.Module):
    """
    Standard MLP policy used as the main classical PPO baseline.
    """

    def __init__(
        self,
        state_dim,
        action_dim,
        hidden_dim=64,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),

            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),

            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, states):
        """
        Returns unnormalized policy logits.
        """
        return self.network(states)

class ClassicalCritic(nn.Module):
    """
    State-value estimator V(s).
    """

    def __init__(
        self,
        state_dim,
        hidden_dim=64,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),

            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),

            nn.Linear(hidden_dim, 1),
        )

    def forward(self, states):
        return self.network(states).squeeze(-1)

class ClassicalActorCritic(nn.Module):
    """
    Combined interface used by PPO training and evaluation.
    """

    def __init__(
        self,
        state_dim,
        action_dim,
        hidden_dim=64,
    ):
        super().__init__()

        self.actor = ClassicalActor(
            state_dim=state_dim,
            action_dim=action_dim,
            hidden_dim=hidden_dim,
        )

        self.critic = ClassicalCritic(
            state_dim=state_dim,
            hidden_dim=hidden_dim,
        )

    @torch.no_grad()
    def act(self, state):
        """
        Sample one action from the current policy while following
        the actual device of the model parameters.
        """

        device = next(
            self.actor.parameters()
        ).device

        state = torch.as_tensor(
            state,
            dtype=torch.float32,
            device=device,
        )

        if state.ndim == 1:
            state = state.unsqueeze(0)

        logits = self.actor(state)
        dist = Categorical(logits=logits)

        action = dist.sample()
        log_prob = dist.log_prob(action)
        value = self.critic(state)

        return (
            int(action.item()),
            float(log_prob.item()),
            float(value.reshape(-1)[0].item()),
        )

    def evaluate_actions(
        self,
        states,
        actions,
    ):
        """
        Evaluate actions already stored in the PPO rollout.
        """

        logits = self.actor(states)

        dist = Categorical(logits=logits)

        log_probs = dist.log_prob(actions)
        entropy = dist.entropy()

        values = self.critic(states)

        return (
            log_probs,
            entropy,
            values,
        )

## QTrust decision logic

The actor update uses a genuine finite-shot PennyLane QNode with parameter-shift gradients. QTrust then draws finite measurement counts for the old and candidate policies and certifies the PPO clipping state relevant to the sign of the advantage. If any rollout sample remains unresolved at the maximum scheduled budget, the actor parameters and actor optimizer state are restored while the critic update is retained.

Analytic Born probabilities are used only as simulator-side hidden truth for drawing finite counts and auditing classification accuracy. They are not supplied to the QTrust decision rule.

PPO clipping is treated as a surrogate-objective mechanism rather than a formal trust-region guarantee.

In [ ]:
import pennylane as qml
import torch
import torch.nn as nn
from torch.distributions import Categorical

# PyTorch tensors stay on CPU while Lightning may execute the
# quantum simulation on GPU internally.
QUANTUM_TORCH_DEVICE = torch.device("cpu")

def probe_quantum_backend(device_name):
    dev = qml.device(device_name, wires=2, shots=None)
    diff_method = (
        "parameter-shift"
        if device_name == "lightning.gpu"
        else "backprop"
    )

    @qml.qnode(dev, interface="torch", diff_method=diff_method)
    def probe_circuit(x, theta):
        qml.RY(x, wires=0)
        qml.RY(theta, wires=1)
        qml.CNOT(wires=[0, 1])
        return qml.probs(wires=[0, 1])

    x = torch.tensor(0.37, dtype=torch.float64)
    theta = torch.tensor(0.21, dtype=torch.float64, requires_grad=True)
    probs = probe_circuit(x, theta)
    loss = -torch.log(probs[0] + 1e-12)
    loss.backward()

    assert torch.isfinite(probs).all()
    assert torch.isfinite(theta.grad)
    assert abs(float(probs.sum()) - 1.0) < 1e-8
    return True

ANALYTIC_QDEVICE_NAME = None

for candidate in ["lightning.gpu", "default.qubit"]:
    try:
        probe_quantum_backend(candidate)
        ANALYTIC_QDEVICE_NAME = candidate
        print("Quantum backend selected :", candidate)
        break
    except Exception as exc:
        print(f"{candidate:<20} unavailable ({type(exc).__name__})")

if ANALYTIC_QDEVICE_NAME is None:
    raise RuntimeError("No compatible PennyLane quantum backend found.")


def build_analytic_quantum_policy(
    input_dim,
    n_qubits=QUANTUM_CONFIG["main_qubits"],
    n_layers=QUANTUM_CONFIG["main_layers"],
):
    """
    VQC policy with balanced |+> initialization.

    Hadamard preparation prevents the initial policy from
    collapsing toward one computational-basis action.
    """

    dev = qml.device(
        ANALYTIC_QDEVICE_NAME,
        wires=n_qubits,
        shots=None,
    )

    diff_method = (
        "parameter-shift"
        if ANALYTIC_QDEVICE_NAME == "lightning.gpu"
        else "backprop"
    )

    @qml.qnode(
        dev,
        interface="torch",
        diff_method=diff_method,
    )
    def quantum_policy(features, weights):

        
        # Balanced quantum initialization
        

        for qubit in range(n_qubits):
            qml.Hadamard(wires=qubit)

        
        # Variational layers + data re-uploading
        

        for layer in range(n_layers):

            
            # State encoding
            

            for qubit in range(n_qubits):

                feature_index = (
                    layer * n_qubits + qubit
                ) % input_dim

                angle = features[feature_index]

                # Moderate encoding scale improves stability
                qml.RY(
                    0.5 * angle,
                    wires=qubit,
                )

                qml.RZ(
                    0.25 * angle,
                    wires=qubit,
                )

            
            # Ring entanglement
            

            for qubit in range(n_qubits - 1):

                qml.CNOT(
                    wires=[
                        qubit,
                        qubit + 1,
                    ]
                )

            qml.CNOT(
                wires=[
                    n_qubits - 1,
                    0,
                ]
            )

            
            # Trainable quantum rotations
            

            for qubit in range(n_qubits):

                qml.RY(
                    weights[
                        layer,
                        qubit,
                        0,
                    ],
                    wires=qubit,
                )

                qml.RZ(
                    weights[
                        layer,
                        qubit,
                        1,
                    ],
                    wires=qubit,
                )

        return qml.probs(
            wires=[0, 1]
        )

    return quantum_policy

def get_action_readout_matrix(
    action_dim,
    dtype=torch.float64,
    device=QUANTUM_TORCH_DEVICE,
):
    """
    Column-stochastic classical post-processing matrix.

    Each quantum basis outcome is mapped probabilistically
    to one environment action.

    For 3 actions:
        |00> -> action 0
        |01> -> action 1
        |10> -> action 2
        |11> -> uniformly randomized across all 3 actions

    This makes uniform quantum basis probabilities map to a
    uniform 3-action policy while preserving physical validity.
    """

    if action_dim == 2:

        matrix = [
            [1.0, 0.0, 1.0, 0.0],
            [0.0, 1.0, 0.0, 1.0],
        ]

    elif action_dim == 3:

        matrix = [
            [1.0, 0.0, 0.0, 1.0 / 3.0],
            [0.0, 1.0, 0.0, 1.0 / 3.0],
            [0.0, 0.0, 1.0, 1.0 / 3.0],
        ]

    elif action_dim == 4:

        matrix = [
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, 1.0],
        ]

    else:
        raise ValueError(
            f"Unsupported action dimension: {action_dim}"
        )

    matrix = torch.tensor(
        matrix,
        dtype=dtype,
        device=device,
    )

    # Every quantum outcome must distribute total mass = 1
    column_sums = matrix.sum(dim=0)

    assert torch.allclose(
        column_sums,
        torch.ones_like(column_sums),
        atol=1e-10,
    )

    return matrix

def balanced_action_probabilities(
    basis_probs,
    action_dim,
):
    """
    Quantum basis probabilities -> action probabilities.
    """

    readout = get_action_readout_matrix(
        action_dim=action_dim,
        dtype=basis_probs.dtype,
        device=basis_probs.device,
    )

    action_probs = (
        readout @ basis_probs
    )

    action_probs = torch.clamp(
        action_probs,
        min=1e-12,
    )

    # Numerical protection
    action_probs = (
        action_probs /
        action_probs.sum()
    )

    return action_probs

class QuantumActor(nn.Module):
    """
    Variational quantum policy actor.

    Quantum parameters:
        n_layers × n_qubits × 2

    Main experiment:
        3 × 4 × 2 = 24 parameters
    """

    def __init__(
        self,
        state_dim,
        action_dim,
        n_qubits=QUANTUM_CONFIG["main_qubits"],
        n_layers=QUANTUM_CONFIG["main_layers"],
    ):
        super().__init__()

        self.state_dim = state_dim
        self.action_dim = action_dim
        self.n_qubits = n_qubits
        self.n_layers = n_layers

        self.qnode = build_analytic_quantum_policy(
            input_dim=state_dim,
            n_qubits=n_qubits,
            n_layers=n_layers,
        )

        initial_weights = (
            0.05
            * torch.randn(
                n_layers,
                n_qubits,
                2,
                dtype=torch.float64,
            )
        )

        self.quantum_weights = nn.Parameter(
            initial_weights
        )

    def encode_state(self, state):
        """
        Raw environment state -> bounded quantum angles.
        """

        if not torch.is_tensor(state):
            state = torch.tensor(
                state,
                dtype=torch.float64,
            )

        state = state.to(
            device=QUANTUM_TORCH_DEVICE,
            dtype=torch.float64,
        )

        # Protect against unexpected numerical extremes
        state = torch.nan_to_num(
            state,
            nan=0.0,
            posinf=10.0,
            neginf=-10.0,
        )

        return (
            torch.pi
            * torch.tanh(state)
        )

    def single_forward(self, state):
        """
        Quantum policy probabilities for one state.
        """

        encoded_state = self.encode_state(
            state
        )

        basis_probs = self.qnode(
            encoded_state,
            self.quantum_weights,
        )

        action_probs = (
            balanced_action_probabilities(
                basis_probs,
                self.action_dim,
            )
        )

        return action_probs

    def forward(self, states):
        """
        Supports either:
            [state_dim]
        or:
            [batch, state_dim]
        """

        if not torch.is_tensor(states):
            states = torch.tensor(
                states,
                dtype=torch.float64,
            )

        if states.ndim == 1:
            return self.single_forward(
                states
            )

        batch_probs = [
            self.single_forward(state)
            for state in states
        ]

        return torch.stack(
            batch_probs,
            dim=0
        )

    @torch.no_grad()
    def act(self, state):
        """
        Sample an action from the analytic VQC policy.
        """

        probs = self.single_forward(state)

        dist = Categorical(
            probs=probs
        )

        action = dist.sample()

        log_prob = dist.log_prob(
            action
        )

        return (
            int(action.item()),
            float(log_prob.item()),
            probs.detach().cpu().numpy(),
        )

    def evaluate_actions(
        self,
        states,
        actions,
    ):
        """
        Evaluate stored actions for PPO optimization.
        """

        probs = self.forward(
            states
        )

        actions = actions.to(
            probs.device,
            dtype=torch.long,
        )

        dist = Categorical(
            probs=probs
        )

        log_probs = dist.log_prob(
            actions
        )

        entropy = dist.entropy()

        return (
            log_probs,
            entropy,
            probs,
        )

class HybridQuantumActorCritic:
    """
    VQC actor + classical critic.

    Actor:
        CPU-facing PyTorch tensors,
        with Lightning-GPU performing quantum simulation.

    Critic:
        Standard PyTorch network on CUDA.
    """

    def __init__(
        self,
        state_dim,
        action_dim,
        critic_hidden_dim=64,
    ):

        self.actor = QuantumActor(
            state_dim=state_dim,
            action_dim=action_dim,
        )

        self.critic = ClassicalCritic(
            state_dim=state_dim,
            hidden_dim=critic_hidden_dim,
        ).to(DEVICE)

    @torch.no_grad()
    def act(self, state):

        action, log_prob, probs = (
            self.actor.act(state)
        )

        critic_state = torch.tensor(
            state,
            dtype=torch.float32,
            device=DEVICE,
        ).unsqueeze(0)

        value = float(
            self.critic(
                critic_state
            ).item()
        )

        return (
            action,
            log_prob,
            value,
            probs,
        )


@torch.no_grad()
def get_analytic_action_probs(actor, state):
    """
    Hidden Born probabilities used only to draw finite-shot
    certification samples and to audit classification truth.
    They are never used directly by the QTrust decision rule.
    """
    probs = actor.single_forward(state)
    return (
        probs.detach()
        .cpu()
        .numpy()
        .astype(np.float64)
    )


In [ ]:
def build_finite_shot_quantum_policy(
    input_dim,
    shots,
    n_qubits=QUANTUM_CONFIG["main_qubits"],
    n_layers=QUANTUM_CONFIG["main_layers"],
):
    """
    Finite-shot VQC using shots at the QNode level.
    """

    dev = qml.device(
        ANALYTIC_QDEVICE_NAME,
        wires=n_qubits,
    )

    @qml.qnode(
        dev,
        interface="torch",
        diff_method="parameter-shift",
        shots=int(shots),
    )
    def finite_qnode(features, weights):

        # Balanced initialization
        for qubit in range(n_qubits):
            qml.Hadamard(wires=qubit)

        # Variational circuit
        for layer in range(n_layers):

            # Data re-uploading
            for qubit in range(n_qubits):

                feature_index = (
                    layer * n_qubits + qubit
                ) % input_dim

                angle = features[feature_index]

                qml.RY(
                    0.5 * angle,
                    wires=qubit,
                )

                qml.RZ(
                    0.25 * angle,
                    wires=qubit,
                )

            # Ring entanglement
            for qubit in range(n_qubits - 1):

                qml.CNOT(
                    wires=[qubit, qubit + 1]
                )

            qml.CNOT(
                wires=[n_qubits - 1, 0]
            )

            # Trainable rotations
            for qubit in range(n_qubits):

                qml.RY(
                    weights[layer, qubit, 0],
                    wires=qubit,
                )

                qml.RZ(
                    weights[layer, qubit, 1],
                    wires=qubit,
                )

        return qml.probs(wires=[0, 1])

    return finite_qnode

class FixedShotQuantumActor(nn.Module):

    def __init__(
        self,
        state_dim,
        action_dim,
        shots,
        n_qubits=QUANTUM_CONFIG["main_qubits"],
        n_layers=QUANTUM_CONFIG["main_layers"],
    ):
        super().__init__()

        self.state_dim = state_dim
        self.action_dim = action_dim
        self.shots = int(shots)

        self.n_qubits = n_qubits
        self.n_layers = n_layers

        self.qnode = build_finite_shot_quantum_policy(
            input_dim=state_dim,
            shots=shots,
            n_qubits=n_qubits,
            n_layers=n_layers,
        )

        self.quantum_weights = nn.Parameter(
            0.05
            * torch.randn(
                n_layers,
                n_qubits,
                2,
                dtype=torch.float64,
            )
        )

    def encode_state(self, state):

        if not torch.is_tensor(state):
            state = torch.tensor(
                state,
                dtype=torch.float64,
            )

        state = state.to(
            QUANTUM_TORCH_DEVICE,
            dtype=torch.float64,
        )

        state = torch.nan_to_num(
            state,
            nan=0.0,
            posinf=10.0,
            neginf=-10.0,
        )

        return torch.pi * torch.tanh(state)

    def single_forward(self, state):

        encoded = self.encode_state(state)

        basis_probs = self.qnode(
            encoded,
            self.quantum_weights,
        )

        return balanced_action_probabilities(
            basis_probs,
            self.action_dim,
        )

    def forward(self, states):

        if not torch.is_tensor(states):
            states = torch.tensor(
                states,
                dtype=torch.float64,
            )

        if states.ndim == 1:
            return self.single_forward(states)

        return torch.stack(
            [
                self.single_forward(state)
                for state in states
            ],
            dim=0,
        )

    @torch.no_grad()
    def act(self, state):

        probs = self.single_forward(state)

        dist = Categorical(probs=probs)

        action = dist.sample()

        log_prob = dist.log_prob(action)

        return (
            int(action.item()),
            float(log_prob.item()),
            probs.detach().cpu().numpy(),
        )

    def evaluate_actions(
        self,
        states,
        actions,
    ):

        probs = self.forward(states)

        actions = actions.to(
            probs.device,
            dtype=torch.long,
        )

        dist = Categorical(probs=probs)

        return (
            dist.log_prob(actions),
            dist.entropy(),
            probs,
        )

class FixedShotHybridActorCritic:

    def __init__(
        self,
        state_dim,
        action_dim,
        shots,
        critic_hidden_dim=64,
    ):

        self.shots = int(shots)

        self.actor = FixedShotQuantumActor(
            state_dim=state_dim,
            action_dim=action_dim,
            shots=shots,
        )

        self.critic = ClassicalCritic(
            state_dim=state_dim,
            hidden_dim=critic_hidden_dim,
        ).to(DEVICE)

    @torch.no_grad()
    def act(self, state):

        action, log_prob, probs = (
            self.actor.act(state)
        )

        critic_state = torch.tensor(
            state,
            dtype=torch.float32,
            device=DEVICE,
        ).unsqueeze(0)

        value = float(
            self.critic(
                critic_state
            ).item()
        )

        return (
            action,
            log_prob,
            value,
            probs,
        )

def quantum_ppo_update(
    model,
    buffer,
    actor_optimizer,
    critic_optimizer,
    update_epochs=None,
):
    """
    PPO optimization for:

        VQC quantum actor
        +
        classical neural critic

    The actor and critic are optimized separately because
    they live on different computational devices.
    """

    if len(buffer) < 2:
        raise ValueError(
            "Quantum PPO update requires at least 2 transitions."
        )

    if update_epochs is None:
        update_epochs = PPO_CONFIG["update_epochs"]

    
    # GAE + returns
    

    advantages, returns = compute_gae(
        rewards=buffer.rewards,
        values=buffer.values,
        next_values=buffer.next_values,
        terminateds=buffer.terminateds,
        truncateds=buffer.truncateds,
        gamma=PPO_CONFIG["gamma"],
        gae_lambda=PPO_CONFIG["gae_lambda"],
    )

    advantages = normalize_advantages(
        advantages
    )

    
    # Actor tensors: CPU float64
    

    actor_states = torch.tensor(
        np.asarray(buffer.states),
        dtype=torch.float64,
        device=QUANTUM_TORCH_DEVICE,
    )

    actions_cpu = torch.tensor(
        buffer.actions,
        dtype=torch.long,
        device=QUANTUM_TORCH_DEVICE,
    )

    old_log_probs_cpu = torch.tensor(
        buffer.log_probs,
        dtype=torch.float64,
        device=QUANTUM_TORCH_DEVICE,
    )

    advantages_cpu = torch.tensor(
        advantages,
        dtype=torch.float64,
        device=QUANTUM_TORCH_DEVICE,
    )

    
    # Critic tensors: GPU float32
    

    critic_states = torch.tensor(
        np.asarray(buffer.states),
        dtype=torch.float32,
        device=DEVICE,
    )

    returns_gpu = torch.tensor(
        returns,
        dtype=torch.float32,
        device=DEVICE,
    )

    n_samples = len(buffer)

    batch_size = min(
        PPO_CONFIG["minibatch_size"],
        n_samples,
    )

    
    # Diagnostics
    

    actor_losses = []
    critic_losses = []
    entropies = []
    approx_kls = []
    clip_fractions = []
    actor_grad_norms = []
    critic_grad_norms = []

    
    # PPO epochs
    

    for epoch in range(update_epochs):

        permutation = torch.randperm(
            n_samples
        )

        for start in range(
            0,
            n_samples,
            batch_size
        ):

            indices = permutation[
                start:start + batch_size
            ]

            
            # Quantum Actor Update
            

            mb_actor_states = actor_states[
                indices
            ]

            mb_actions = actions_cpu[
                indices
            ]

            mb_old_log_probs = old_log_probs_cpu[
                indices
            ]

            mb_advantages = advantages_cpu[
                indices
            ]

            (
                new_log_probs,
                entropy,
                _
            ) = model.actor.evaluate_actions(
                mb_actor_states,
                mb_actions,
            )

            log_ratio = (
                new_log_probs
                - mb_old_log_probs
            )

            ratio = torch.exp(
                log_ratio
            )

            unclipped_objective = (
                ratio
                * mb_advantages
            )

            clipped_ratio = torch.clamp(
                ratio,
                1.0 - PPO_CONFIG[
                    "clip_epsilon"
                ],
                1.0 + PPO_CONFIG[
                    "clip_epsilon"
                ],
            )

            clipped_objective = (
                clipped_ratio
                * mb_advantages
            )

            policy_loss = -torch.min(
                unclipped_objective,
                clipped_objective,
            ).mean()

            entropy_mean = (
                entropy.mean()
            )

            actor_loss = (
                policy_loss
                - PPO_CONFIG[
                    "entropy_coef"
                ]
                * entropy_mean
            )

            actor_optimizer.zero_grad()

            actor_loss.backward()

            actor_grad_norm = (
                torch.nn.utils.clip_grad_norm_(
                    model.actor.parameters(),
                    PPO_CONFIG[
                        "max_grad_norm"
                    ],
                )
            )

            actor_optimizer.step()

            
            # Classical Critic Update
            

            gpu_indices = indices.to(
                DEVICE
            )

            mb_critic_states = (
                critic_states[
                    gpu_indices
                ]
            )

            mb_returns = (
                returns_gpu[
                    gpu_indices
                ]
            )

            predicted_values = (
                model.critic(
                    mb_critic_states
                )
            )

            critic_loss = F.mse_loss(
                predicted_values,
                mb_returns,
            )

            critic_optimizer.zero_grad()

            critic_loss.backward()

            critic_grad_norm = (
                torch.nn.utils.clip_grad_norm_(
                    model.critic.parameters(),
                    PPO_CONFIG[
                        "max_grad_norm"
                    ],
                )
            )

            critic_optimizer.step()

            
            # Diagnostics
            

            with torch.no_grad():

                approx_kl = (
                    (ratio - 1.0)
                    - log_ratio
                ).mean()

                clip_fraction = (
                    (
                        torch.abs(
                            ratio - 1.0
                        )
                        > PPO_CONFIG[
                            "clip_epsilon"
                        ]
                    )
                    .double()
                    .mean()
                )

            actor_losses.append(
                float(
                    actor_loss.detach()
                )
            )

            critic_losses.append(
                float(
                    critic_loss.detach()
                )
            )

            entropies.append(
                float(
                    entropy_mean.detach()
                )
            )

            approx_kls.append(
                float(
                    approx_kl.detach()
                )
            )

            clip_fractions.append(
                float(
                    clip_fraction.detach()
                )
            )

            actor_grad_norms.append(
                float(
                    actor_grad_norm.detach()
                )
            )

            critic_grad_norms.append(
                float(
                    critic_grad_norm.detach()
                )
            )

    
    # Summary
    

    metrics = {

        "actor_loss":
            float(np.mean(actor_losses)),

        "critic_loss":
            float(np.mean(critic_losses)),

        "entropy":
            float(np.mean(entropies)),

        "approx_kl":
            float(np.mean(approx_kls)),

        "clip_fraction":
            float(np.mean(clip_fractions)),

        "actor_grad_norm":
            float(np.mean(actor_grad_norms)),

        "critic_grad_norm":
            float(np.mean(critic_grad_norms)),

        "rollout_size":
            int(n_samples),
    }

    return metrics

In [ ]:
# QTrust-PPO EXACT FINITE-HORIZON CONTROLLER


import copy
from scipy.stats import beta

QTRUST_SHOT_SCHEDULE = [
    32, 64, 128, 256, 512, 1024, 2048, 4096
]

def cp_lower_one_sided(k, n, alpha):
    if k == 0:
        return 0.0
    return float(beta.ppf(alpha, k, n - k + 1))

def cp_upper_one_sided(k, n, alpha):
    if k == n:
        return 1.0
    return float(beta.ppf(1.0 - alpha, k + 1, n - k))

def exact_ratio_bounds(
    new_successes,
    old_successes,
    trials,
    stage_alpha,
    eps=1e-12,
):
    """
    Simultaneous lower/upper confidence bounds for p_new/p_old.

    The per-stage familywise error budget is stage_alpha.
    Four one-sided binomial tails are used:
      new lower, new upper, old lower, old upper.
    Therefore each tail receives stage_alpha/4.

    Across K scheduled stages, stage_alpha = delta/K, so a
    direct Bonferroni union bound controls the full finite
    horizon at no more than delta.
    """
    tail_alpha = stage_alpha / 4.0

    new_low = cp_lower_one_sided(
        new_successes, trials, tail_alpha
    )
    new_high = cp_upper_one_sided(
        new_successes, trials, tail_alpha
    )
    old_low = cp_lower_one_sided(
        old_successes, trials, tail_alpha
    )
    old_high = cp_upper_one_sided(
        old_successes, trials, tail_alpha
    )

    ratio_low = new_low / max(old_high, eps)
    ratio_high = (
        np.inf
        if old_low <= eps
        else new_high / old_low
    )

    return float(ratio_low), float(ratio_high)


def qtrust_finite_horizon_test(
    p_old,
    p_new,
    action,
    advantage,
    seed,
    shot_schedule=QTRUST_SHOT_SCHEDULE,
    delta=QUANTUM_CONFIG["delta"],
    clip_epsilon=PPO_CONFIG["clip_epsilon"],
):
    """
    Advantage-aware finite-shot certification of the PPO
    clipping decision for one stored state-action pair.

    Positive advantage: only the upper clipping boundary
    1 + epsilon matters.
    Negative advantage: only the lower clipping boundary
    1 - epsilon matters.

    The controller stops at the first shot stage whose exact
    ratio confidence interval lies wholly on one side of the
    relevant boundary. Otherwise it abstains (UNCERTAIN) at
    the finite maximum budget.
    """
    p_old_a = float(p_old[action])
    p_new_a = float(p_new[action])

    lower_clip = 1.0 - clip_epsilon
    upper_clip = 1.0 + clip_epsilon

    rng_old = np.random.default_rng(seed)
    rng_new = np.random.default_rng(seed + 1_000_000)

    old_successes = 0
    new_successes = 0
    previous_shots = 0
    trace = []
    final_decision = "UNCERTAIN"

    K = len(shot_schedule)
    stage_alpha = delta / K

    for stage, target_shots in enumerate(shot_schedule, start=1):
        additional_shots = target_shots - previous_shots
        if additional_shots <= 0:
            raise ValueError("Shot schedule must be strictly increasing.")

        old_successes += int(
            rng_old.binomial(additional_shots, p_old_a)
        )
        new_successes += int(
            rng_new.binomial(additional_shots, p_new_a)
        )

        ratio_low, ratio_high = exact_ratio_bounds(
            new_successes=new_successes,
            old_successes=old_successes,
            trials=target_shots,
            stage_alpha=stage_alpha,
        )

        old_hat = old_successes / target_shots
        new_hat = new_successes / target_shots
        ratio_hat = new_hat / max(old_hat, 1e-12)

        if advantage >= 0:
            if ratio_high <= upper_clip:
                final_decision = "SAFE_NO_UPPER_CLIP"
            elif ratio_low > upper_clip:
                final_decision = "CONFIRMED_UPPER_CLIP"
            else:
                final_decision = "UNCERTAIN"
        else:
            if ratio_low >= lower_clip:
                final_decision = "SAFE_NO_LOWER_CLIP"
            elif ratio_high < lower_clip:
                final_decision = "CONFIRMED_LOWER_CLIP"
            else:
                final_decision = "UNCERTAIN"

        trace.append(
            {
                "Stage": stage,
                "Shots / Policy": target_shots,
                "Ratio-hat": ratio_hat,
                "Ratio Low": ratio_low,
                "Ratio High": ratio_high,
                "Stage Alpha": stage_alpha,
                "Tail Alpha": stage_alpha / 4.0,
                "Decision": final_decision,
            }
        )

        previous_shots = target_shots

        if final_decision != "UNCERTAIN":
            break

    true_ratio = p_new_a / max(p_old_a, 1e-12)

    return (
        pd.DataFrame(trace),
        {
            "true_ratio": float(true_ratio),
            "decision": final_decision,
            "resolved": final_decision != "UNCERTAIN",
            "shots_per_policy": int(previous_shots),
            "total_probability_shots": int(2 * previous_shots),
            "stages_used": len(trace),
        },
    )


def get_buffer_advantages(buffer):

    advantages, returns = compute_gae(
        rewards=buffer.rewards,
        values=buffer.values,
        next_values=buffer.next_values,
        terminateds=buffer.terminateds,
        truncateds=buffer.truncateds,
        gamma=PPO_CONFIG["gamma"],
        gae_lambda=PPO_CONFIG["gae_lambda"],
    )

    advantages = normalize_advantages(
        advantages
    )

    return (
        advantages,
        returns,
    )

def make_analytic_shadow(actor):
    """
    Reconstruct an analytic QuantumActor with exactly the same
    VQC parameters as a finite-shot actor.

    This is NOT used for optimization.
    """

    shadow = QuantumActor(
        state_dim=actor.state_dim,
        action_dim=actor.action_dim,
        n_qubits=actor.n_qubits,
        n_layers=actor.n_layers,
    )

    with torch.no_grad():

        shadow.quantum_weights.copy_(
            actor.quantum_weights.detach()
        )

    for p in shadow.parameters():
        p.requires_grad_(False)

    return shadow

def certify_qtrust_update(
    old_actor,
    new_actor,
    buffer,
    advantages,
    seed,
    shot_schedule=QTRUST_SHOT_SCHEDULE,
):

    records = []

    lower_clip = (
        1.0
        - PPO_CONFIG["clip_epsilon"]
    )

    upper_clip = (
        1.0
        + PPO_CONFIG["clip_epsilon"]
    )

    for i in range(len(buffer)):

        state = buffer.states[i]
        action = buffer.actions[i]

        advantage = float(
            advantages[i]
        )

        
        # Hidden Born probabilities:
        # used ONLY for finite-measurement simulation and
        # ground-truth evaluation.
        

        p_old = get_analytic_action_probs(
            old_actor,
            state,
        )

        p_new = get_analytic_action_probs(
            new_actor,
            state,
        )

        
        
        

        _, summary = (
            qtrust_finite_horizon_test(
                p_old=p_old,
                p_new=p_new,
                action=action,
                advantage=advantage,
                seed=(
                    seed
                    + i * 1009
                ),
                shot_schedule=shot_schedule,
                delta=QUANTUM_CONFIG[
                    "delta"
                ],
                clip_epsilon=PPO_CONFIG[
                    "clip_epsilon"
                ],
            )
        )

        decision = summary[
            "decision"
        ]

        true_ratio = float(
            summary[
                "true_ratio"
            ]
        )

        
        # True PPO clipping state
        # Evaluation only
        

        if advantage >= 0:

            true_should_clip = (
                true_ratio
                > upper_clip
            )

        else:

            true_should_clip = (
                true_ratio
                < lower_clip
            )

        if decision in {
            "CONFIRMED_UPPER_CLIP",
            "CONFIRMED_LOWER_CLIP",
        }:

            predicted_clip = True

        elif decision in {
            "SAFE_NO_UPPER_CLIP",
            "SAFE_NO_LOWER_CLIP",
        }:

            predicted_clip = False

        else:

            predicted_clip = None

        if predicted_clip is None:

            classification_correct = np.nan

        else:

            classification_correct = (
                predicted_clip
                == true_should_clip
            )

        records.append(
            {
                "sample":
                    i,

                "advantage":
                    advantage,

                "true_ratio":
                    true_ratio,

                "decision":
                    decision,

                "resolved":
                    summary[
                        "resolved"
                    ],

                "shots_per_policy":
                    summary[
                        "shots_per_policy"
                    ],

                "total_shots":
                    summary[
                        "total_probability_shots"
                    ],

                "true_should_clip":
                    true_should_clip,

                "classification_correct":
                    classification_correct,
            }
        )

    
    # Aggregate certification statistics
    

    certification_df = pd.DataFrame(
        records
    )

    unresolved_fraction = float(
        1.0
        - certification_df[
            "resolved"
        ].mean()
    )

    total_shots = int(
        certification_df[
            "total_shots"
        ].sum()
    )

    mean_shots = float(
        certification_df[
            "total_shots"
        ].mean()
    )

    median_shots = float(
        certification_df[
            "total_shots"
        ].median()
    )

    resolved_rows = (
        certification_df[
            certification_df[
                "classification_correct"
            ].notna()
        ]
    )

    if len(resolved_rows) > 0:

        classification_accuracy = float(
            resolved_rows[
                "classification_correct"
            ].mean()
        )

    else:

        classification_accuracy = np.nan

    return (
        certification_df,

        {
            "unresolved_fraction":
                unresolved_fraction,

            "total_qtrust_shots":
                total_shots,

            "mean_qtrust_shots":
                mean_shots,

            "median_qtrust_shots":
                median_shots,

            "classification_accuracy":
                classification_accuracy,
        }
    )

In [ ]:
# Quantum training and evaluation


import copy
import time
import pennylane as qml


def train_fixed_qppo_exact_steps(
    env_name,
    shots,
    total_steps,
    rollout_steps,
    seed=MASTER_SEED,
    update_epochs=1,
):

    assert total_steps % rollout_steps == 0

    set_global_seed(seed)

    env = make_env(
        env_name,
        seed=seed,
    )

    state_dim = ENV_CONFIG[
        env_name
    ]["state_dim"]

    action_dim = ENV_CONFIG[
        env_name
    ]["action_dim"]

    model = FixedShotHybridActorCritic(
        state_dim=state_dim,
        action_dim=action_dim,
        shots=shots,
    )

    actor_optimizer = torch.optim.Adam(
        model.actor.parameters(),
        lr=PPO_CONFIG["actor_lr"],
    )

    critic_optimizer = torch.optim.Adam(
        model.critic.parameters(),
        lr=PPO_CONFIG["critic_lr"],
    )

    buffer = RolloutBuffer()

    global_step = 0
    update_number = 0
    episode_number = 1

    cumulative_action_shots = 0
    cumulative_gradient_shots = 0

    update_records = []

    state, _ = env.reset(
        seed=seed + episode_number
    )

    state = np.asarray(
        state,
        dtype=np.float32,
    )

    start_time = time.perf_counter()

    
    # Exact-step interaction
    

    while global_step < total_steps:

        (
            action,
            log_prob,
            value,
            _
        ) = model.act(state)

        cumulative_action_shots += shots

        (
            next_state,
            reward,
            terminated,
            truncated,
            _
        ) = env.step(action)

        next_state = np.asarray(
            next_state,
            dtype=np.float32,
        )

        if terminated:

            next_value = 0.0

        else:

            next_tensor = torch.tensor(
                next_state,
                dtype=torch.float32,
                device=DEVICE,
            ).unsqueeze(0)

            with torch.no_grad():

                next_value = float(
                    model.critic(
                        next_tensor
                    ).item()
                )

        buffer.add(
            state=state,
            action=action,
            log_prob=log_prob,
            reward=reward,
            value=value,
            next_value=next_value,
            terminated=terminated,
            truncated=truncated,
            shots=shots,
        )

        global_step += 1

        
        # Exact rollout update
        

        if len(buffer) == rollout_steps:

            update_number += 1

            tracker = qml.Tracker(
                model.actor.qnode.device
            )

            with tracker:

                metrics = quantum_ppo_update(
                    model=model,
                    buffer=buffer,
                    actor_optimizer=actor_optimizer,
                    critic_optimizer=critic_optimizer,
                    update_epochs=update_epochs,
                )

            gradient_shots = int(
                tracker.totals.get(
                    "shots",
                    0
                )
            )

            gradient_executions = int(
                tracker.totals.get(
                    "executions",
                    0
                )
            )

            cumulative_gradient_shots += (
                gradient_shots
            )

            metrics.update(
                {
                    "update":
                        update_number,

                    "global_step":
                        global_step,

                    "action_shots":
                        cumulative_action_shots,

                    "gradient_shots_update":
                        gradient_shots,

                    "gradient_executions":
                        gradient_executions,

                    "cumulative_gradient_shots":
                        cumulative_gradient_shots,

                    "cumulative_total_shots":
                        (
                            cumulative_action_shots
                            + cumulative_gradient_shots
                        ),
                }
            )

            update_records.append(
                metrics
            )

            buffer.clear()

        
        # Episode transition
        

        if terminated or truncated:

            episode_number += 1

            state, _ = env.reset(
                seed=seed + episode_number
            )

            state = np.asarray(
                state,
                dtype=np.float32,
            )

        else:

            state = next_state

    elapsed = (
        time.perf_counter()
        - start_time
    )

    env.close()

    return (
        model,
        pd.DataFrame(update_records),
        {
            "method":
                f"Fixed-QPPO-{shots}",

            "environment_steps":
                global_step,

            "ppo_updates":
                update_number,

            "action_shots":
                cumulative_action_shots,

            "gradient_shots":
                cumulative_gradient_shots,

            "certification_shots":
                0,

            "total_quantum_shots":
                (
                    cumulative_action_shots
                    + cumulative_gradient_shots
                ),

            "elapsed_seconds":
                elapsed,
        }
    )

def train_qtrust_exact_steps(
    env_name,
    base_shots,
    total_steps,
    rollout_steps,
    seed=MASTER_SEED,
    update_epochs=1,
):

    assert total_steps % rollout_steps == 0

    set_global_seed(seed)

    env = make_env(
        env_name,
        seed=seed,
    )

    state_dim = ENV_CONFIG[
        env_name
    ]["state_dim"]

    action_dim = ENV_CONFIG[
        env_name
    ]["action_dim"]

    model = FixedShotHybridActorCritic(
        state_dim=state_dim,
        action_dim=action_dim,
        shots=base_shots,
    )

    actor_optimizer = torch.optim.Adam(
        model.actor.parameters(),
        lr=PPO_CONFIG["actor_lr"],
    )

    critic_optimizer = torch.optim.Adam(
        model.critic.parameters(),
        lr=PPO_CONFIG["critic_lr"],
    )

    buffer = RolloutBuffer()

    global_step = 0
    update_number = 0
    episode_number = 1

    accepted_updates = 0
    rolled_back_updates = 0

    cumulative_action_shots = 0
    cumulative_gradient_shots = 0
    cumulative_qtrust_shots = 0

    update_records = []

    state, _ = env.reset(
        seed=seed + episode_number
    )

    state = np.asarray(
        state,
        dtype=np.float32,
    )

    start_time = time.perf_counter()

    
    # Exact-step interaction
    

    while global_step < total_steps:

        (
            action,
            log_prob,
            value,
            _
        ) = model.act(state)

        cumulative_action_shots += (
            base_shots
        )

        (
            next_state,
            reward,
            terminated,
            truncated,
            _
        ) = env.step(action)

        next_state = np.asarray(
            next_state,
            dtype=np.float32,
        )

        if terminated:

            next_value = 0.0

        else:

            next_tensor = torch.tensor(
                next_state,
                dtype=torch.float32,
                device=DEVICE,
            ).unsqueeze(0)

            with torch.no_grad():

                next_value = float(
                    model.critic(
                        next_tensor
                    ).item()
                )

        buffer.add(
            state=state,
            action=action,
            log_prob=log_prob,
            reward=reward,
            value=value,
            next_value=next_value,
            terminated=terminated,
            truncated=truncated,
            shots=base_shots,
        )

        global_step += 1

        
        # Exact rollout update
        

        if len(buffer) == rollout_steps:

            update_number += 1

            
            # OLD policy snapshot
            

            old_shadow = make_analytic_shadow(
                model.actor
            )

            old_weights = (
                model.actor
                .quantum_weights
                .detach()
                .clone()
            )

            old_optimizer_state = copy.deepcopy(
                actor_optimizer.state_dict()
            )

            advantages, _ = (
                get_buffer_advantages(
                    buffer
                )
            )

            
            # Finite-shot parameter-shift PPO
            

            tracker = qml.Tracker(
                model.actor.qnode.device
            )

            with tracker:

                metrics = quantum_ppo_update(
                    model=model,
                    buffer=buffer,
                    actor_optimizer=actor_optimizer,
                    critic_optimizer=critic_optimizer,
                    update_epochs=update_epochs,
                )

            gradient_shots = int(
                tracker.totals.get(
                    "shots",
                    0
                )
            )

            gradient_executions = int(
                tracker.totals.get(
                    "executions",
                    0
                )
            )

            cumulative_gradient_shots += (
                gradient_shots
            )

            
            # Proposed NEW policy
            

            new_shadow = make_analytic_shadow(
                model.actor
            )

            
            # Adaptive QTrust certification
            

            (
                certification_df,
                trust_metrics,
            ) = certify_qtrust_update(
                old_actor=old_shadow,
                new_actor=new_shadow,
                buffer=buffer,
                advantages=advantages,
                seed=(
                    seed
                    + update_number
                    * 100_000
                ),
                shot_schedule=
                    QTRUST_SHOT_SCHEDULE,
            )

            qtrust_shots = int(
                trust_metrics[
                    "total_qtrust_shots"
                ]
            )

            cumulative_qtrust_shots += (
                qtrust_shots
            )

            
            # QTrust gate
            

            if (
                trust_metrics[
                    "unresolved_fraction"
                ]
                == 0.0
            ):

                accepted = True
                accepted_updates += 1

            else:

                accepted = False
                rolled_back_updates += 1

                with torch.no_grad():

                    model.actor.quantum_weights.copy_(
                        old_weights
                    )

                actor_optimizer.load_state_dict(
                    old_optimizer_state
                )

            metrics.update(
                {
                    "update":
                        update_number,

                    "global_step":
                        global_step,

                    "update_accepted":
                        accepted,

                    "unresolved_fraction":
                        trust_metrics[
                            "unresolved_fraction"
                        ],

                    "classification_accuracy":
                        trust_metrics[
                            "classification_accuracy"
                        ],

                    "mean_qtrust_shots":
                        trust_metrics[
                            "mean_qtrust_shots"
                        ],

                    "action_shots":
                        cumulative_action_shots,

                    "gradient_shots_update":
                        gradient_shots,

                    "gradient_executions":
                        gradient_executions,

                    "qtrust_shots_update":
                        qtrust_shots,

                    "cumulative_gradient_shots":
                        cumulative_gradient_shots,

                    "cumulative_qtrust_shots":
                        cumulative_qtrust_shots,

                    "cumulative_total_shots":
                        (
                            cumulative_action_shots
                            + cumulative_gradient_shots
                            + cumulative_qtrust_shots
                        ),
                }
            )

            update_records.append(
                metrics
            )

            buffer.clear()

        
        # Episode transition
        

        if terminated or truncated:

            episode_number += 1

            state, _ = env.reset(
                seed=seed + episode_number
            )

            state = np.asarray(
                state,
                dtype=np.float32,
            )

        else:

            state = next_state

    elapsed = (
        time.perf_counter()
        - start_time
    )

    env.close()

    return (
        model,
        pd.DataFrame(update_records),
        {
            "method":
                f"QTrust-PPO-{base_shots}",

            "environment_steps":
                global_step,

            "ppo_updates":
                update_number,

            "accepted_updates":
                accepted_updates,

            "rolled_back_updates":
                rolled_back_updates,

            "action_shots":
                cumulative_action_shots,

            "gradient_shots":
                cumulative_gradient_shots,

            "certification_shots":
                cumulative_qtrust_shots,

            "total_quantum_shots":
                (
                    cumulative_action_shots
                    + cumulative_gradient_shots
                    + cumulative_qtrust_shots
                ),

            "elapsed_seconds":
                elapsed,
        }
    )

def evaluate_finite_shot_policy(
    model,
    env_name,
    seeds,
    deterministic=True,
):

    actor = model.actor

    if not hasattr(actor, "shots"):
        raise ValueError(
            "Evaluation requires a finite-shot actor."
        )

    actor_shots = int(
        actor.shots
    )

    records = []

    total_shots = 0

    start_time = time.perf_counter()

    for episode_idx, eval_seed in enumerate(
        seeds,
        start=1,
    ):

        env = make_env(
            env_name,
            seed=eval_seed,
        )

        state, _ = env.reset(
            seed=eval_seed
        )

        state = np.asarray(
            state,
            dtype=np.float32,
        )

        terminated = False
        truncated = False

        reward_sum = 0.0
        steps = 0

        while not (
            terminated or truncated
        ):

            
            # Genuine finite-shot quantum measurement
            

            with torch.no_grad():

                probs = actor.single_forward(
                    state
                )

            total_shots += actor_shots

            
            # Evaluation action
            

            if deterministic:

                action = int(
                    torch.argmax(
                        probs
                    ).item()
                )

            else:

                action = int(
                    torch.distributions
                    .Categorical(
                        probs=probs
                    )
                    .sample()
                    .item()
                )

            (
                next_state,
                reward,
                terminated,
                truncated,
                _
            ) = env.step(action)

            state = np.asarray(
                next_state,
                dtype=np.float32,
            )

            reward_sum += float(
                reward
            )

            steps += 1

        env.close()

        records.append(
            {
                "episode":
                    episode_idx,

                "seed":
                    eval_seed,

                "reward":
                    reward_sum,

                "steps":
                    steps,

                "actor_shots":
                    steps * actor_shots,
            }
        )

    elapsed = (
        time.perf_counter()
        - start_time
    )

    df = pd.DataFrame(
        records
    )

    rewards = df[
        "reward"
    ].to_numpy()

    n = len(rewards)

    mean_reward = float(
        np.mean(rewards)
    )

    std_reward = float(
        np.std(
            rewards,
            ddof=1
        )
        if n > 1
        else 0.0
    )

    sem = (
        std_reward / np.sqrt(n)
        if n > 1
        else 0.0
    )

    summary = {
        "episodes":
            n,

        "mean_reward":
            mean_reward,

        "std_reward":
            std_reward,

        "median_reward":
            float(
                np.median(
                    rewards
                )
            ),

        "min_reward":
            float(
                np.min(
                    rewards
                )
            ),

        "max_reward":
            float(
                np.max(
                    rewards
                )
            ),

        "ci95_low":
            mean_reward
            - 1.96 * sem,

        "ci95_high":
            mean_reward
            + 1.96 * sem,

        "mean_steps":
            float(
                df[
                    "steps"
                ].mean()
            ),

        "total_evaluation_shots":
            int(total_shots),

        "elapsed_seconds":
            elapsed,

        "deterministic":
            deterministic,
    }

    return df, summary



# CPU-Safe Classical PPO Baseline


@torch.no_grad()
def classical_act_safe(model, state):
    device = next(model.actor.parameters()).device
    state_tensor = torch.as_tensor(
        state, dtype=torch.float32, device=device
    )
    if state_tensor.ndim == 1:
        state_tensor = state_tensor.unsqueeze(0)

    logits = model.actor(state_tensor)
    dist = Categorical(logits=logits)
    action = dist.sample()
    log_prob = dist.log_prob(action)
    value = model.critic(state_tensor)

    return (
        int(action.item()),
        float(log_prob.item()),
        float(value.reshape(-1)[0].item()),
    )


def classical_ppo_update_final(
    model,
    buffer,
    actor_optimizer,
    critic_optimizer,
    update_epochs=1,
):
    assert update_epochs == 1

    device = next(model.actor.parameters()).device

    advantages, returns = compute_gae(
        rewards=buffer.rewards,
        values=buffer.values,
        next_values=buffer.next_values,
        terminateds=buffer.terminateds,
        truncateds=buffer.truncateds,
        gamma=PPO_CONFIG["gamma"],
        gae_lambda=PPO_CONFIG["gae_lambda"],
    )
    advantages = normalize_advantages(advantages)

    states = torch.as_tensor(
        np.asarray(buffer.states, dtype=np.float32),
        dtype=torch.float32,
        device=device,
    )
    actions = torch.as_tensor(
        buffer.actions, dtype=torch.long, device=device
    )
    old_log_probs = torch.as_tensor(
        buffer.log_probs, dtype=torch.float32, device=device
    )
    advantages_t = torch.as_tensor(
        advantages, dtype=torch.float32, device=device
    )
    returns_t = torch.as_tensor(
        returns, dtype=torch.float32, device=device
    )

    n_samples = len(buffer)
    batch_size = min(PPO_CONFIG["minibatch_size"], n_samples)

    actor_losses = []
    critic_losses = []
    entropies = []
    approx_kls = []
    clip_fractions = []
    actor_grad_norms = []
    critic_grad_norms = []

    for _ in range(update_epochs):
        permutation = torch.randperm(n_samples, device=device)

        for start in range(0, n_samples, batch_size):
            idx = permutation[start:start + batch_size]

            logits = model.actor(states[idx])
            dist = Categorical(logits=logits)
            new_log_probs = dist.log_prob(actions[idx])
            entropy = dist.entropy().mean()
            values = model.critic(states[idx]).reshape(-1)

            log_ratio = new_log_probs - old_log_probs[idx]
            ratio = torch.exp(log_ratio)

            unclipped = ratio * advantages_t[idx]
            clipped_ratio = torch.clamp(
                ratio,
                1.0 - PPO_CONFIG["clip_epsilon"],
                1.0 + PPO_CONFIG["clip_epsilon"],
            )
            clipped = clipped_ratio * advantages_t[idx]

            actor_loss = -torch.min(unclipped, clipped).mean()
            critic_loss = F.mse_loss(values, returns_t[idx])

            total_loss = (
                actor_loss
                + PPO_CONFIG["value_coef"] * critic_loss
                - PPO_CONFIG["entropy_coef"] * entropy
            )

            actor_optimizer.zero_grad(set_to_none=True)
            critic_optimizer.zero_grad(set_to_none=True)
            total_loss.backward()

            actor_grad_norm = torch.nn.utils.clip_grad_norm_(
                model.actor.parameters(),
                PPO_CONFIG["max_grad_norm"],
            )
            critic_grad_norm = torch.nn.utils.clip_grad_norm_(
                model.critic.parameters(),
                PPO_CONFIG["max_grad_norm"],
            )

            actor_optimizer.step()
            critic_optimizer.step()

            with torch.no_grad():
                approx_kl = (
                    ratio - 1.0 - log_ratio
                ).mean()
                clip_fraction = (
                    (
                        torch.abs(ratio - 1.0)
                        > PPO_CONFIG["clip_epsilon"]
                    )
                    .float()
                    .mean()
                )

            actor_losses.append(float(actor_loss.detach().cpu()))
            critic_losses.append(float(critic_loss.detach().cpu()))
            entropies.append(float(entropy.detach().cpu()))
            approx_kls.append(float(approx_kl.detach().cpu()))
            clip_fractions.append(float(clip_fraction.detach().cpu()))
            actor_grad_norms.append(float(actor_grad_norm.detach().cpu()))
            critic_grad_norms.append(float(critic_grad_norm.detach().cpu()))

    return {
        "actor_loss": float(np.mean(actor_losses)),
        "critic_loss": float(np.mean(critic_losses)),
        "entropy": float(np.mean(entropies)),
        "approx_kl": float(np.mean(approx_kls)),
        "clip_fraction": float(np.mean(clip_fractions)),
        "actor_grad_norm": float(np.mean(actor_grad_norms)),
        "critic_grad_norm": float(np.mean(critic_grad_norms)),
        "rollout_size": int(n_samples),
    }


def train_classical_ppo_exact_steps_final(
    env_name,
    total_steps,
    rollout_steps,
    seed,
    update_epochs=1,
):
    assert total_steps % rollout_steps == 0
    assert update_epochs == 1

    set_global_seed(seed)
    classical_device = torch.device("cpu")

    env = make_env(env_name, seed=seed)
    state_dim = ENV_CONFIG[env_name]["state_dim"]
    action_dim = ENV_CONFIG[env_name]["action_dim"]

    model = ClassicalActorCritic(
        state_dim=state_dim,
        action_dim=action_dim,
    ).to(classical_device)

    actor_optimizer = torch.optim.Adam(
        model.actor.parameters(),
        lr=PPO_CONFIG["actor_lr"],
    )
    critic_optimizer = torch.optim.Adam(
        model.critic.parameters(),
        lr=PPO_CONFIG["critic_lr"],
    )

    buffer = RolloutBuffer()
    update_records = []

    global_step = 0
    update_number = 0
    episode_number = 1

    state, _ = env.reset(seed=seed + episode_number)
    state = np.asarray(state, dtype=np.float32)

    start_time = time.perf_counter()

    while global_step < total_steps:
        action, log_prob, value = classical_act_safe(model, state)

        next_state, reward, terminated, truncated, _ = env.step(action)
        next_state = np.asarray(next_state, dtype=np.float32)

        if terminated:
            next_value = 0.0
        else:
            next_tensor = torch.as_tensor(
                next_state,
                dtype=torch.float32,
                device=classical_device,
            ).unsqueeze(0)
            with torch.no_grad():
                next_value = float(
                    model.critic(next_tensor).reshape(-1)[0].item()
                )

        buffer.add(
            state=state,
            action=action,
            log_prob=log_prob,
            reward=reward,
            value=value,
            next_value=next_value,
            terminated=terminated,
            truncated=truncated,
            shots=0,
        )

        global_step += 1

        if len(buffer) == rollout_steps:
            update_number += 1
            metrics = classical_ppo_update_final(
                model=model,
                buffer=buffer,
                actor_optimizer=actor_optimizer,
                critic_optimizer=critic_optimizer,
                update_epochs=1,
            )
            metrics.update(
                {"update": update_number, "global_step": global_step}
            )
            update_records.append(metrics)
            buffer.clear()

        if terminated or truncated:
            episode_number += 1
            state, _ = env.reset(seed=seed + episode_number)
            state = np.asarray(state, dtype=np.float32)
        else:
            state = next_state

    elapsed = time.perf_counter() - start_time
    env.close()

    assert update_number == total_steps // rollout_steps

    summary = {
        "environment_steps": global_step,
        "ppo_updates": update_number,
        "total_quantum_shots": 0,
        "elapsed_seconds": elapsed,
    }

    return model, pd.DataFrame(update_records), summary


def evaluate_classical_final(model, env_name, seeds):
    device = next(model.actor.parameters()).device
    records = []
    start_time = time.perf_counter()

    for episode_idx, eval_seed in enumerate(seeds, start=1):
        env = make_env(env_name, seed=eval_seed)
        state, _ = env.reset(seed=eval_seed)
        state = np.asarray(state, dtype=np.float32)

        terminated = False
        truncated = False
        reward_sum = 0.0
        steps = 0

        while not (terminated or truncated):
            state_tensor = torch.as_tensor(
                state, dtype=torch.float32, device=device
            ).unsqueeze(0)

            with torch.no_grad():
                logits = model.actor(state_tensor)
                action = int(torch.argmax(logits, dim=-1).item())

            state, reward, terminated, truncated, _ = env.step(action)
            state = np.asarray(state, dtype=np.float32)
            reward_sum += float(reward)
            steps += 1

        env.close()

        records.append(
            {
                "episode": episode_idx,
                "seed": eval_seed,
                "reward": reward_sum,
                "steps": steps,
            }
        )

    elapsed = time.perf_counter() - start_time
    df = pd.DataFrame(records)
    rewards = df["reward"].to_numpy(dtype=np.float64)
    n = len(rewards)

    mean_reward = float(np.mean(rewards))
    std_reward = float(np.std(rewards, ddof=1)) if n > 1 else 0.0
    sem = std_reward / np.sqrt(n) if n > 1 else 0.0

    return df, {
        "evaluation_episodes": n,
        "mean_eval_reward": mean_reward,
        "std_eval_reward": std_reward,
        "median_eval_reward": float(np.median(rewards)),
        "min_eval_reward": float(np.min(rewards)),
        "max_eval_reward": float(np.max(rewards)),
        "eval_ci95_low": float(mean_reward - 1.96 * sem),
        "eval_ci95_high": float(mean_reward + 1.96 * sem),
        "mean_eval_steps": float(df["steps"].mean()),
        "evaluation_quantum_shots": 0,
        "evaluation_runtime_seconds": elapsed,
    }


In [ ]:
print("=" * 78)
print("FINAL PRE-FLIGHT VALIDATION")
print("=" * 78)

# 1) Classical PPO interface/device safety
(
    _classical_model,
    _classical_updates,
    _classical_summary,
) = train_classical_ppo_exact_steps_final(
    env_name="CartPole-v1",
    total_steps=32,
    rollout_steps=32,
    seed=99991,
    update_epochs=1,
)

assert _classical_summary["environment_steps"] == 32
assert _classical_summary["ppo_updates"] == 1
assert next(
    _classical_model.actor.parameters()
).device.type == "cpu"

print("Classical PPO                 : PASS")

# 2) Finite-shot quantum actor
set_global_seed(99992)

_quantum_model = FixedShotHybridActorCritic(
    state_dim=4,
    action_dim=2,
    shots=32,
)

_test_state = np.zeros(
    4,
    dtype=np.float32,
)

with torch.no_grad():
    _test_probs = (
        _quantum_model.actor
        .single_forward(
            _test_state
        )
    )

assert _test_probs.shape == (2,)
assert torch.isfinite(
    _test_probs
).all()

assert abs(
    float(
        _test_probs.sum()
    )
    - 1.0
) < 1e-8

print("Finite-shot VQC              : PASS")

# 3) Exact finite-horizon QTrust controller
_trace, _summary = (
    qtrust_finite_horizon_test(
        p_old=np.array(
            [0.5, 0.5]
        ),
        p_new=np.array(
            [0.5, 0.5]
        ),
        action=0,
        advantage=+1.0,
        seed=99993,
    )
)

assert (
    _summary[
        "total_probability_shots"
    ]
    <= 2 * max(
        QTRUST_SHOT_SCHEDULE
    )
)

assert np.isclose(
    _trace["Tail Alpha"].iloc[0],
    QUANTUM_CONFIG["delta"]
    / len(QTRUST_SHOT_SCHEDULE)
    / 4.0,
)

print("QTrust confidence controller : PASS")

del _classical_model
del _classical_updates
del _classical_summary
del _quantum_model
del _test_probs
del _trace
del _summary

print("=" * 78)
print("FINAL PRE-FLIGHT : PASS")
print("=" * 78)


## Controlled mechanism validation

This benchmark isolates finite-shot clipping decisions near the PPO ratio boundaries. Resolved accuracy is reported together with unresolved rate and coverage so that abstention is kept separate from misclassification.

In [ ]:
# Controlled PPO-boundary reliability benchmark



# finite-shot uncertainty in the PPO clipping decision.

# It does not replace RL evaluation. It is a controlled
# measurement-reliability experiment around r=0.8 and r=1.2.


RATIO_GRID = [
    0.70, 0.75, 0.78, 0.79, 0.80, 0.81, 0.82, 0.85, 0.90,
    1.00, 1.10, 1.15, 1.18, 1.19, 1.20, 1.21, 1.22, 1.25, 1.30,
]
P_OLD_VALUES = [0.30, 0.50, 0.70]
FIXED_SHOTS_STRESS = [32, 64, 128, 256, 512, 1024]
STRESS_REPEATS = 200

def fixed_shot_clip_decision(
    p_old,
    p_new,
    action,
    advantage,
    shots,
    rng,
):
    old_successes = rng.binomial(
        shots, float(p_old[action])
    )
    new_successes = rng.binomial(
        shots, float(p_new[action])
    )

    # Jeffreys-style continuity correction only for the
    # fixed-shot plug-in comparator.
    old_hat = (old_successes + 0.5) / (shots + 1.0)
    new_hat = (new_successes + 0.5) / (shots + 1.0)
    ratio_hat = new_hat / old_hat

    lower_clip = 1.0 - PPO_CONFIG["clip_epsilon"]
    upper_clip = 1.0 + PPO_CONFIG["clip_epsilon"]

    predicted_clip = (
        ratio_hat > upper_clip
        if advantage >= 0
        else ratio_hat < lower_clip
    )

    return bool(predicted_clip), float(ratio_hat)


def run_boundary_reliability_benchmark(
    repeats=STRESS_REPEATS,
    save_path="qtrust_boundary_reliability.csv",
):
    cases = []

    for p_old_a in P_OLD_VALUES:
        for ratio in RATIO_GRID:
            p_new_a = p_old_a * ratio

            if not (0.0 < p_new_a < 1.0):
                continue

            p_old = np.array([p_old_a, 1.0 - p_old_a])
            p_new = np.array([p_new_a, 1.0 - p_new_a])

            cases.append(
                {
                    "p_old": p_old,
                    "p_new": p_new,
                    "action": 0,
                    "advantage": +1.0,
                    "true_ratio": ratio,
                    "true_clip": ratio > 1.2,
                }
            )

            cases.append(
                {
                    "p_old": p_old,
                    "p_new": p_new,
                    "action": 0,
                    "advantage": -1.0,
                    "true_ratio": ratio,
                    "true_clip": ratio < 0.8,
                }
            )

    records = []

    for shots in FIXED_SHOTS_STRESS:
        total = 0
        correct = 0
        false_positive = 0
        false_negative = 0

        for case_idx, case in enumerate(cases):
            for repeat in range(repeats):
                rng = np.random.default_rng(
                    MASTER_SEED
                    + shots * 1_000_000
                    + case_idx * 10_000
                    + repeat
                )

                prediction, _ = fixed_shot_clip_decision(
                    p_old=case["p_old"],
                    p_new=case["p_new"],
                    action=case["action"],
                    advantage=case["advantage"],
                    shots=shots,
                    rng=rng,
                )

                truth = case["true_clip"]
                total += 1
                correct += int(prediction == truth)
                false_positive += int(prediction and not truth)
                false_negative += int((not prediction) and truth)

        records.append(
            {
                "Method": f"Fixed-{shots}",
                "Mean Shots / Ratio": 2 * shots,
                "Accuracy": correct / total,
                "False Positive Rate": false_positive / total,
                "False Negative Rate": false_negative / total,
                "Unresolved Rate": 0.0,
                "Resolved Coverage": 1.0,
                "Resolved Accuracy": correct / total,
            }
        )

    q_total = 0
    q_resolved = 0
    q_correct = 0
    q_fp = 0
    q_fn = 0
    q_unresolved = 0
    q_shots = []

    for case_idx, case in enumerate(cases):
        for repeat in range(repeats):
            _, summary = qtrust_finite_horizon_test(
                p_old=case["p_old"],
                p_new=case["p_new"],
                action=case["action"],
                advantage=case["advantage"],
                seed=(
                    MASTER_SEED
                    + 50_000_000
                    + case_idx * 10_000
                    + repeat
                ),
                shot_schedule=QTRUST_SHOT_SCHEDULE,
                delta=QUANTUM_CONFIG["delta"],
                clip_epsilon=PPO_CONFIG["clip_epsilon"],
            )

            q_total += 1
            q_shots.append(summary["total_probability_shots"])

            decision = summary["decision"]

            if decision == "UNCERTAIN":
                q_unresolved += 1
                continue

            q_resolved += 1
            prediction = decision in {
                "CONFIRMED_UPPER_CLIP",
                "CONFIRMED_LOWER_CLIP",
            }
            truth = case["true_clip"]

            q_correct += int(prediction == truth)
            q_fp += int(prediction and not truth)
            q_fn += int((not prediction) and truth)

    records.append(
        {
            "Method": "QTrust",
            "Mean Shots / Ratio": float(np.mean(q_shots)),
            # Overall accuracy treats abstentions as not classified.
            "Accuracy": q_correct / q_total,
            "False Positive Rate": q_fp / q_total,
            "False Negative Rate": q_fn / q_total,
            "Unresolved Rate": q_unresolved / q_total,
            "Resolved Coverage": q_resolved / q_total,
            "Resolved Accuracy": (
                q_correct / q_resolved
                if q_resolved > 0
                else np.nan
            ),
        }
    )

    df = pd.DataFrame(records)
    df.to_csv(save_path, index=False)

    print("=" * 78)
    print("PPO-boundary finite-shot reliability benchmark")
    print("=" * 78)
    display(df)
    print(
        "Important: QTrust resolved accuracy must always be reported "
        "together with unresolved rate / resolved coverage."
    )
    print("=" * 78)

    return df


BOUNDARY_RELIABILITY_RESULTS = run_boundary_reliability_benchmark()


## Execution and recovery

The primary benchmark uses two independent CUDA worker processes, one per Tesla T4 GPU. Each completed run is written atomically, completed runs are skipped on resume, and integrity checks require the full 120-run design with ten seeds in every environment-by-method condition before aggregate statistics are produced.

In [ ]:
# IMPORTANT: the protocol hash deliberately excludes the seed
# list, so PRIMARY_5 runs can be reused when extending to the
# exact same FINAL_10 experiment.


import json
import hashlib
import pandas as pd

RUN_SCOPE = "FINAL_10"    # "PRIMARY_5" or "FINAL_10"
EXECUTION_GPU_WORKERS = 2  # execution-only; one isolated process per T4

FINAL_ENVIRONMENTS = {
    "CartPole-v1": {
        "training_steps": 2048,
        "rollout_steps": 32,
        "evaluation_episodes": 30,
    },
    "Acrobot-v1": {
        "training_steps": 4096,
        "rollout_steps": 32,
        "evaluation_episodes": 30,
    },
    "LunarLander-v3": {
        "training_steps": 4096,
        "rollout_steps": 32,
        "evaluation_episodes": 30,
    },
}

FINAL_METHODS = [
    "Classical-PPO",
    "Fixed-QPPO-128",
    "Fixed-QPPO-256",
    "QTrust-PPO-128",
]

ACTIVE_SEEDS = (
    DEV_SEEDS
    if RUN_SCOPE == "PRIMARY_5"
    else FINAL_SEEDS
)

def make_final_eval_seeds(training_seed, n_episodes):
    base = 100_000 + int(training_seed) * 100
    return list(range(base, base + int(n_episodes)))

ALGORITHM_PROTOCOL = {
    "paper": "QTrust-PPO",
    "environments": FINAL_ENVIRONMENTS,
    "methods": FINAL_METHODS,
    "ppo": PPO_CONFIG,
    "quantum": {
        "qubits": QUANTUM_CONFIG["main_qubits"],
        "layers": QUANTUM_CONFIG["main_layers"],
        "fixed_main_shots": [128, 256],
        "qtrust_base_shots": 128,
        "qtrust_schedule": QTRUST_SHOT_SCHEDULE,
        "delta": QUANTUM_CONFIG["delta"],
        "confidence_rule": (
            "finite-horizon Bonferroni with four one-sided "
            "Clopper-Pearson tails per stage"
        ),
        "decision_rule": "advantage-aware PPO clipping",
        "rollback_on_unresolved": True,
    },
    "evaluation": {
        "episodes_per_training_seed": 30,
        "deterministic_action_rule": (
            "argmax of finite-shot estimated action probabilities"
        ),
    },
    "resource_accounting": [
        "action-selection shots",
        "parameter-shift gradient shots",
        "QTrust certification shots",
    ],
}

PROTOCOL_HASH = hashlib.sha256(
    json.dumps(
        ALGORITHM_PROTOCOL,
        sort_keys=True,
        default=str,
    ).encode("utf-8")
).hexdigest()[:16]

PROTOCOL_BUNDLE = {
    **ALGORITHM_PROTOCOL,
    "run_scope": RUN_SCOPE,
    "active_seeds": ACTIVE_SEEDS,
    "all_final_seeds": FINAL_SEEDS,
    "protocol_hash": PROTOCOL_HASH,
}

manifest_rows = []

for env_name, env_cfg in FINAL_ENVIRONMENTS.items():
    for method in FINAL_METHODS:
        for seed in ACTIVE_SEEDS:
            manifest_rows.append(
                {
                    "environment": env_name,
                    "method": method,
                    "seed": int(seed),
                    "training_steps": int(env_cfg["training_steps"]),
                    "rollout_steps": int(env_cfg["rollout_steps"]),
                    "evaluation_episodes": int(
                        env_cfg["evaluation_episodes"]
                    ),
                    "update_epochs": 1,
                    "protocol_hash": PROTOCOL_HASH,
                    "status": "PENDING",
                }
            )

FINAL_EXPERIMENT_MANIFEST = pd.DataFrame(manifest_rows)

print("=" * 78)
print("QTrust-PPO FROZEN PUBLICATION PROTOCOL")
print("=" * 78)
print("Run scope               :", RUN_SCOPE)
print("Training seeds          :", ACTIVE_SEEDS)
print("Main runs               :", len(FINAL_EXPERIMENT_MANIFEST))
print("Execution GPU workers   :", EXECUTION_GPU_WORKERS)
print("Protocol hash           :", PROTOCOL_HASH)
print("QTrust schedule         :", QTRUST_SHOT_SCHEDULE)
print(
    "Per-tail stage alpha    :",
    QUANTUM_CONFIG["delta"]
    / len(QTRUST_SHOT_SCHEDULE)
    / 4.0,
)
print("=" * 78)


In [ ]:
# Science Is Unchanged.
# This cell only changes execution architecture:
#   - two independent OS workers
#   - one physical T4 per worker

#   - atomic per-run completion markers
#   - automatic fresh-process restart if a worker crashes
#   - checkpoint ZIP refreshed after every completed run
#   - explicit live checkpoint dashboard + restart status files


# exactly the same. No environment, seed, horizon, shot budget,
# VQC architecture, optimizer, PPO rule, or QTrust rule is cut.


import os
import re
import sys
import json
import time
import shutil
import zipfile
import platform
import subprocess
from pathlib import Path


# Execution-only settings


DUAL_GPU_WORKERS = 2
MAX_RESTART_ROUNDS = 3
CHECKPOINT_EVERY_COMPLETIONS = 1
CHECKPOINT_VIEW_EVERY_COMPLETIONS = 1
MONITOR_SECONDS = 15

WORK_ROOT = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path.cwd()
)

RESULT_ROOT = WORK_ROOT / "qtrust_final_results"
SUMMARY_DIR = RESULT_ROOT / "summaries"
UPDATE_DIR = RESULT_ROOT / "updates"
EVAL_DIR = RESULT_ROOT / "evaluations"
MODEL_DIR = RESULT_ROOT / "models"
ERROR_DIR = RESULT_ROOT / "errors"
LOG_DIR = RESULT_ROOT / "worker_logs"

for directory in [
    RESULT_ROOT,
    SUMMARY_DIR,
    UPDATE_DIR,
    EVAL_DIR,
    MODEL_DIR,
    ERROR_DIR,
    LOG_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


def safe_run_id(environment, method, seed):
    return (
        f"{environment}__{method}__seed{int(seed)}"
        .replace("/", "_")
    )


def load_completed_summaries():
    rows = []

    for filepath in sorted(
        SUMMARY_DIR.glob("*.json")
    ):
        try:
            with open(filepath, "r") as f:
                row = json.load(f)

            if (
                row.get("status") == "COMPLETE"
                and row.get("protocol_hash") == PROTOCOL_HASH
            ):
                rows.append(row)

        except Exception:
            continue

    return (
        pd.DataFrame(rows)
        if rows
        else pd.DataFrame()
    )


def checkpoint_count(path):
    """
    Read the completed-run count encoded in a numbered checkpoint name.
    """
    match = re.search(
        r"qtrust_checkpoint_(\d+)_of_(\d+)\.zip$",
        path.name,
    )
    return int(match.group(1)) if match else -1


def checkpoint_protocol_hash(path):
    """
    Read the frozen protocol hash stored inside a portable checkpoint.

    A checkpoint from a different scientific protocol is NEVER restored.
    """
    try:
        with zipfile.ZipFile(path, "r") as zf:
            candidates = [
                name
                for name in zf.namelist()
                if name.endswith(
                    "qtrust_final_results/frozen_protocol.json"
                )
            ]

            if not candidates:
                return None

            with zf.open(candidates[0]) as f:
                payload = json.loads(
                    f.read().decode("utf-8")
                )

            return payload.get("protocol_hash")

    except Exception:
        return None


def checkpoint_archive_count(path):
    """
    Determine how many COMPLETE runs are stored in a checkpoint.

    Numbered filenames are used first. For qtrust_checkpoint_latest.zip,
    the aggregate CSV or summary files inside the archive are inspected.
    """
    numeric = checkpoint_count(path)

    if numeric >= 0:
        return numeric

    try:
        with zipfile.ZipFile(path, "r") as zf:
            aggregate = [
                name
                for name in zf.namelist()
                if name.endswith(
                    "qtrust_final_results/all_completed_runs.csv"
                )
            ]

            if aggregate:
                with zf.open(aggregate[0]) as f:
                    frame = pd.read_csv(f)
                return int(len(frame))

            summary_files = [
                name
                for name in zf.namelist()
                if (
                    "/summaries/" in name
                    and name.endswith(".json")
                )
            ]

            return int(len(summary_files))

    except Exception:
        return -1


def find_portable_checkpoints():
    """
    Search both the current working directory and attached Kaggle inputs.

    On a fresh Kaggle session, attach the latest checkpoint ZIP as a
    notebook input. This function will discover it automatically.
    """
    candidates = []

    kaggle_input = Path("/kaggle/input")

    if kaggle_input.exists():
        candidates.extend(
            kaggle_input.glob(
                "**/qtrust_checkpoint_*.zip"
            )
        )

    candidates.extend(
        WORK_ROOT.glob(
            "qtrust_checkpoint_*.zip"
        )
    )

    unique = {}

    for path in candidates:
        try:
            unique[str(path.resolve())] = path
        except Exception:
            unique[str(path)] = path

    return list(unique.values())


def restore_latest_checkpoint_if_needed():
    """
    Restore the MOST COMPLETE compatible checkpoint.

    Resume semantics:
      * every fully completed (environment, method, seed) run is preserved;
      * completed runs are never intentionally rerun;
      * if interruption occurs inside one run, that single incomplete run
        restarts from its deterministic seed on the next launch.

    This is intentionally run-level checkpointing. Mid-trajectory restart
    is avoided because restoring simulator RNG, environment state, model,
    optimizer, finite-shot measurement RNG, and rollout state halfway
    through a run could compromise reproducibility.
    """
    local_count = len(
        load_completed_summaries()
    )

    candidates = find_portable_checkpoints()

    compatible = []

    for path in candidates:
        protocol_hash = checkpoint_protocol_hash(path)
        archive_count = checkpoint_archive_count(path)

        if (
            protocol_hash == PROTOCOL_HASH
            and archive_count >= 0
        ):
            compatible.append(
                (
                    archive_count,
                    path,
                )
            )

    if not compatible:
        print(
            "Portable checkpoint restore : "
            f"none compatible; local runs={local_count}"
        )
        return None

    best_count, best = max(
        compatible,
        key=lambda item: (
            item[0],
            str(item[1]),
        ),
    )

    if best_count <= local_count:
        print(
            "Portable checkpoint restore : "
            f"local state is already newest "
            f"({local_count}/{len(FINAL_EXPERIMENT_MANIFEST)})"
        )
        return None

    print(
        "Portable checkpoint candidate :",
        best,
    )
    print(
        "Checkpoint completed runs      :",
        f"{best_count}/{len(FINAL_EXPERIMENT_MANIFEST)}",
    )

    with zipfile.ZipFile(
        best,
        "r",
    ) as zf:
        zf.extractall(
            WORK_ROOT
        )

    restored_count = len(
        load_completed_summaries()
    )

    if restored_count < best_count:
        raise RuntimeError(
            "Checkpoint extraction did not restore the expected "
            f"number of runs: expected at least {best_count}, "
            f"found {restored_count}."
        )

    print(
        "Portable checkpoint restored  :",
        f"{restored_count}/{len(FINAL_EXPERIMENT_MANIFEST)} complete",
    )

    return best

def persist_aggregate_files():
    completed = load_completed_summaries()

    completed_ids = (
        set(
            completed["run_id"]
        )
        if len(completed) > 0
        else set()
    )

    manifest_out = (
        FINAL_EXPERIMENT_MANIFEST.copy()
    )

    manifest_out["run_id"] = (
        manifest_out.apply(
            lambda row: safe_run_id(
                row["environment"],
                row["method"],
                row["seed"],
            ),
            axis=1,
        )
    )

    manifest_out["status"] = (
        manifest_out[
            "run_id"
        ].map(
            lambda run_id:
                "COMPLETE"
                if run_id in completed_ids
                else "PENDING"
        )
    )

    manifest_out.to_csv(
        RESULT_ROOT
        / "experiment_manifest.csv",
        index=False,
    )

    if len(completed) > 0:
        (
            completed
            .sort_values(
                [
                    "environment",
                    "method",
                    "seed",
                ]
            )
            .reset_index(
                drop=True
            )
            .to_csv(
                RESULT_ROOT
                / "all_completed_runs.csv",
                index=False,
            )
        )

    with open(
        RESULT_ROOT
        / "frozen_protocol.json",
        "w",
    ) as f:
        json.dump(
            PROTOCOL_BUNDLE,
            f,
            indent=2,
            default=str,
        )

    metadata = {
        "protocol_hash":
            PROTOCOL_HASH,

        "execution_architecture":
            "two independent CUDA workers; one T4 per process",

        "python":
            platform.python_version(),

        "torch":
            torch.__version__,

        "pennylane":
            qml.__version__,

        "gymnasium":
            gym.__version__,

        "parent_cuda_available":
            torch.cuda.is_available(),

        "parent_gpu_count":
            torch.cuda.device_count(),

        "parent_gpu_names":
            [
                torch.cuda.get_device_name(i)
                for i in range(
                    torch.cuda.device_count()
                )
            ],

        "quantum_backend":
            ANALYTIC_QDEVICE_NAME,
    }

    with open(
        RESULT_ROOT
        / "environment_metadata.json",
        "w",
    ) as f:
        json.dump(
            metadata,
            f,
            indent=2,
        )



def checkpoint_progress_frame():
    """
    Build an explicit environment × method checkpoint view.
    """
    completed = load_completed_summaries()

    completed_ids = (
        set(completed["run_id"])
        if len(completed) > 0
        else set()
    )

    manifest = FINAL_EXPERIMENT_MANIFEST.copy()

    manifest["run_id"] = manifest.apply(
        lambda row: safe_run_id(
            row["environment"],
            row["method"],
            row["seed"],
        ),
        axis=1,
    )

    manifest["is_complete"] = (
        manifest["run_id"].isin(
            completed_ids
        )
    )

    view = (
        manifest
        .groupby(
            [
                "environment",
                "method",
            ],
            sort=False,
        )
        .agg(
            Completed=(
                "is_complete",
                "sum",
            ),
            Total=(
                "run_id",
                "size",
            ),
        )
        .reset_index()
    )

    view["Completed"] = (
        view["Completed"]
        .astype(int)
    )

    view["Total"] = (
        view["Total"]
        .astype(int)
    )

    view["Pending"] = (
        view["Total"]
        - view["Completed"]
    )

    view["Percent"] = (
        100.0
        * view["Completed"]
        / view["Total"]
    ).round(1)

    return view


def write_checkpoint_status_files():
    """
    Write machine-readable checkpoint progress files.
    """
    completed = load_completed_summaries()

    completed_count = int(
        len(completed)
    )

    total_count = int(
        len(
            FINAL_EXPERIMENT_MANIFEST
        )
    )

    pending_count = int(
        total_count
        - completed_count
    )

    percent = (
        100.0
        * completed_count
        / max(total_count, 1)
    )

    progress = checkpoint_progress_frame()

    status = {
        "protocol_hash":
            PROTOCOL_HASH,

        "completed_runs":
            completed_count,

        "total_runs":
            total_count,

        "pending_runs":
            pending_count,

        "percent_complete":
            float(percent),

        "resume_semantics":
            (
                "resume from last fully completed run; "
                "an interrupted in-progress run restarts from its seed"
            ),
    }

    with open(
        RESULT_ROOT
        / "checkpoint_status.json",
        "w",
    ) as f:
        json.dump(
            status,
            f,
            indent=2,
        )

    progress.to_csv(
        RESULT_ROOT
        / "checkpoint_progress.csv",
        index=False,
    )

    return (
        status,
        progress,
    )


def show_checkpoint_view(
    detailed=True,
):
    """
    Human-readable checkpoint dashboard.

    This can also be called manually at any time while the parent
    notebook process is alive.
    """
    status, progress = (
        write_checkpoint_status_files()
    )

    latest_zip = (
        WORK_ROOT
        / "qtrust_checkpoint_latest.zip"
    )

    latest_text = (
        latest_zip.name
        if latest_zip.exists()
        else "not created yet"
    )

    latest_size_mb = (
        latest_zip.stat().st_size
        / (1024 ** 2)
        if latest_zip.exists()
        else 0.0
    )

    print()
    print(
        "=" * 78
    )
    print(
        "QTRUST CHECKPOINT VIEW"
    )
    print(
        "=" * 78
    )
    print(
        "Completed :",
        f"{status['completed_runs']} / {status['total_runs']}",
    )
    print(
        "Pending   :",
        status[
            "pending_runs"
        ],
    )
    print(
        "Progress  :",
        f"{status['percent_complete']:.1f}%",
    )
    print(
        "Latest ZIP:",
        latest_text,
        (
            f"({latest_size_mb:.2f} MB)"
            if latest_zip.exists()
            else ""
        ),
    )
    print(
        "Resume    : last fully completed run",
    )

    if detailed:
        print(
            "-" * 78
        )

        try:
            from IPython.display import display
            display(progress)
        except Exception:
            print(
                progress.to_string(
                    index=False
                )
            )

    print(
        "=" * 78
    )

    return status, progress

def create_portable_checkpoint():
    """
    Build a checkpoint from COMPLETE runs only.

    Active/incomplete run traces are intentionally excluded.
    This avoids packaging a half-written run while both GPU
    workers are executing concurrently.
    """

    persist_aggregate_files()

    # Refresh explicit checkpoint status/progress files before zipping.
    write_checkpoint_status_files()

    completed = (
        load_completed_summaries()
    )

    completed_count = len(
        completed
    )

    total_count = len(
        FINAL_EXPERIMENT_MANIFEST
    )

    numbered_path = (
        WORK_ROOT
        / (
            f"qtrust_checkpoint_"
            f"{completed_count:03d}"
            f"_of_{total_count:03d}.zip"
        )
    )

    latest_path = (
        WORK_ROOT
        / "qtrust_checkpoint_latest.zip"
    )

    temp_path = (
        WORK_ROOT
        / "_qtrust_checkpoint_tmp.zip"
    )

    with zipfile.ZipFile(
        temp_path,
        "w",
        compression=
            zipfile.ZIP_DEFLATED,
    ) as zf:

        # Include only atomically completed scientific runs.
        if len(completed) > 0:

            for _, row in completed.iterrows():

                run_id = row[
                    "run_id"
                ]

                for directory, suffix in [
                    (
                        SUMMARY_DIR,
                        ".json",
                    ),
                    (
                        UPDATE_DIR,
                        ".csv",
                    ),
                    (
                        EVAL_DIR,
                        ".csv",
                    ),
                ]:

                    file = (
                        directory
                        / f"{run_id}{suffix}"
                    )

                    if file.exists():
                        zf.write(
                            file,
                            arcname=str(
                                file.relative_to(
                                    WORK_ROOT
                                )
                            ),
                        )

        # Protocol and aggregate metadata.
        for filename in [
            "experiment_manifest.csv",
            "all_completed_runs.csv",
            "frozen_protocol.json",
            "environment_metadata.json",
            "checkpoint_status.json",
            "checkpoint_progress.csv",
        ]:

            file = (
                RESULT_ROOT
                / filename
            )

            if file.exists():
                zf.write(
                    file,
                    arcname=str(
                        file.relative_to(
                            WORK_ROOT
                        )
                    ),
                )

        # Mechanism-validation benchmark.
        boundary_file = (
            WORK_ROOT
            / "qtrust_boundary_reliability.csv"
        )

        if boundary_file.exists():
            zf.write(
                boundary_file,
                arcname=
                    boundary_file.name,
            )

    # Atomic replacement of the rolling latest checkpoint.
    os.replace(
        temp_path,
        latest_path,
    )

    shutil.copy2(
        latest_path,
        numbered_path,
    )

    # Prevent checkpoint copies from consuming the session disk.
    numbered = sorted(
        WORK_ROOT.glob(
            "qtrust_checkpoint_*_of_*.zip"
        ),
        key=lambda p:
            checkpoint_count(p),
    )

    for old in numbered[:-2]:
        try:
            old.unlink()
        except Exception:
            pass

    return (
        numbered_path,
        latest_path,
    )


# Restore the most complete compatible checkpoint first.
RESTORED_CHECKPOINT = restore_latest_checkpoint_if_needed()

# Explicit starting dashboard.
show_checkpoint_view(detailed=True)



# Hardware check


print(
    "=" * 78
)
print(
    "DUAL-T4 EXECUTION PRE-FLIGHT"
)
print(
    "=" * 78
)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. "
        "Select Kaggle T4 x2 before Run All."
    )

if torch.cuda.device_count() < 2:
    raise RuntimeError(
        "This notebook requires two visible CUDA GPUs. "
        f"Detected {torch.cuda.device_count()}."
    )

if (
    ANALYTIC_QDEVICE_NAME
    != "lightning.gpu"
):
    raise RuntimeError(
        "PennyLane lightning.gpu is not active. "
        "Stopping before the publication sweep."
    )

for gpu_id in range(2):
    print(
        f"Physical GPU {gpu_id} : "
        f"{torch.cuda.get_device_name(gpu_id)}"
    )

print(
    "Scientific protocol hash:",
    PROTOCOL_HASH,
)
print(
    "=" * 78
)



# Worker process


WORKER_SOURCE = '\n# Auto-generated dual-T4 worker for QTrust-PPO.\n# Execution architecture only; scientific protocol is unchanged.\n\nimport os\nimport sys\nimport json\nfrom pathlib import Path\n\n# ===== SOURCE CELL 2 =====\n\n# ============================================================\n# CELL 3/28\n# Imports, Version Check, and Hardware Verification\n# ============================================================\n\nimport os\nimport random\nimport platform\nimport numpy as np\nimport pandas as pd\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.optim as optim\n\nimport gymnasium as gym\nimport pennylane as qml\n\nfrom tqdm.auto import tqdm\n\nprint("=" * 78)\nprint("QTrust-PPO Environment Verification")\nprint("=" * 78)\n\nprint(f"Python version     : {platform.python_version()}")\nprint(f"PyTorch version    : {torch.__version__}")\nprint(f"PennyLane version  : {qml.__version__}")\nprint(f"Gymnasium version  : {gym.__version__}")\n\nprint("-" * 78)\n\nCUDA_AVAILABLE = torch.cuda.is_available()\nGPU_COUNT = torch.cuda.device_count()\n\nprint(f"CUDA available     : {CUDA_AVAILABLE}")\nprint(f"Detected GPUs      : {GPU_COUNT}")\n\nif CUDA_AVAILABLE:\n    for i in range(GPU_COUNT):\n        print(f"GPU {i}             : {torch.cuda.get_device_name(i)}")\n    \n    DEVICE = torch.device("cuda:0")\nelse:\n    DEVICE = torch.device("cpu")\n\nprint(f"Primary device     : {DEVICE}")\n\nprint("-" * 78)\n\n# Quick PennyLane sanity check\ntry:\n    test_dev = qml.device("default.qubit", wires=2)\n\n    @qml.qnode(test_dev)\n    def test_circuit(x):\n        qml.RY(x, wires=0)\n        qml.CNOT(wires=[0, 1])\n        return qml.expval(qml.PauliZ(0))\n\n    test_value = test_circuit(0.5)\n\n    print(f"PennyLane test     : PASS")\n    print(f"Test expectation   : {float(test_value):.6f}")\n\nexcept Exception as e:\n    print(f"PennyLane test     : FAIL")\n    print(f"Error              : {e}")\n\nprint("=" * 78)\n\n# ===== SOURCE CELL 3 =====\n\n\n# ============================================================\n# FINAL GLOBAL CONFIGURATION — PUBLICATION PROTOCOL\n# ============================================================\n\nimport os\nimport random\nimport numpy as np\nimport torch\n\nMASTER_SEED = 42\n\ndef set_global_seed(seed: int = MASTER_SEED):\n    os.environ["PYTHONHASHSEED"] = str(seed)\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n    torch.use_deterministic_algorithms(True, warn_only=True)\n\nset_global_seed(MASTER_SEED)\n\nDEV_SEEDS = [11, 23, 37, 53, 71]\nFINAL_SEEDS = [11, 23, 37, 53, 71, 89, 107, 131, 157, 181]\n\nENV_CONFIG = {\n    "CartPole-v1": {"state_dim": 4, "action_dim": 2},\n    "Acrobot-v1": {"state_dim": 6, "action_dim": 3},\n    "LunarLander-v3": {"state_dim": 8, "action_dim": 4},\n}\n\nQUANTUM_CONFIG = {\n    "main_qubits": 4,\n    "main_layers": 3,\n    "ablation_qubits": [4, 6],\n    "ablation_layers": [2, 3, 4],\n    "fixed_shots": [32, 64, 128, 256, 512, 1024],\n    "adaptive_shot_schedule": [32, 64, 128, 256, 512, 1024, 2048, 4096],\n    "min_shots": 32,\n    "max_shots": 4096,\n    "delta": 0.05,\n}\n\n# Final protocol uses ONE PPO update epoch, frozen before the final experiment.\nPPO_CONFIG = {\n    "gamma": 0.99,\n    "gae_lambda": 0.95,\n    "clip_epsilon": 0.20,\n    "actor_lr": 3e-4,\n    "critic_lr": 1e-3,\n    "update_epochs": 1,\n    "minibatch_size": 64,\n    "entropy_coef": 0.01,\n    "value_coef": 0.50,\n    "max_grad_norm": 0.50,\n}\n\nCOMPUTE_CONFIG = {\n    "primary_device": str(DEVICE),\n    "gpu_count": torch.cuda.device_count(),\n    "mixed_precision": False,\n}\n\nassert QUANTUM_CONFIG["main_qubits"] == 4\nassert QUANTUM_CONFIG["adaptive_shot_schedule"][-1] == 4096\nassert PPO_CONFIG["update_epochs"] == 1\nassert 0 < QUANTUM_CONFIG["delta"] < 1\n\nprint("=" * 78)\nprint("QTrust-PPO FINAL CONFIGURATION")\nprint("=" * 78)\nprint("Final statistical seeds :", len(FINAL_SEEDS))\nprint("Environments            :", list(ENV_CONFIG))\nprint("Main VQC                : 4 qubits × 3 layers = 24 trainable angles")\nprint("Adaptive shot schedule  :", QUANTUM_CONFIG["adaptive_shot_schedule"])\nprint("PPO update epochs       :", PPO_CONFIG["update_epochs"])\nprint("Primary device          :", DEVICE)\nprint("=" * 78)\n\n\n# ===== SOURCE CELL 5 =====\n\nfrom dataclasses import dataclass, field\nfrom typing import List\nfrom torch.distributions import Categorical\n\ndef make_env(env_name: str, seed: int = MASTER_SEED):\n    """\n    Create and seed a Gymnasium environment reproducibly.\n    """\n    if env_name not in ENV_CONFIG:\n        raise ValueError(f"Unknown environment: {env_name}")\n\n    env = gym.make(env_name)\n\n    obs, info = env.reset(seed=seed)\n\n    env.action_space.seed(seed)\n    env.observation_space.seed(seed)\n\n    return env\n\ndef preprocess_state(state, device=DEVICE):\n    """\n    Convert an environment observation into a finite float32 tensor.\n\n    Step 1: replace possible NaN/Inf values safely\n    Step 2: softly bound values using tanh\n    Step 3: map bounded features to quantum rotation angles [-pi, pi]\n\n    No state feature is discarded.\n    """\n\n    state = np.asarray(state, dtype=np.float32)\n\n    # Safety against unexpected numerical values\n    state = np.nan_to_num(\n        state,\n        nan=0.0,\n        posinf=10.0,\n        neginf=-10.0\n    )\n\n    # Soft normalization to [-1, 1]\n    bounded_state = np.tanh(state)\n\n    # Quantum-friendly rotation angles [-pi, pi]\n    angle_state = np.pi * bounded_state\n\n    return torch.tensor(\n        angle_state,\n        dtype=torch.float32,\n        device=device\n    )\n\n@dataclass\nclass RolloutBuffer:\n    """\n    Shared trajectory buffer for all PPO variants.\n    """\n\n    states: List[np.ndarray] = field(default_factory=list)\n    actions: List[int] = field(default_factory=list)\n    log_probs: List[float] = field(default_factory=list)\n\n    rewards: List[float] = field(default_factory=list)\n\n    values: List[float] = field(default_factory=list)\n    next_values: List[float] = field(default_factory=list)\n\n    terminateds: List[bool] = field(default_factory=list)\n    truncateds: List[bool] = field(default_factory=list)\n\n    # Used later for finite-shot QPPO / QTrust-PPO\n    shot_counts: List[int] = field(default_factory=list)\n\n    def add(\n        self,\n        state,\n        action,\n        log_prob,\n        reward,\n        value,\n        next_value,\n        terminated,\n        truncated,\n        shots=0,\n    ):\n        self.states.append(\n            np.asarray(state, dtype=np.float32).copy()\n        )\n\n        self.actions.append(int(action))\n        self.log_probs.append(float(log_prob))\n        self.rewards.append(float(reward))\n\n        self.values.append(float(value))\n        self.next_values.append(float(next_value))\n\n        self.terminateds.append(bool(terminated))\n        self.truncateds.append(bool(truncated))\n\n        self.shot_counts.append(int(shots))\n\n    def clear(self):\n        self.states.clear()\n        self.actions.clear()\n        self.log_probs.clear()\n        self.rewards.clear()\n\n        self.values.clear()\n        self.next_values.clear()\n\n        self.terminateds.clear()\n        self.truncateds.clear()\n\n        self.shot_counts.clear()\n\n    def __len__(self):\n        return len(self.rewards)\n\ndef compute_gae(\n    rewards,\n    values,\n    next_values,\n    terminateds,\n    truncateds,\n    gamma=PPO_CONFIG["gamma"],\n    gae_lambda=PPO_CONFIG["gae_lambda"],\n):\n    """\n    Compute Generalized Advantage Estimation.\n\n    Important Gymnasium distinction:\n\n    terminated=True:\n        true terminal state -> no value bootstrap.\n\n    truncated=True:\n        time-limit/artificial ending -> bootstrap remains valid,\n        but recursive GAE must stop before a reset state.\n    """\n\n    rewards = np.asarray(rewards, dtype=np.float64)\n    values = np.asarray(values, dtype=np.float64)\n    next_values = np.asarray(next_values, dtype=np.float64)\n\n    terminateds = np.asarray(\n        terminateds,\n        dtype=np.float64\n    )\n\n    truncateds = np.asarray(\n        truncateds,\n        dtype=np.float64\n    )\n\n    n_steps = len(rewards)\n\n    if not (\n        len(values)\n        == len(next_values)\n        == len(terminateds)\n        == len(truncateds)\n        == n_steps\n    ):\n        raise ValueError(\n            "All GAE input arrays must have identical lengths."\n        )\n\n    advantages = np.zeros(\n        n_steps,\n        dtype=np.float64\n    )\n\n    gae = 0.0\n\n    for t in reversed(range(n_steps)):\n\n        # True termination removes value bootstrap.\n        bootstrap_mask = 1.0 - terminateds[t]\n\n        # Both termination and truncation stop recursive GAE,\n        # because the next stored transition may belong to a reset episode.\n        episode_end = max(\n            terminateds[t],\n            truncateds[t]\n        )\n\n        recursion_mask = 1.0 - episode_end\n\n        delta = (\n            rewards[t]\n            + gamma\n            * next_values[t]\n            * bootstrap_mask\n            - values[t]\n        )\n\n        gae = (\n            delta\n            + gamma\n            * gae_lambda\n            * recursion_mask\n            * gae\n        )\n\n        advantages[t] = gae\n\n    returns = advantages + values\n\n    return advantages, returns\n\ndef normalize_advantages(\n    advantages,\n    eps=1e-8\n):\n    """\n    Normalize PPO advantages for more stable optimization.\n    """\n\n    advantages = np.asarray(\n        advantages,\n        dtype=np.float32\n    )\n\n    if len(advantages) <= 1:\n        return advantages\n\n    mean = advantages.mean()\n    std = advantages.std()\n\n    return (\n        advantages - mean\n    ) / (std + eps)\n\nclass ClassicalActor(nn.Module):\n    """\n    Standard MLP policy used as the main classical PPO baseline.\n    """\n\n    def __init__(\n        self,\n        state_dim,\n        action_dim,\n        hidden_dim=64,\n    ):\n        super().__init__()\n\n        self.network = nn.Sequential(\n            nn.Linear(state_dim, hidden_dim),\n            nn.Tanh(),\n\n            nn.Linear(hidden_dim, hidden_dim),\n            nn.Tanh(),\n\n            nn.Linear(hidden_dim, action_dim),\n        )\n\n    def forward(self, states):\n        """\n        Returns unnormalized policy logits.\n        """\n        return self.network(states)\n\nclass ClassicalCritic(nn.Module):\n    """\n    State-value estimator V(s).\n    """\n\n    def __init__(\n        self,\n        state_dim,\n        hidden_dim=64,\n    ):\n        super().__init__()\n\n        self.network = nn.Sequential(\n            nn.Linear(state_dim, hidden_dim),\n            nn.Tanh(),\n\n            nn.Linear(hidden_dim, hidden_dim),\n            nn.Tanh(),\n\n            nn.Linear(hidden_dim, 1),\n        )\n\n    def forward(self, states):\n        return self.network(states).squeeze(-1)\n\nclass ClassicalActorCritic(nn.Module):\n    """\n    Combined interface used by PPO training and evaluation.\n    """\n\n    def __init__(\n        self,\n        state_dim,\n        action_dim,\n        hidden_dim=64,\n    ):\n        super().__init__()\n\n        self.actor = ClassicalActor(\n            state_dim=state_dim,\n            action_dim=action_dim,\n            hidden_dim=hidden_dim,\n        )\n\n        self.critic = ClassicalCritic(\n            state_dim=state_dim,\n            hidden_dim=hidden_dim,\n        )\n\n    @torch.no_grad()\n    def act(self, state):\n        """\n        Sample one action from the current policy while following\n        the actual device of the model parameters.\n        """\n\n        device = next(\n            self.actor.parameters()\n        ).device\n\n        state = torch.as_tensor(\n            state,\n            dtype=torch.float32,\n            device=device,\n        )\n\n        if state.ndim == 1:\n            state = state.unsqueeze(0)\n\n        logits = self.actor(state)\n        dist = Categorical(logits=logits)\n\n        action = dist.sample()\n        log_prob = dist.log_prob(action)\n        value = self.critic(state)\n\n        return (\n            int(action.item()),\n            float(log_prob.item()),\n            float(value.reshape(-1)[0].item()),\n        )\n\n    def evaluate_actions(\n        self,\n        states,\n        actions,\n    ):\n        """\n        Evaluate actions already stored in the PPO rollout.\n        """\n\n        logits = self.actor(states)\n\n        dist = Categorical(logits=logits)\n\n        log_probs = dist.log_prob(actions)\n        entropy = dist.entropy()\n\n        values = self.critic(states)\n\n        return (\n            log_probs,\n            entropy,\n            values,\n        )\n\n# ===== SOURCE CELL 7 =====\n\n\n# ============================================================\n# QUANTUM BACKEND + FINAL ANALYTIC VQC\n# ============================================================\n\nimport pennylane as qml\nimport torch\nimport torch.nn as nn\nfrom torch.distributions import Categorical\n\n# PyTorch tensors stay on CPU while Lightning may execute the\n# quantum simulation on GPU internally.\nQUANTUM_TORCH_DEVICE = torch.device("cpu")\n\ndef probe_quantum_backend(device_name):\n    dev = qml.device(device_name, wires=2, shots=None)\n    diff_method = (\n        "parameter-shift"\n        if device_name == "lightning.gpu"\n        else "backprop"\n    )\n\n    @qml.qnode(dev, interface="torch", diff_method=diff_method)\n    def probe_circuit(x, theta):\n        qml.RY(x, wires=0)\n        qml.RY(theta, wires=1)\n        qml.CNOT(wires=[0, 1])\n        return qml.probs(wires=[0, 1])\n\n    x = torch.tensor(0.37, dtype=torch.float64)\n    theta = torch.tensor(0.21, dtype=torch.float64, requires_grad=True)\n    probs = probe_circuit(x, theta)\n    loss = -torch.log(probs[0] + 1e-12)\n    loss.backward()\n\n    assert torch.isfinite(probs).all()\n    assert torch.isfinite(theta.grad)\n    assert abs(float(probs.sum()) - 1.0) < 1e-8\n    return True\n\nANALYTIC_QDEVICE_NAME = None\n\nfor candidate in ["lightning.gpu", "default.qubit"]:\n    try:\n        probe_quantum_backend(candidate)\n        ANALYTIC_QDEVICE_NAME = candidate\n        print("Quantum backend selected :", candidate)\n        break\n    except Exception as exc:\n        print(f"{candidate:<20} unavailable ({type(exc).__name__})")\n\nif ANALYTIC_QDEVICE_NAME is None:\n    raise RuntimeError("No compatible PennyLane quantum backend found.")\n\n\ndef build_analytic_quantum_policy(\n    input_dim,\n    n_qubits=QUANTUM_CONFIG["main_qubits"],\n    n_layers=QUANTUM_CONFIG["main_layers"],\n):\n    """\n    VQC policy with balanced |+> initialization.\n\n    Hadamard preparation prevents the initial policy from\n    collapsing toward one computational-basis action.\n    """\n\n    dev = qml.device(\n        ANALYTIC_QDEVICE_NAME,\n        wires=n_qubits,\n        shots=None,\n    )\n\n    diff_method = (\n        "parameter-shift"\n        if ANALYTIC_QDEVICE_NAME == "lightning.gpu"\n        else "backprop"\n    )\n\n    @qml.qnode(\n        dev,\n        interface="torch",\n        diff_method=diff_method,\n    )\n    def quantum_policy(features, weights):\n\n        # ====================================================\n        # Balanced quantum initialization\n        # ====================================================\n\n        for qubit in range(n_qubits):\n            qml.Hadamard(wires=qubit)\n\n        # ====================================================\n        # Variational layers + data re-uploading\n        # ====================================================\n\n        for layer in range(n_layers):\n\n            # ------------------------------------------------\n            # State encoding\n            # ------------------------------------------------\n\n            for qubit in range(n_qubits):\n\n                feature_index = (\n                    layer * n_qubits + qubit\n                ) % input_dim\n\n                angle = features[feature_index]\n\n                # Moderate encoding scale improves stability\n                qml.RY(\n                    0.5 * angle,\n                    wires=qubit,\n                )\n\n                qml.RZ(\n                    0.25 * angle,\n                    wires=qubit,\n                )\n\n            # ------------------------------------------------\n            # Ring entanglement\n            # ------------------------------------------------\n\n            for qubit in range(n_qubits - 1):\n\n                qml.CNOT(\n                    wires=[\n                        qubit,\n                        qubit + 1,\n                    ]\n                )\n\n            qml.CNOT(\n                wires=[\n                    n_qubits - 1,\n                    0,\n                ]\n            )\n\n            # ------------------------------------------------\n            # Trainable quantum rotations\n            # ------------------------------------------------\n\n            for qubit in range(n_qubits):\n\n                qml.RY(\n                    weights[\n                        layer,\n                        qubit,\n                        0,\n                    ],\n                    wires=qubit,\n                )\n\n                qml.RZ(\n                    weights[\n                        layer,\n                        qubit,\n                        1,\n                    ],\n                    wires=qubit,\n                )\n\n        return qml.probs(\n            wires=[0, 1]\n        )\n\n    return quantum_policy\n\ndef get_action_readout_matrix(\n    action_dim,\n    dtype=torch.float64,\n    device=QUANTUM_TORCH_DEVICE,\n):\n    """\n    Column-stochastic classical post-processing matrix.\n\n    Each quantum basis outcome is mapped probabilistically\n    to one environment action.\n\n    For 3 actions:\n        |00> -> action 0\n        |01> -> action 1\n        |10> -> action 2\n        |11> -> uniformly randomized across all 3 actions\n\n    This makes uniform quantum basis probabilities map to a\n    uniform 3-action policy while preserving physical validity.\n    """\n\n    if action_dim == 2:\n\n        matrix = [\n            [1.0, 0.0, 1.0, 0.0],\n            [0.0, 1.0, 0.0, 1.0],\n        ]\n\n    elif action_dim == 3:\n\n        matrix = [\n            [1.0, 0.0, 0.0, 1.0 / 3.0],\n            [0.0, 1.0, 0.0, 1.0 / 3.0],\n            [0.0, 0.0, 1.0, 1.0 / 3.0],\n        ]\n\n    elif action_dim == 4:\n\n        matrix = [\n            [1.0, 0.0, 0.0, 0.0],\n            [0.0, 1.0, 0.0, 0.0],\n            [0.0, 0.0, 1.0, 0.0],\n            [0.0, 0.0, 0.0, 1.0],\n        ]\n\n    else:\n        raise ValueError(\n            f"Unsupported action dimension: {action_dim}"\n        )\n\n    matrix = torch.tensor(\n        matrix,\n        dtype=dtype,\n        device=device,\n    )\n\n    # Every quantum outcome must distribute total mass = 1\n    column_sums = matrix.sum(dim=0)\n\n    assert torch.allclose(\n        column_sums,\n        torch.ones_like(column_sums),\n        atol=1e-10,\n    )\n\n    return matrix\n\ndef balanced_action_probabilities(\n    basis_probs,\n    action_dim,\n):\n    """\n    Quantum basis probabilities -> action probabilities.\n    """\n\n    readout = get_action_readout_matrix(\n        action_dim=action_dim,\n        dtype=basis_probs.dtype,\n        device=basis_probs.device,\n    )\n\n    action_probs = (\n        readout @ basis_probs\n    )\n\n    action_probs = torch.clamp(\n        action_probs,\n        min=1e-12,\n    )\n\n    # Numerical protection\n    action_probs = (\n        action_probs /\n        action_probs.sum()\n    )\n\n    return action_probs\n\nclass QuantumActor(nn.Module):\n    """\n    Variational quantum policy actor.\n\n    Quantum parameters:\n        n_layers × n_qubits × 2\n\n    Main experiment:\n        3 × 4 × 2 = 24 parameters\n    """\n\n    def __init__(\n        self,\n        state_dim,\n        action_dim,\n        n_qubits=QUANTUM_CONFIG["main_qubits"],\n        n_layers=QUANTUM_CONFIG["main_layers"],\n    ):\n        super().__init__()\n\n        self.state_dim = state_dim\n        self.action_dim = action_dim\n        self.n_qubits = n_qubits\n        self.n_layers = n_layers\n\n        self.qnode = build_analytic_quantum_policy(\n            input_dim=state_dim,\n            n_qubits=n_qubits,\n            n_layers=n_layers,\n        )\n\n        initial_weights = (\n            0.05\n            * torch.randn(\n                n_layers,\n                n_qubits,\n                2,\n                dtype=torch.float64,\n            )\n        )\n\n        self.quantum_weights = nn.Parameter(\n            initial_weights\n        )\n\n    def encode_state(self, state):\n        """\n        Raw environment state -> bounded quantum angles.\n        """\n\n        if not torch.is_tensor(state):\n            state = torch.tensor(\n                state,\n                dtype=torch.float64,\n            )\n\n        state = state.to(\n            device=QUANTUM_TORCH_DEVICE,\n            dtype=torch.float64,\n        )\n\n        # Protect against unexpected numerical extremes\n        state = torch.nan_to_num(\n            state,\n            nan=0.0,\n            posinf=10.0,\n            neginf=-10.0,\n        )\n\n        return (\n            torch.pi\n            * torch.tanh(state)\n        )\n\n    def single_forward(self, state):\n        """\n        Quantum policy probabilities for one state.\n        """\n\n        encoded_state = self.encode_state(\n            state\n        )\n\n        basis_probs = self.qnode(\n            encoded_state,\n            self.quantum_weights,\n        )\n\n        action_probs = (\n            balanced_action_probabilities(\n                basis_probs,\n                self.action_dim,\n            )\n        )\n\n        return action_probs\n\n    def forward(self, states):\n        """\n        Supports either:\n            [state_dim]\n        or:\n            [batch, state_dim]\n        """\n\n        if not torch.is_tensor(states):\n            states = torch.tensor(\n                states,\n                dtype=torch.float64,\n            )\n\n        if states.ndim == 1:\n            return self.single_forward(\n                states\n            )\n\n        batch_probs = [\n            self.single_forward(state)\n            for state in states\n        ]\n\n        return torch.stack(\n            batch_probs,\n            dim=0\n        )\n\n    @torch.no_grad()\n    def act(self, state):\n        """\n        Sample an action from the analytic VQC policy.\n        """\n\n        probs = self.single_forward(state)\n\n        dist = Categorical(\n            probs=probs\n        )\n\n        action = dist.sample()\n\n        log_prob = dist.log_prob(\n            action\n        )\n\n        return (\n            int(action.item()),\n            float(log_prob.item()),\n            probs.detach().cpu().numpy(),\n        )\n\n    def evaluate_actions(\n        self,\n        states,\n        actions,\n    ):\n        """\n        Evaluate stored actions for PPO optimization.\n        """\n\n        probs = self.forward(\n            states\n        )\n\n        actions = actions.to(\n            probs.device,\n            dtype=torch.long,\n        )\n\n        dist = Categorical(\n            probs=probs\n        )\n\n        log_probs = dist.log_prob(\n            actions\n        )\n\n        entropy = dist.entropy()\n\n        return (\n            log_probs,\n            entropy,\n            probs,\n        )\n\nclass HybridQuantumActorCritic:\n    """\n    VQC actor + classical critic.\n\n    Actor:\n        CPU-facing PyTorch tensors,\n        with Lightning-GPU performing quantum simulation.\n\n    Critic:\n        Standard PyTorch network on CUDA.\n    """\n\n    def __init__(\n        self,\n        state_dim,\n        action_dim,\n        critic_hidden_dim=64,\n    ):\n\n        self.actor = QuantumActor(\n            state_dim=state_dim,\n            action_dim=action_dim,\n        )\n\n        self.critic = ClassicalCritic(\n            state_dim=state_dim,\n            hidden_dim=critic_hidden_dim,\n        ).to(DEVICE)\n\n    @torch.no_grad()\n    def act(self, state):\n\n        action, log_prob, probs = (\n            self.actor.act(state)\n        )\n\n        critic_state = torch.tensor(\n            state,\n            dtype=torch.float32,\n            device=DEVICE,\n        ).unsqueeze(0)\n\n        value = float(\n            self.critic(\n                critic_state\n            ).item()\n        )\n\n        return (\n            action,\n            log_prob,\n            value,\n            probs,\n        )\n\n\n@torch.no_grad()\ndef get_analytic_action_probs(actor, state):\n    """\n    Hidden Born probabilities used only to draw finite-shot\n    certification samples and to audit classification truth.\n    They are never used directly by the QTrust decision rule.\n    """\n    probs = actor.single_forward(state)\n    return (\n        probs.detach()\n        .cpu()\n        .numpy()\n        .astype(np.float64)\n    )\n\n\n# ===== SOURCE CELL 8 =====\n\ndef build_finite_shot_quantum_policy(\n    input_dim,\n    shots,\n    n_qubits=QUANTUM_CONFIG["main_qubits"],\n    n_layers=QUANTUM_CONFIG["main_layers"],\n):\n    """\n    Finite-shot VQC using shots at the QNode level.\n    """\n\n    dev = qml.device(\n        ANALYTIC_QDEVICE_NAME,\n        wires=n_qubits,\n    )\n\n    @qml.qnode(\n        dev,\n        interface="torch",\n        diff_method="parameter-shift",\n        shots=int(shots),\n    )\n    def finite_qnode(features, weights):\n\n        # Balanced initialization\n        for qubit in range(n_qubits):\n            qml.Hadamard(wires=qubit)\n\n        # Variational circuit\n        for layer in range(n_layers):\n\n            # Data re-uploading\n            for qubit in range(n_qubits):\n\n                feature_index = (\n                    layer * n_qubits + qubit\n                ) % input_dim\n\n                angle = features[feature_index]\n\n                qml.RY(\n                    0.5 * angle,\n                    wires=qubit,\n                )\n\n                qml.RZ(\n                    0.25 * angle,\n                    wires=qubit,\n                )\n\n            # Ring entanglement\n            for qubit in range(n_qubits - 1):\n\n                qml.CNOT(\n                    wires=[qubit, qubit + 1]\n                )\n\n            qml.CNOT(\n                wires=[n_qubits - 1, 0]\n            )\n\n            # Trainable rotations\n            for qubit in range(n_qubits):\n\n                qml.RY(\n                    weights[layer, qubit, 0],\n                    wires=qubit,\n                )\n\n                qml.RZ(\n                    weights[layer, qubit, 1],\n                    wires=qubit,\n                )\n\n        return qml.probs(wires=[0, 1])\n\n    return finite_qnode\n\nclass FixedShotQuantumActor(nn.Module):\n\n    def __init__(\n        self,\n        state_dim,\n        action_dim,\n        shots,\n        n_qubits=QUANTUM_CONFIG["main_qubits"],\n        n_layers=QUANTUM_CONFIG["main_layers"],\n    ):\n        super().__init__()\n\n        self.state_dim = state_dim\n        self.action_dim = action_dim\n        self.shots = int(shots)\n\n        self.n_qubits = n_qubits\n        self.n_layers = n_layers\n\n        self.qnode = build_finite_shot_quantum_policy(\n            input_dim=state_dim,\n            shots=shots,\n            n_qubits=n_qubits,\n            n_layers=n_layers,\n        )\n\n        self.quantum_weights = nn.Parameter(\n            0.05\n            * torch.randn(\n                n_layers,\n                n_qubits,\n                2,\n                dtype=torch.float64,\n            )\n        )\n\n    def encode_state(self, state):\n\n        if not torch.is_tensor(state):\n            state = torch.tensor(\n                state,\n                dtype=torch.float64,\n            )\n\n        state = state.to(\n            QUANTUM_TORCH_DEVICE,\n            dtype=torch.float64,\n        )\n\n        state = torch.nan_to_num(\n            state,\n            nan=0.0,\n            posinf=10.0,\n            neginf=-10.0,\n        )\n\n        return torch.pi * torch.tanh(state)\n\n    def single_forward(self, state):\n\n        encoded = self.encode_state(state)\n\n        basis_probs = self.qnode(\n            encoded,\n            self.quantum_weights,\n        )\n\n        return balanced_action_probabilities(\n            basis_probs,\n            self.action_dim,\n        )\n\n    def forward(self, states):\n\n        if not torch.is_tensor(states):\n            states = torch.tensor(\n                states,\n                dtype=torch.float64,\n            )\n\n        if states.ndim == 1:\n            return self.single_forward(states)\n\n        return torch.stack(\n            [\n                self.single_forward(state)\n                for state in states\n            ],\n            dim=0,\n        )\n\n    @torch.no_grad()\n    def act(self, state):\n\n        probs = self.single_forward(state)\n\n        dist = Categorical(probs=probs)\n\n        action = dist.sample()\n\n        log_prob = dist.log_prob(action)\n\n        return (\n            int(action.item()),\n            float(log_prob.item()),\n            probs.detach().cpu().numpy(),\n        )\n\n    def evaluate_actions(\n        self,\n        states,\n        actions,\n    ):\n\n        probs = self.forward(states)\n\n        actions = actions.to(\n            probs.device,\n            dtype=torch.long,\n        )\n\n        dist = Categorical(probs=probs)\n\n        return (\n            dist.log_prob(actions),\n            dist.entropy(),\n            probs,\n        )\n\nclass FixedShotHybridActorCritic:\n\n    def __init__(\n        self,\n        state_dim,\n        action_dim,\n        shots,\n        critic_hidden_dim=64,\n    ):\n\n        self.shots = int(shots)\n\n        self.actor = FixedShotQuantumActor(\n            state_dim=state_dim,\n            action_dim=action_dim,\n            shots=shots,\n        )\n\n        self.critic = ClassicalCritic(\n            state_dim=state_dim,\n            hidden_dim=critic_hidden_dim,\n        ).to(DEVICE)\n\n    @torch.no_grad()\n    def act(self, state):\n\n        action, log_prob, probs = (\n            self.actor.act(state)\n        )\n\n        critic_state = torch.tensor(\n            state,\n            dtype=torch.float32,\n            device=DEVICE,\n        ).unsqueeze(0)\n\n        value = float(\n            self.critic(\n                critic_state\n            ).item()\n        )\n\n        return (\n            action,\n            log_prob,\n            value,\n            probs,\n        )\n\ndef quantum_ppo_update(\n    model,\n    buffer,\n    actor_optimizer,\n    critic_optimizer,\n    update_epochs=None,\n):\n    """\n    PPO optimization for:\n\n        VQC quantum actor\n        +\n        classical neural critic\n\n    The actor and critic are optimized separately because\n    they live on different computational devices.\n    """\n\n    if len(buffer) < 2:\n        raise ValueError(\n            "Quantum PPO update requires at least 2 transitions."\n        )\n\n    if update_epochs is None:\n        update_epochs = PPO_CONFIG["update_epochs"]\n\n    # --------------------------------------------------------\n    # GAE + returns\n    # --------------------------------------------------------\n\n    advantages, returns = compute_gae(\n        rewards=buffer.rewards,\n        values=buffer.values,\n        next_values=buffer.next_values,\n        terminateds=buffer.terminateds,\n        truncateds=buffer.truncateds,\n        gamma=PPO_CONFIG["gamma"],\n        gae_lambda=PPO_CONFIG["gae_lambda"],\n    )\n\n    advantages = normalize_advantages(\n        advantages\n    )\n\n    # --------------------------------------------------------\n    # Actor tensors: CPU float64\n    # --------------------------------------------------------\n\n    actor_states = torch.tensor(\n        np.asarray(buffer.states),\n        dtype=torch.float64,\n        device=QUANTUM_TORCH_DEVICE,\n    )\n\n    actions_cpu = torch.tensor(\n        buffer.actions,\n        dtype=torch.long,\n        device=QUANTUM_TORCH_DEVICE,\n    )\n\n    old_log_probs_cpu = torch.tensor(\n        buffer.log_probs,\n        dtype=torch.float64,\n        device=QUANTUM_TORCH_DEVICE,\n    )\n\n    advantages_cpu = torch.tensor(\n        advantages,\n        dtype=torch.float64,\n        device=QUANTUM_TORCH_DEVICE,\n    )\n\n    # --------------------------------------------------------\n    # Critic tensors: GPU float32\n    # --------------------------------------------------------\n\n    critic_states = torch.tensor(\n        np.asarray(buffer.states),\n        dtype=torch.float32,\n        device=DEVICE,\n    )\n\n    returns_gpu = torch.tensor(\n        returns,\n        dtype=torch.float32,\n        device=DEVICE,\n    )\n\n    n_samples = len(buffer)\n\n    batch_size = min(\n        PPO_CONFIG["minibatch_size"],\n        n_samples,\n    )\n\n    # --------------------------------------------------------\n    # Diagnostics\n    # --------------------------------------------------------\n\n    actor_losses = []\n    critic_losses = []\n    entropies = []\n    approx_kls = []\n    clip_fractions = []\n    actor_grad_norms = []\n    critic_grad_norms = []\n\n    # ========================================================\n    # PPO epochs\n    # ========================================================\n\n    for epoch in range(update_epochs):\n\n        permutation = torch.randperm(\n            n_samples\n        )\n\n        for start in range(\n            0,\n            n_samples,\n            batch_size\n        ):\n\n            indices = permutation[\n                start:start + batch_size\n            ]\n\n            # =================================================\n            # QUANTUM ACTOR UPDATE\n            # =================================================\n\n            mb_actor_states = actor_states[\n                indices\n            ]\n\n            mb_actions = actions_cpu[\n                indices\n            ]\n\n            mb_old_log_probs = old_log_probs_cpu[\n                indices\n            ]\n\n            mb_advantages = advantages_cpu[\n                indices\n            ]\n\n            (\n                new_log_probs,\n                entropy,\n                _\n            ) = model.actor.evaluate_actions(\n                mb_actor_states,\n                mb_actions,\n            )\n\n            log_ratio = (\n                new_log_probs\n                - mb_old_log_probs\n            )\n\n            ratio = torch.exp(\n                log_ratio\n            )\n\n            unclipped_objective = (\n                ratio\n                * mb_advantages\n            )\n\n            clipped_ratio = torch.clamp(\n                ratio,\n                1.0 - PPO_CONFIG[\n                    "clip_epsilon"\n                ],\n                1.0 + PPO_CONFIG[\n                    "clip_epsilon"\n                ],\n            )\n\n            clipped_objective = (\n                clipped_ratio\n                * mb_advantages\n            )\n\n            policy_loss = -torch.min(\n                unclipped_objective,\n                clipped_objective,\n            ).mean()\n\n            entropy_mean = (\n                entropy.mean()\n            )\n\n            actor_loss = (\n                policy_loss\n                - PPO_CONFIG[\n                    "entropy_coef"\n                ]\n                * entropy_mean\n            )\n\n            actor_optimizer.zero_grad()\n\n            actor_loss.backward()\n\n            actor_grad_norm = (\n                torch.nn.utils.clip_grad_norm_(\n                    model.actor.parameters(),\n                    PPO_CONFIG[\n                        "max_grad_norm"\n                    ],\n                )\n            )\n\n            actor_optimizer.step()\n\n            # =================================================\n            # CLASSICAL CRITIC UPDATE\n            # =================================================\n\n            gpu_indices = indices.to(\n                DEVICE\n            )\n\n            mb_critic_states = (\n                critic_states[\n                    gpu_indices\n                ]\n            )\n\n            mb_returns = (\n                returns_gpu[\n                    gpu_indices\n                ]\n            )\n\n            predicted_values = (\n                model.critic(\n                    mb_critic_states\n                )\n            )\n\n            critic_loss = F.mse_loss(\n                predicted_values,\n                mb_returns,\n            )\n\n            critic_optimizer.zero_grad()\n\n            critic_loss.backward()\n\n            critic_grad_norm = (\n                torch.nn.utils.clip_grad_norm_(\n                    model.critic.parameters(),\n                    PPO_CONFIG[\n                        "max_grad_norm"\n                    ],\n                )\n            )\n\n            critic_optimizer.step()\n\n            # =================================================\n            # Diagnostics\n            # =================================================\n\n            with torch.no_grad():\n\n                approx_kl = (\n                    (ratio - 1.0)\n                    - log_ratio\n                ).mean()\n\n                clip_fraction = (\n                    (\n                        torch.abs(\n                            ratio - 1.0\n                        )\n                        > PPO_CONFIG[\n                            "clip_epsilon"\n                        ]\n                    )\n                    .double()\n                    .mean()\n                )\n\n            actor_losses.append(\n                float(\n                    actor_loss.detach()\n                )\n            )\n\n            critic_losses.append(\n                float(\n                    critic_loss.detach()\n                )\n            )\n\n            entropies.append(\n                float(\n                    entropy_mean.detach()\n                )\n            )\n\n            approx_kls.append(\n                float(\n                    approx_kl.detach()\n                )\n            )\n\n            clip_fractions.append(\n                float(\n                    clip_fraction.detach()\n                )\n            )\n\n            actor_grad_norms.append(\n                float(\n                    actor_grad_norm.detach()\n                )\n            )\n\n            critic_grad_norms.append(\n                float(\n                    critic_grad_norm.detach()\n                )\n            )\n\n    # ========================================================\n    # Summary\n    # ========================================================\n\n    metrics = {\n\n        "actor_loss":\n            float(np.mean(actor_losses)),\n\n        "critic_loss":\n            float(np.mean(critic_losses)),\n\n        "entropy":\n            float(np.mean(entropies)),\n\n        "approx_kl":\n            float(np.mean(approx_kls)),\n\n        "clip_fraction":\n            float(np.mean(clip_fractions)),\n\n        "actor_grad_norm":\n            float(np.mean(actor_grad_norms)),\n\n        "critic_grad_norm":\n            float(np.mean(critic_grad_norms)),\n\n        "rollout_size":\n            int(n_samples),\n    }\n\n    return metrics\n\n# ===== SOURCE CELL 9 =====\n\n\n# ============================================================\n# QTrust-PPO EXACT FINITE-HORIZON CONTROLLER\n# ============================================================\n\nimport copy\nfrom scipy.stats import beta\n\nQTRUST_SHOT_SCHEDULE = [\n    32, 64, 128, 256, 512, 1024, 2048, 4096\n]\n\ndef cp_lower_one_sided(k, n, alpha):\n    if k == 0:\n        return 0.0\n    return float(beta.ppf(alpha, k, n - k + 1))\n\ndef cp_upper_one_sided(k, n, alpha):\n    if k == n:\n        return 1.0\n    return float(beta.ppf(1.0 - alpha, k + 1, n - k))\n\ndef exact_ratio_bounds(\n    new_successes,\n    old_successes,\n    trials,\n    stage_alpha,\n    eps=1e-12,\n):\n    """\n    Simultaneous lower/upper confidence bounds for p_new/p_old.\n\n    The per-stage familywise error budget is stage_alpha.\n    Four one-sided binomial tails are used:\n      new lower, new upper, old lower, old upper.\n    Therefore each tail receives stage_alpha/4.\n\n    Across K scheduled stages, stage_alpha = delta/K, so a\n    direct Bonferroni union bound controls the full finite\n    horizon at no more than delta.\n    """\n    tail_alpha = stage_alpha / 4.0\n\n    new_low = cp_lower_one_sided(\n        new_successes, trials, tail_alpha\n    )\n    new_high = cp_upper_one_sided(\n        new_successes, trials, tail_alpha\n    )\n    old_low = cp_lower_one_sided(\n        old_successes, trials, tail_alpha\n    )\n    old_high = cp_upper_one_sided(\n        old_successes, trials, tail_alpha\n    )\n\n    ratio_low = new_low / max(old_high, eps)\n    ratio_high = (\n        np.inf\n        if old_low <= eps\n        else new_high / old_low\n    )\n\n    return float(ratio_low), float(ratio_high)\n\n\ndef qtrust_finite_horizon_test(\n    p_old,\n    p_new,\n    action,\n    advantage,\n    seed,\n    shot_schedule=QTRUST_SHOT_SCHEDULE,\n    delta=QUANTUM_CONFIG["delta"],\n    clip_epsilon=PPO_CONFIG["clip_epsilon"],\n):\n    """\n    Advantage-aware finite-shot certification of the PPO\n    clipping decision for one stored state-action pair.\n\n    Positive advantage: only the upper clipping boundary\n    1 + epsilon matters.\n    Negative advantage: only the lower clipping boundary\n    1 - epsilon matters.\n\n    The controller stops at the first shot stage whose exact\n    ratio confidence interval lies wholly on one side of the\n    relevant boundary. Otherwise it abstains (UNCERTAIN) at\n    the finite maximum budget.\n    """\n    p_old_a = float(p_old[action])\n    p_new_a = float(p_new[action])\n\n    lower_clip = 1.0 - clip_epsilon\n    upper_clip = 1.0 + clip_epsilon\n\n    rng_old = np.random.default_rng(seed)\n    rng_new = np.random.default_rng(seed + 1_000_000)\n\n    old_successes = 0\n    new_successes = 0\n    previous_shots = 0\n    trace = []\n    final_decision = "UNCERTAIN"\n\n    K = len(shot_schedule)\n    stage_alpha = delta / K\n\n    for stage, target_shots in enumerate(shot_schedule, start=1):\n        additional_shots = target_shots - previous_shots\n        if additional_shots <= 0:\n            raise ValueError("Shot schedule must be strictly increasing.")\n\n        old_successes += int(\n            rng_old.binomial(additional_shots, p_old_a)\n        )\n        new_successes += int(\n            rng_new.binomial(additional_shots, p_new_a)\n        )\n\n        ratio_low, ratio_high = exact_ratio_bounds(\n            new_successes=new_successes,\n            old_successes=old_successes,\n            trials=target_shots,\n            stage_alpha=stage_alpha,\n        )\n\n        old_hat = old_successes / target_shots\n        new_hat = new_successes / target_shots\n        ratio_hat = new_hat / max(old_hat, 1e-12)\n\n        if advantage >= 0:\n            if ratio_high <= upper_clip:\n                final_decision = "SAFE_NO_UPPER_CLIP"\n            elif ratio_low > upper_clip:\n                final_decision = "CONFIRMED_UPPER_CLIP"\n            else:\n                final_decision = "UNCERTAIN"\n        else:\n            if ratio_low >= lower_clip:\n                final_decision = "SAFE_NO_LOWER_CLIP"\n            elif ratio_high < lower_clip:\n                final_decision = "CONFIRMED_LOWER_CLIP"\n            else:\n                final_decision = "UNCERTAIN"\n\n        trace.append(\n            {\n                "Stage": stage,\n                "Shots / Policy": target_shots,\n                "Ratio-hat": ratio_hat,\n                "Ratio Low": ratio_low,\n                "Ratio High": ratio_high,\n                "Stage Alpha": stage_alpha,\n                "Tail Alpha": stage_alpha / 4.0,\n                "Decision": final_decision,\n            }\n        )\n\n        previous_shots = target_shots\n\n        if final_decision != "UNCERTAIN":\n            break\n\n    true_ratio = p_new_a / max(p_old_a, 1e-12)\n\n    return (\n        pd.DataFrame(trace),\n        {\n            "true_ratio": float(true_ratio),\n            "decision": final_decision,\n            "resolved": final_decision != "UNCERTAIN",\n            "shots_per_policy": int(previous_shots),\n            "total_probability_shots": int(2 * previous_shots),\n            "stages_used": len(trace),\n        },\n    )\n\n\ndef get_buffer_advantages(buffer):\n\n    advantages, returns = compute_gae(\n        rewards=buffer.rewards,\n        values=buffer.values,\n        next_values=buffer.next_values,\n        terminateds=buffer.terminateds,\n        truncateds=buffer.truncateds,\n        gamma=PPO_CONFIG["gamma"],\n        gae_lambda=PPO_CONFIG["gae_lambda"],\n    )\n\n    advantages = normalize_advantages(\n        advantages\n    )\n\n    return (\n        advantages,\n        returns,\n    )\n\ndef make_analytic_shadow(actor):\n    """\n    Reconstruct an analytic QuantumActor with exactly the same\n    VQC parameters as a finite-shot actor.\n\n    This is NOT used for optimization.\n    """\n\n    shadow = QuantumActor(\n        state_dim=actor.state_dim,\n        action_dim=actor.action_dim,\n        n_qubits=actor.n_qubits,\n        n_layers=actor.n_layers,\n    )\n\n    with torch.no_grad():\n\n        shadow.quantum_weights.copy_(\n            actor.quantum_weights.detach()\n        )\n\n    for p in shadow.parameters():\n        p.requires_grad_(False)\n\n    return shadow\n\ndef certify_qtrust_update(\n    old_actor,\n    new_actor,\n    buffer,\n    advantages,\n    seed,\n    shot_schedule=QTRUST_SHOT_SCHEDULE,\n):\n\n    records = []\n\n    lower_clip = (\n        1.0\n        - PPO_CONFIG["clip_epsilon"]\n    )\n\n    upper_clip = (\n        1.0\n        + PPO_CONFIG["clip_epsilon"]\n    )\n\n    for i in range(len(buffer)):\n\n        state = buffer.states[i]\n        action = buffer.actions[i]\n\n        advantage = float(\n            advantages[i]\n        )\n\n        # ----------------------------------------------------\n        # Hidden Born probabilities:\n        # used ONLY for finite-measurement simulation and\n        # ground-truth evaluation.\n        # ----------------------------------------------------\n\n        p_old = get_analytic_action_probs(\n            old_actor,\n            state,\n        )\n\n        p_new = get_analytic_action_probs(\n            new_actor,\n            state,\n        )\n\n        # ----------------------------------------------------\n        # FINAL finite-horizon exact QTrust controller\n        # ----------------------------------------------------\n\n        _, summary = (\n            qtrust_finite_horizon_test(\n                p_old=p_old,\n                p_new=p_new,\n                action=action,\n                advantage=advantage,\n                seed=(\n                    seed\n                    + i * 1009\n                ),\n                shot_schedule=shot_schedule,\n                delta=QUANTUM_CONFIG[\n                    "delta"\n                ],\n                clip_epsilon=PPO_CONFIG[\n                    "clip_epsilon"\n                ],\n            )\n        )\n\n        decision = summary[\n            "decision"\n        ]\n\n        true_ratio = float(\n            summary[\n                "true_ratio"\n            ]\n        )\n\n        # ====================================================\n        # True PPO clipping state\n        # Evaluation only\n        # ====================================================\n\n        if advantage >= 0:\n\n            true_should_clip = (\n                true_ratio\n                > upper_clip\n            )\n\n        else:\n\n            true_should_clip = (\n                true_ratio\n                < lower_clip\n            )\n\n        if decision in {\n            "CONFIRMED_UPPER_CLIP",\n            "CONFIRMED_LOWER_CLIP",\n        }:\n\n            predicted_clip = True\n\n        elif decision in {\n            "SAFE_NO_UPPER_CLIP",\n            "SAFE_NO_LOWER_CLIP",\n        }:\n\n            predicted_clip = False\n\n        else:\n\n            predicted_clip = None\n\n        if predicted_clip is None:\n\n            classification_correct = np.nan\n\n        else:\n\n            classification_correct = (\n                predicted_clip\n                == true_should_clip\n            )\n\n        records.append(\n            {\n                "sample":\n                    i,\n\n                "advantage":\n                    advantage,\n\n                "true_ratio":\n                    true_ratio,\n\n                "decision":\n                    decision,\n\n                "resolved":\n                    summary[\n                        "resolved"\n                    ],\n\n                "shots_per_policy":\n                    summary[\n                        "shots_per_policy"\n                    ],\n\n                "total_shots":\n                    summary[\n                        "total_probability_shots"\n                    ],\n\n                "true_should_clip":\n                    true_should_clip,\n\n                "classification_correct":\n                    classification_correct,\n            }\n        )\n\n    # ========================================================\n    # 2. Aggregate certification statistics\n    # ========================================================\n\n    certification_df = pd.DataFrame(\n        records\n    )\n\n    unresolved_fraction = float(\n        1.0\n        - certification_df[\n            "resolved"\n        ].mean()\n    )\n\n    total_shots = int(\n        certification_df[\n            "total_shots"\n        ].sum()\n    )\n\n    mean_shots = float(\n        certification_df[\n            "total_shots"\n        ].mean()\n    )\n\n    median_shots = float(\n        certification_df[\n            "total_shots"\n        ].median()\n    )\n\n    resolved_rows = (\n        certification_df[\n            certification_df[\n                "classification_correct"\n            ].notna()\n        ]\n    )\n\n    if len(resolved_rows) > 0:\n\n        classification_accuracy = float(\n            resolved_rows[\n                "classification_correct"\n            ].mean()\n        )\n\n    else:\n\n        classification_accuracy = np.nan\n\n    return (\n        certification_df,\n\n        {\n            "unresolved_fraction":\n                unresolved_fraction,\n\n            "total_qtrust_shots":\n                total_shots,\n\n            "mean_qtrust_shots":\n                mean_shots,\n\n            "median_qtrust_shots":\n                median_shots,\n\n            "classification_accuracy":\n                classification_accuracy,\n        }\n    )\n\n# ===== SOURCE CELL 10 =====\n\n\n# ============================================================\n# EXACT-STEP QUANTUM TRAINERS + EVALUATION\n# ============================================================\n\nimport copy\nimport time\nimport pennylane as qml\n\n\ndef train_fixed_qppo_exact_steps(\n    env_name,\n    shots,\n    total_steps,\n    rollout_steps,\n    seed=MASTER_SEED,\n    update_epochs=1,\n):\n\n    assert total_steps % rollout_steps == 0\n\n    set_global_seed(seed)\n\n    env = make_env(\n        env_name,\n        seed=seed,\n    )\n\n    state_dim = ENV_CONFIG[\n        env_name\n    ]["state_dim"]\n\n    action_dim = ENV_CONFIG[\n        env_name\n    ]["action_dim"]\n\n    model = FixedShotHybridActorCritic(\n        state_dim=state_dim,\n        action_dim=action_dim,\n        shots=shots,\n    )\n\n    actor_optimizer = torch.optim.Adam(\n        model.actor.parameters(),\n        lr=PPO_CONFIG["actor_lr"],\n    )\n\n    critic_optimizer = torch.optim.Adam(\n        model.critic.parameters(),\n        lr=PPO_CONFIG["critic_lr"],\n    )\n\n    buffer = RolloutBuffer()\n\n    global_step = 0\n    update_number = 0\n    episode_number = 1\n\n    cumulative_action_shots = 0\n    cumulative_gradient_shots = 0\n\n    update_records = []\n\n    state, _ = env.reset(\n        seed=seed + episode_number\n    )\n\n    state = np.asarray(\n        state,\n        dtype=np.float32,\n    )\n\n    start_time = time.perf_counter()\n\n    # ========================================================\n    # Exact-step interaction\n    # ========================================================\n\n    while global_step < total_steps:\n\n        (\n            action,\n            log_prob,\n            value,\n            _\n        ) = model.act(state)\n\n        cumulative_action_shots += shots\n\n        (\n            next_state,\n            reward,\n            terminated,\n            truncated,\n            _\n        ) = env.step(action)\n\n        next_state = np.asarray(\n            next_state,\n            dtype=np.float32,\n        )\n\n        if terminated:\n\n            next_value = 0.0\n\n        else:\n\n            next_tensor = torch.tensor(\n                next_state,\n                dtype=torch.float32,\n                device=DEVICE,\n            ).unsqueeze(0)\n\n            with torch.no_grad():\n\n                next_value = float(\n                    model.critic(\n                        next_tensor\n                    ).item()\n                )\n\n        buffer.add(\n            state=state,\n            action=action,\n            log_prob=log_prob,\n            reward=reward,\n            value=value,\n            next_value=next_value,\n            terminated=terminated,\n            truncated=truncated,\n            shots=shots,\n        )\n\n        global_step += 1\n\n        # ----------------------------------------------------\n        # Exact rollout update\n        # ----------------------------------------------------\n\n        if len(buffer) == rollout_steps:\n\n            update_number += 1\n\n            tracker = qml.Tracker(\n                model.actor.qnode.device\n            )\n\n            with tracker:\n\n                metrics = quantum_ppo_update(\n                    model=model,\n                    buffer=buffer,\n                    actor_optimizer=actor_optimizer,\n                    critic_optimizer=critic_optimizer,\n                    update_epochs=update_epochs,\n                )\n\n            gradient_shots = int(\n                tracker.totals.get(\n                    "shots",\n                    0\n                )\n            )\n\n            gradient_executions = int(\n                tracker.totals.get(\n                    "executions",\n                    0\n                )\n            )\n\n            cumulative_gradient_shots += (\n                gradient_shots\n            )\n\n            metrics.update(\n                {\n                    "update":\n                        update_number,\n\n                    "global_step":\n                        global_step,\n\n                    "action_shots":\n                        cumulative_action_shots,\n\n                    "gradient_shots_update":\n                        gradient_shots,\n\n                    "gradient_executions":\n                        gradient_executions,\n\n                    "cumulative_gradient_shots":\n                        cumulative_gradient_shots,\n\n                    "cumulative_total_shots":\n                        (\n                            cumulative_action_shots\n                            + cumulative_gradient_shots\n                        ),\n                }\n            )\n\n            update_records.append(\n                metrics\n            )\n\n            buffer.clear()\n\n        # ----------------------------------------------------\n        # Episode transition\n        # ----------------------------------------------------\n\n        if terminated or truncated:\n\n            episode_number += 1\n\n            state, _ = env.reset(\n                seed=seed + episode_number\n            )\n\n            state = np.asarray(\n                state,\n                dtype=np.float32,\n            )\n\n        else:\n\n            state = next_state\n\n    elapsed = (\n        time.perf_counter()\n        - start_time\n    )\n\n    env.close()\n\n    return (\n        model,\n        pd.DataFrame(update_records),\n        {\n            "method":\n                f"Fixed-QPPO-{shots}",\n\n            "environment_steps":\n                global_step,\n\n            "ppo_updates":\n                update_number,\n\n            "action_shots":\n                cumulative_action_shots,\n\n            "gradient_shots":\n                cumulative_gradient_shots,\n\n            "certification_shots":\n                0,\n\n            "total_quantum_shots":\n                (\n                    cumulative_action_shots\n                    + cumulative_gradient_shots\n                ),\n\n            "elapsed_seconds":\n                elapsed,\n        }\n    )\n\ndef train_qtrust_exact_steps(\n    env_name,\n    base_shots,\n    total_steps,\n    rollout_steps,\n    seed=MASTER_SEED,\n    update_epochs=1,\n):\n\n    assert total_steps % rollout_steps == 0\n\n    set_global_seed(seed)\n\n    env = make_env(\n        env_name,\n        seed=seed,\n    )\n\n    state_dim = ENV_CONFIG[\n        env_name\n    ]["state_dim"]\n\n    action_dim = ENV_CONFIG[\n        env_name\n    ]["action_dim"]\n\n    model = FixedShotHybridActorCritic(\n        state_dim=state_dim,\n        action_dim=action_dim,\n        shots=base_shots,\n    )\n\n    actor_optimizer = torch.optim.Adam(\n        model.actor.parameters(),\n        lr=PPO_CONFIG["actor_lr"],\n    )\n\n    critic_optimizer = torch.optim.Adam(\n        model.critic.parameters(),\n        lr=PPO_CONFIG["critic_lr"],\n    )\n\n    buffer = RolloutBuffer()\n\n    global_step = 0\n    update_number = 0\n    episode_number = 1\n\n    accepted_updates = 0\n    rolled_back_updates = 0\n\n    cumulative_action_shots = 0\n    cumulative_gradient_shots = 0\n    cumulative_qtrust_shots = 0\n\n    update_records = []\n\n    state, _ = env.reset(\n        seed=seed + episode_number\n    )\n\n    state = np.asarray(\n        state,\n        dtype=np.float32,\n    )\n\n    start_time = time.perf_counter()\n\n    # ========================================================\n    # Exact-step interaction\n    # ========================================================\n\n    while global_step < total_steps:\n\n        (\n            action,\n            log_prob,\n            value,\n            _\n        ) = model.act(state)\n\n        cumulative_action_shots += (\n            base_shots\n        )\n\n        (\n            next_state,\n            reward,\n            terminated,\n            truncated,\n            _\n        ) = env.step(action)\n\n        next_state = np.asarray(\n            next_state,\n            dtype=np.float32,\n        )\n\n        if terminated:\n\n            next_value = 0.0\n\n        else:\n\n            next_tensor = torch.tensor(\n                next_state,\n                dtype=torch.float32,\n                device=DEVICE,\n            ).unsqueeze(0)\n\n            with torch.no_grad():\n\n                next_value = float(\n                    model.critic(\n                        next_tensor\n                    ).item()\n                )\n\n        buffer.add(\n            state=state,\n            action=action,\n            log_prob=log_prob,\n            reward=reward,\n            value=value,\n            next_value=next_value,\n            terminated=terminated,\n            truncated=truncated,\n            shots=base_shots,\n        )\n\n        global_step += 1\n\n        # ====================================================\n        # Exact rollout update\n        # ====================================================\n\n        if len(buffer) == rollout_steps:\n\n            update_number += 1\n\n            # -----------------------------------------------\n            # OLD policy snapshot\n            # -----------------------------------------------\n\n            old_shadow = make_analytic_shadow(\n                model.actor\n            )\n\n            old_weights = (\n                model.actor\n                .quantum_weights\n                .detach()\n                .clone()\n            )\n\n            old_optimizer_state = copy.deepcopy(\n                actor_optimizer.state_dict()\n            )\n\n            advantages, _ = (\n                get_buffer_advantages(\n                    buffer\n                )\n            )\n\n            # -----------------------------------------------\n            # Finite-shot parameter-shift PPO\n            # -----------------------------------------------\n\n            tracker = qml.Tracker(\n                model.actor.qnode.device\n            )\n\n            with tracker:\n\n                metrics = quantum_ppo_update(\n                    model=model,\n                    buffer=buffer,\n                    actor_optimizer=actor_optimizer,\n                    critic_optimizer=critic_optimizer,\n                    update_epochs=update_epochs,\n                )\n\n            gradient_shots = int(\n                tracker.totals.get(\n                    "shots",\n                    0\n                )\n            )\n\n            gradient_executions = int(\n                tracker.totals.get(\n                    "executions",\n                    0\n                )\n            )\n\n            cumulative_gradient_shots += (\n                gradient_shots\n            )\n\n            # -----------------------------------------------\n            # Proposed NEW policy\n            # -----------------------------------------------\n\n            new_shadow = make_analytic_shadow(\n                model.actor\n            )\n\n            # -----------------------------------------------\n            # Adaptive QTrust certification\n            # -----------------------------------------------\n\n            (\n                certification_df,\n                trust_metrics,\n            ) = certify_qtrust_update(\n                old_actor=old_shadow,\n                new_actor=new_shadow,\n                buffer=buffer,\n                advantages=advantages,\n                seed=(\n                    seed\n                    + update_number\n                    * 100_000\n                ),\n                shot_schedule=\n                    QTRUST_SHOT_SCHEDULE,\n            )\n\n            qtrust_shots = int(\n                trust_metrics[\n                    "total_qtrust_shots"\n                ]\n            )\n\n            cumulative_qtrust_shots += (\n                qtrust_shots\n            )\n\n            # -----------------------------------------------\n            # QTrust gate\n            # -----------------------------------------------\n\n            if (\n                trust_metrics[\n                    "unresolved_fraction"\n                ]\n                == 0.0\n            ):\n\n                accepted = True\n                accepted_updates += 1\n\n            else:\n\n                accepted = False\n                rolled_back_updates += 1\n\n                with torch.no_grad():\n\n                    model.actor.quantum_weights.copy_(\n                        old_weights\n                    )\n\n                actor_optimizer.load_state_dict(\n                    old_optimizer_state\n                )\n\n            metrics.update(\n                {\n                    "update":\n                        update_number,\n\n                    "global_step":\n                        global_step,\n\n                    "update_accepted":\n                        accepted,\n\n                    "unresolved_fraction":\n                        trust_metrics[\n                            "unresolved_fraction"\n                        ],\n\n                    "classification_accuracy":\n                        trust_metrics[\n                            "classification_accuracy"\n                        ],\n\n                    "mean_qtrust_shots":\n                        trust_metrics[\n                            "mean_qtrust_shots"\n                        ],\n\n                    "action_shots":\n                        cumulative_action_shots,\n\n                    "gradient_shots_update":\n                        gradient_shots,\n\n                    "gradient_executions":\n                        gradient_executions,\n\n                    "qtrust_shots_update":\n                        qtrust_shots,\n\n                    "cumulative_gradient_shots":\n                        cumulative_gradient_shots,\n\n                    "cumulative_qtrust_shots":\n                        cumulative_qtrust_shots,\n\n                    "cumulative_total_shots":\n                        (\n                            cumulative_action_shots\n                            + cumulative_gradient_shots\n                            + cumulative_qtrust_shots\n                        ),\n                }\n            )\n\n            update_records.append(\n                metrics\n            )\n\n            buffer.clear()\n\n        # ----------------------------------------------------\n        # Episode transition\n        # ----------------------------------------------------\n\n        if terminated or truncated:\n\n            episode_number += 1\n\n            state, _ = env.reset(\n                seed=seed + episode_number\n            )\n\n            state = np.asarray(\n                state,\n                dtype=np.float32,\n            )\n\n        else:\n\n            state = next_state\n\n    elapsed = (\n        time.perf_counter()\n        - start_time\n    )\n\n    env.close()\n\n    return (\n        model,\n        pd.DataFrame(update_records),\n        {\n            "method":\n                f"QTrust-PPO-{base_shots}",\n\n            "environment_steps":\n                global_step,\n\n            "ppo_updates":\n                update_number,\n\n            "accepted_updates":\n                accepted_updates,\n\n            "rolled_back_updates":\n                rolled_back_updates,\n\n            "action_shots":\n                cumulative_action_shots,\n\n            "gradient_shots":\n                cumulative_gradient_shots,\n\n            "certification_shots":\n                cumulative_qtrust_shots,\n\n            "total_quantum_shots":\n                (\n                    cumulative_action_shots\n                    + cumulative_gradient_shots\n                    + cumulative_qtrust_shots\n                ),\n\n            "elapsed_seconds":\n                elapsed,\n        }\n    )\n\ndef evaluate_finite_shot_policy(\n    model,\n    env_name,\n    seeds,\n    deterministic=True,\n):\n\n    actor = model.actor\n\n    if not hasattr(actor, "shots"):\n        raise ValueError(\n            "Evaluation requires a finite-shot actor."\n        )\n\n    actor_shots = int(\n        actor.shots\n    )\n\n    records = []\n\n    total_shots = 0\n\n    start_time = time.perf_counter()\n\n    for episode_idx, eval_seed in enumerate(\n        seeds,\n        start=1,\n    ):\n\n        env = make_env(\n            env_name,\n            seed=eval_seed,\n        )\n\n        state, _ = env.reset(\n            seed=eval_seed\n        )\n\n        state = np.asarray(\n            state,\n            dtype=np.float32,\n        )\n\n        terminated = False\n        truncated = False\n\n        reward_sum = 0.0\n        steps = 0\n\n        while not (\n            terminated or truncated\n        ):\n\n            # ------------------------------------------------\n            # Genuine finite-shot quantum measurement\n            # ------------------------------------------------\n\n            with torch.no_grad():\n\n                probs = actor.single_forward(\n                    state\n                )\n\n            total_shots += actor_shots\n\n            # ------------------------------------------------\n            # Evaluation action\n            # ------------------------------------------------\n\n            if deterministic:\n\n                action = int(\n                    torch.argmax(\n                        probs\n                    ).item()\n                )\n\n            else:\n\n                action = int(\n                    torch.distributions\n                    .Categorical(\n                        probs=probs\n                    )\n                    .sample()\n                    .item()\n                )\n\n            (\n                next_state,\n                reward,\n                terminated,\n                truncated,\n                _\n            ) = env.step(action)\n\n            state = np.asarray(\n                next_state,\n                dtype=np.float32,\n            )\n\n            reward_sum += float(\n                reward\n            )\n\n            steps += 1\n\n        env.close()\n\n        records.append(\n            {\n                "episode":\n                    episode_idx,\n\n                "seed":\n                    eval_seed,\n\n                "reward":\n                    reward_sum,\n\n                "steps":\n                    steps,\n\n                "actor_shots":\n                    steps * actor_shots,\n            }\n        )\n\n    elapsed = (\n        time.perf_counter()\n        - start_time\n    )\n\n    df = pd.DataFrame(\n        records\n    )\n\n    rewards = df[\n        "reward"\n    ].to_numpy()\n\n    n = len(rewards)\n\n    mean_reward = float(\n        np.mean(rewards)\n    )\n\n    std_reward = float(\n        np.std(\n            rewards,\n            ddof=1\n        )\n        if n > 1\n        else 0.0\n    )\n\n    sem = (\n        std_reward / np.sqrt(n)\n        if n > 1\n        else 0.0\n    )\n\n    summary = {\n        "episodes":\n            n,\n\n        "mean_reward":\n            mean_reward,\n\n        "std_reward":\n            std_reward,\n\n        "median_reward":\n            float(\n                np.median(\n                    rewards\n                )\n            ),\n\n        "min_reward":\n            float(\n                np.min(\n                    rewards\n                )\n            ),\n\n        "max_reward":\n            float(\n                np.max(\n                    rewards\n                )\n            ),\n\n        "ci95_low":\n            mean_reward\n            - 1.96 * sem,\n\n        "ci95_high":\n            mean_reward\n            + 1.96 * sem,\n\n        "mean_steps":\n            float(\n                df[\n                    "steps"\n                ].mean()\n            ),\n\n        "total_evaluation_shots":\n            int(total_shots),\n\n        "elapsed_seconds":\n            elapsed,\n\n        "deterministic":\n            deterministic,\n    }\n\n    return df, summary\n\n\n# ============================================================\n# CPU-SAFE CLASSICAL PPO BASELINE\n# ============================================================\n\n@torch.no_grad()\ndef classical_act_safe(model, state):\n    device = next(model.actor.parameters()).device\n    state_tensor = torch.as_tensor(\n        state, dtype=torch.float32, device=device\n    )\n    if state_tensor.ndim == 1:\n        state_tensor = state_tensor.unsqueeze(0)\n\n    logits = model.actor(state_tensor)\n    dist = Categorical(logits=logits)\n    action = dist.sample()\n    log_prob = dist.log_prob(action)\n    value = model.critic(state_tensor)\n\n    return (\n        int(action.item()),\n        float(log_prob.item()),\n        float(value.reshape(-1)[0].item()),\n    )\n\n\ndef classical_ppo_update_final(\n    model,\n    buffer,\n    actor_optimizer,\n    critic_optimizer,\n    update_epochs=1,\n):\n    assert update_epochs == 1\n\n    device = next(model.actor.parameters()).device\n\n    advantages, returns = compute_gae(\n        rewards=buffer.rewards,\n        values=buffer.values,\n        next_values=buffer.next_values,\n        terminateds=buffer.terminateds,\n        truncateds=buffer.truncateds,\n        gamma=PPO_CONFIG["gamma"],\n        gae_lambda=PPO_CONFIG["gae_lambda"],\n    )\n    advantages = normalize_advantages(advantages)\n\n    states = torch.as_tensor(\n        np.asarray(buffer.states, dtype=np.float32),\n        dtype=torch.float32,\n        device=device,\n    )\n    actions = torch.as_tensor(\n        buffer.actions, dtype=torch.long, device=device\n    )\n    old_log_probs = torch.as_tensor(\n        buffer.log_probs, dtype=torch.float32, device=device\n    )\n    advantages_t = torch.as_tensor(\n        advantages, dtype=torch.float32, device=device\n    )\n    returns_t = torch.as_tensor(\n        returns, dtype=torch.float32, device=device\n    )\n\n    n_samples = len(buffer)\n    batch_size = min(PPO_CONFIG["minibatch_size"], n_samples)\n\n    actor_losses = []\n    critic_losses = []\n    entropies = []\n    approx_kls = []\n    clip_fractions = []\n    actor_grad_norms = []\n    critic_grad_norms = []\n\n    for _ in range(update_epochs):\n        permutation = torch.randperm(n_samples, device=device)\n\n        for start in range(0, n_samples, batch_size):\n            idx = permutation[start:start + batch_size]\n\n            logits = model.actor(states[idx])\n            dist = Categorical(logits=logits)\n            new_log_probs = dist.log_prob(actions[idx])\n            entropy = dist.entropy().mean()\n            values = model.critic(states[idx]).reshape(-1)\n\n            log_ratio = new_log_probs - old_log_probs[idx]\n            ratio = torch.exp(log_ratio)\n\n            unclipped = ratio * advantages_t[idx]\n            clipped_ratio = torch.clamp(\n                ratio,\n                1.0 - PPO_CONFIG["clip_epsilon"],\n                1.0 + PPO_CONFIG["clip_epsilon"],\n            )\n            clipped = clipped_ratio * advantages_t[idx]\n\n            actor_loss = -torch.min(unclipped, clipped).mean()\n            critic_loss = F.mse_loss(values, returns_t[idx])\n\n            total_loss = (\n                actor_loss\n                + PPO_CONFIG["value_coef"] * critic_loss\n                - PPO_CONFIG["entropy_coef"] * entropy\n            )\n\n            actor_optimizer.zero_grad(set_to_none=True)\n            critic_optimizer.zero_grad(set_to_none=True)\n            total_loss.backward()\n\n            actor_grad_norm = torch.nn.utils.clip_grad_norm_(\n                model.actor.parameters(),\n                PPO_CONFIG["max_grad_norm"],\n            )\n            critic_grad_norm = torch.nn.utils.clip_grad_norm_(\n                model.critic.parameters(),\n                PPO_CONFIG["max_grad_norm"],\n            )\n\n            actor_optimizer.step()\n            critic_optimizer.step()\n\n            with torch.no_grad():\n                approx_kl = (\n                    ratio - 1.0 - log_ratio\n                ).mean()\n                clip_fraction = (\n                    (\n                        torch.abs(ratio - 1.0)\n                        > PPO_CONFIG["clip_epsilon"]\n                    )\n                    .float()\n                    .mean()\n                )\n\n            actor_losses.append(float(actor_loss.detach().cpu()))\n            critic_losses.append(float(critic_loss.detach().cpu()))\n            entropies.append(float(entropy.detach().cpu()))\n            approx_kls.append(float(approx_kl.detach().cpu()))\n            clip_fractions.append(float(clip_fraction.detach().cpu()))\n            actor_grad_norms.append(float(actor_grad_norm.detach().cpu()))\n            critic_grad_norms.append(float(critic_grad_norm.detach().cpu()))\n\n    return {\n        "actor_loss": float(np.mean(actor_losses)),\n        "critic_loss": float(np.mean(critic_losses)),\n        "entropy": float(np.mean(entropies)),\n        "approx_kl": float(np.mean(approx_kls)),\n        "clip_fraction": float(np.mean(clip_fractions)),\n        "actor_grad_norm": float(np.mean(actor_grad_norms)),\n        "critic_grad_norm": float(np.mean(critic_grad_norms)),\n        "rollout_size": int(n_samples),\n    }\n\n\ndef train_classical_ppo_exact_steps_final(\n    env_name,\n    total_steps,\n    rollout_steps,\n    seed,\n    update_epochs=1,\n):\n    assert total_steps % rollout_steps == 0\n    assert update_epochs == 1\n\n    set_global_seed(seed)\n    classical_device = torch.device("cpu")\n\n    env = make_env(env_name, seed=seed)\n    state_dim = ENV_CONFIG[env_name]["state_dim"]\n    action_dim = ENV_CONFIG[env_name]["action_dim"]\n\n    model = ClassicalActorCritic(\n        state_dim=state_dim,\n        action_dim=action_dim,\n    ).to(classical_device)\n\n    actor_optimizer = torch.optim.Adam(\n        model.actor.parameters(),\n        lr=PPO_CONFIG["actor_lr"],\n    )\n    critic_optimizer = torch.optim.Adam(\n        model.critic.parameters(),\n        lr=PPO_CONFIG["critic_lr"],\n    )\n\n    buffer = RolloutBuffer()\n    update_records = []\n\n    global_step = 0\n    update_number = 0\n    episode_number = 1\n\n    state, _ = env.reset(seed=seed + episode_number)\n    state = np.asarray(state, dtype=np.float32)\n\n    start_time = time.perf_counter()\n\n    while global_step < total_steps:\n        action, log_prob, value = classical_act_safe(model, state)\n\n        next_state, reward, terminated, truncated, _ = env.step(action)\n        next_state = np.asarray(next_state, dtype=np.float32)\n\n        if terminated:\n            next_value = 0.0\n        else:\n            next_tensor = torch.as_tensor(\n                next_state,\n                dtype=torch.float32,\n                device=classical_device,\n            ).unsqueeze(0)\n            with torch.no_grad():\n                next_value = float(\n                    model.critic(next_tensor).reshape(-1)[0].item()\n                )\n\n        buffer.add(\n            state=state,\n            action=action,\n            log_prob=log_prob,\n            reward=reward,\n            value=value,\n            next_value=next_value,\n            terminated=terminated,\n            truncated=truncated,\n            shots=0,\n        )\n\n        global_step += 1\n\n        if len(buffer) == rollout_steps:\n            update_number += 1\n            metrics = classical_ppo_update_final(\n                model=model,\n                buffer=buffer,\n                actor_optimizer=actor_optimizer,\n                critic_optimizer=critic_optimizer,\n                update_epochs=1,\n            )\n            metrics.update(\n                {"update": update_number, "global_step": global_step}\n            )\n            update_records.append(metrics)\n            buffer.clear()\n\n        if terminated or truncated:\n            episode_number += 1\n            state, _ = env.reset(seed=seed + episode_number)\n            state = np.asarray(state, dtype=np.float32)\n        else:\n            state = next_state\n\n    elapsed = time.perf_counter() - start_time\n    env.close()\n\n    assert update_number == total_steps // rollout_steps\n\n    summary = {\n        "environment_steps": global_step,\n        "ppo_updates": update_number,\n        "total_quantum_shots": 0,\n        "elapsed_seconds": elapsed,\n    }\n\n    return model, pd.DataFrame(update_records), summary\n\n\ndef evaluate_classical_final(model, env_name, seeds):\n    device = next(model.actor.parameters()).device\n    records = []\n    start_time = time.perf_counter()\n\n    for episode_idx, eval_seed in enumerate(seeds, start=1):\n        env = make_env(env_name, seed=eval_seed)\n        state, _ = env.reset(seed=eval_seed)\n        state = np.asarray(state, dtype=np.float32)\n\n        terminated = False\n        truncated = False\n        reward_sum = 0.0\n        steps = 0\n\n        while not (terminated or truncated):\n            state_tensor = torch.as_tensor(\n                state, dtype=torch.float32, device=device\n            ).unsqueeze(0)\n\n            with torch.no_grad():\n                logits = model.actor(state_tensor)\n                action = int(torch.argmax(logits, dim=-1).item())\n\n            state, reward, terminated, truncated, _ = env.step(action)\n            state = np.asarray(state, dtype=np.float32)\n            reward_sum += float(reward)\n            steps += 1\n\n        env.close()\n\n        records.append(\n            {\n                "episode": episode_idx,\n                "seed": eval_seed,\n                "reward": reward_sum,\n                "steps": steps,\n            }\n        )\n\n    elapsed = time.perf_counter() - start_time\n    df = pd.DataFrame(records)\n    rewards = df["reward"].to_numpy(dtype=np.float64)\n    n = len(rewards)\n\n    mean_reward = float(np.mean(rewards))\n    std_reward = float(np.std(rewards, ddof=1)) if n > 1 else 0.0\n    sem = std_reward / np.sqrt(n) if n > 1 else 0.0\n\n    return df, {\n        "evaluation_episodes": n,\n        "mean_eval_reward": mean_reward,\n        "std_eval_reward": std_reward,\n        "median_eval_reward": float(np.median(rewards)),\n        "min_eval_reward": float(np.min(rewards)),\n        "max_eval_reward": float(np.max(rewards)),\n        "eval_ci95_low": float(mean_reward - 1.96 * sem),\n        "eval_ci95_high": float(mean_reward + 1.96 * sem),\n        "mean_eval_steps": float(df["steps"].mean()),\n        "evaluation_quantum_shots": 0,\n        "evaluation_runtime_seconds": elapsed,\n    }\n\n\n# ===== SOURCE CELL 15 =====\n\n\n# ============================================================\n# FROZEN PUBLICATION PROTOCOL\n# ============================================================\n#\n# The final paper uses the original 10 final seeds.\n# Publication execution is locked to FINAL_10.\n#\n# IMPORTANT: the protocol hash deliberately excludes the seed\n# list, so PRIMARY_5 runs can be reused when extending to the\n# exact same FINAL_10 experiment.\n# ============================================================\n\nimport json\nimport hashlib\nimport pandas as pd\n\nRUN_SCOPE = "FINAL_10"    # "PRIMARY_5" or "FINAL_10"\nBATCH_SIZE = 120          # Run All: execute every frozen FINAL_10 run\n\nFINAL_ENVIRONMENTS = {\n    "CartPole-v1": {\n        "training_steps": 2048,\n        "rollout_steps": 32,\n        "evaluation_episodes": 30,\n    },\n    "Acrobot-v1": {\n        "training_steps": 4096,\n        "rollout_steps": 32,\n        "evaluation_episodes": 30,\n    },\n    "LunarLander-v3": {\n        "training_steps": 4096,\n        "rollout_steps": 32,\n        "evaluation_episodes": 30,\n    },\n}\n\nFINAL_METHODS = [\n    "Classical-PPO",\n    "Fixed-QPPO-128",\n    "Fixed-QPPO-256",\n    "QTrust-PPO-128",\n]\n\nACTIVE_SEEDS = (\n    DEV_SEEDS\n    if RUN_SCOPE == "PRIMARY_5"\n    else FINAL_SEEDS\n)\n\ndef make_final_eval_seeds(training_seed, n_episodes):\n    base = 100_000 + int(training_seed) * 100\n    return list(range(base, base + int(n_episodes)))\n\nALGORITHM_PROTOCOL = {\n    "paper": "QTrust-PPO",\n    "environments": FINAL_ENVIRONMENTS,\n    "methods": FINAL_METHODS,\n    "ppo": PPO_CONFIG,\n    "quantum": {\n        "qubits": QUANTUM_CONFIG["main_qubits"],\n        "layers": QUANTUM_CONFIG["main_layers"],\n        "fixed_main_shots": [128, 256],\n        "qtrust_base_shots": 128,\n        "qtrust_schedule": QTRUST_SHOT_SCHEDULE,\n        "delta": QUANTUM_CONFIG["delta"],\n        "confidence_rule": (\n            "finite-horizon Bonferroni with four one-sided "\n            "Clopper-Pearson tails per stage"\n        ),\n        "decision_rule": "advantage-aware PPO clipping",\n        "rollback_on_unresolved": True,\n    },\n    "evaluation": {\n        "episodes_per_training_seed": 30,\n        "deterministic_action_rule": (\n            "argmax of finite-shot estimated action probabilities"\n        ),\n    },\n    "resource_accounting": [\n        "action-selection shots",\n        "parameter-shift gradient shots",\n        "QTrust certification shots",\n    ],\n}\n\nPROTOCOL_HASH = hashlib.sha256(\n    json.dumps(\n        ALGORITHM_PROTOCOL,\n        sort_keys=True,\n        default=str,\n    ).encode("utf-8")\n).hexdigest()[:16]\n\nPROTOCOL_BUNDLE = {\n    **ALGORITHM_PROTOCOL,\n    "run_scope": RUN_SCOPE,\n    "active_seeds": ACTIVE_SEEDS,\n    "all_final_seeds": FINAL_SEEDS,\n    "protocol_hash": PROTOCOL_HASH,\n}\n\nmanifest_rows = []\n\nfor env_name, env_cfg in FINAL_ENVIRONMENTS.items():\n    for method in FINAL_METHODS:\n        for seed in ACTIVE_SEEDS:\n            manifest_rows.append(\n                {\n                    "environment": env_name,\n                    "method": method,\n                    "seed": int(seed),\n                    "training_steps": int(env_cfg["training_steps"]),\n                    "rollout_steps": int(env_cfg["rollout_steps"]),\n                    "evaluation_episodes": int(\n                        env_cfg["evaluation_episodes"]\n                    ),\n                    "update_epochs": 1,\n                    "protocol_hash": PROTOCOL_HASH,\n                    "status": "PENDING",\n                }\n            )\n\nFINAL_EXPERIMENT_MANIFEST = pd.DataFrame(manifest_rows)\n\nprint("=" * 78)\nprint("QTrust-PPO FROZEN PUBLICATION PROTOCOL")\nprint("=" * 78)\nprint("Run scope               :", RUN_SCOPE)\nprint("Training seeds          :", ACTIVE_SEEDS)\nprint("Main runs               :", len(FINAL_EXPERIMENT_MANIFEST))\nprint("Batch size              :", BATCH_SIZE)\nprint("Protocol hash           :", PROTOCOL_HASH)\nprint("QTrust schedule         :", QTRUST_SHOT_SCHEDULE)\nprint(\n    "Per-tail stage alpha    :",\n    QUANTUM_CONFIG["delta"]\n    / len(QTRUST_SHOT_SCHEDULE)\n    / 4.0,\n)\nprint("=" * 78)\n\n\n# ============================================================\n# Worker result paths\n# ============================================================\n\nWORK_ROOT = (\n    Path("/kaggle/working")\n    if Path("/kaggle/working").exists()\n    else Path.cwd()\n)\n\nRESULT_ROOT = WORK_ROOT / "qtrust_final_results"\nSUMMARY_DIR = RESULT_ROOT / "summaries"\nUPDATE_DIR = RESULT_ROOT / "updates"\nEVAL_DIR = RESULT_ROOT / "evaluations"\nMODEL_DIR = RESULT_ROOT / "models"\nERROR_DIR = RESULT_ROOT / "errors"\n\nfor _directory in [\n    RESULT_ROOT,\n    SUMMARY_DIR,\n    UPDATE_DIR,\n    EVAL_DIR,\n    MODEL_DIR,\n    ERROR_DIR,\n]:\n    _directory.mkdir(\n        parents=True,\n        exist_ok=True,\n    )\n\ndef safe_run_id(environment, method, seed):\n    return (\n        f"{environment}__{method}__seed{int(seed)}"\n        .replace("/", "_")\n    )\n\ndef _python_value(value):\n    if isinstance(value, np.generic):\n        return value.item()\n    if isinstance(value, np.ndarray):\n        return value.tolist()\n    return value\n\ndef _json_safe(dictionary):\n    return {\n        key: _python_value(value)\n        for key, value in dictionary.items()\n    }\n\ndef save_model_state(model, method, filepath):\n    checkpoint = {\n        "method": method,\n        "actor_state_dict": {\n            key: value.detach().cpu()\n            for key, value\n            in model.actor.state_dict().items()\n        },\n        "critic_state_dict": {\n            key: value.detach().cpu()\n            for key, value\n            in model.critic.state_dict().items()\n        },\n    }\n    torch.save(checkpoint, filepath)\n\ndef execute_final_run(row):\n    env_name = row["environment"]\n    method = row["method"]\n    seed = int(row["seed"])\n    total_steps = int(row["training_steps"])\n    rollout_steps = int(row["rollout_steps"])\n    eval_episodes = int(row["evaluation_episodes"])\n\n    run_id = safe_run_id(env_name, method, seed)\n\n    summary_path = SUMMARY_DIR / f"{run_id}.json"\n    update_path = UPDATE_DIR / f"{run_id}.csv"\n    eval_path = EVAL_DIR / f"{run_id}.csv"\n    model_path = MODEL_DIR / f"{run_id}.pt"\n\n    if summary_path.exists():\n        try:\n            with open(summary_path, "r") as f:\n                saved = json.load(f)\n\n            if (\n                saved.get("status") == "COMPLETE"\n                and saved.get("protocol_hash") == PROTOCOL_HASH\n            ):\n                return saved, True\n        except Exception:\n            pass\n\n    print("=" * 78)\n    print(\n        f"RUNNING | {env_name} | {method} | seed={seed}"\n    )\n    print("=" * 78)\n\n    run_start = time.perf_counter()\n\n    # --------------------------------------------------------\n    # Training\n    # --------------------------------------------------------\n    if method == "Classical-PPO":\n        model, updates_df, train_summary = (\n            train_classical_ppo_exact_steps_final(\n                env_name=env_name,\n                total_steps=total_steps,\n                rollout_steps=rollout_steps,\n                seed=seed,\n                update_epochs=1,\n            )\n        )\n\n    elif method == "Fixed-QPPO-128":\n        model, updates_df, train_summary = (\n            train_fixed_qppo_exact_steps(\n                env_name=env_name,\n                shots=128,\n                total_steps=total_steps,\n                rollout_steps=rollout_steps,\n                seed=seed,\n                update_epochs=1,\n            )\n        )\n\n    elif method == "Fixed-QPPO-256":\n        model, updates_df, train_summary = (\n            train_fixed_qppo_exact_steps(\n                env_name=env_name,\n                shots=256,\n                total_steps=total_steps,\n                rollout_steps=rollout_steps,\n                seed=seed,\n                update_epochs=1,\n            )\n        )\n\n    elif method == "QTrust-PPO-128":\n        model, updates_df, train_summary = (\n            train_qtrust_exact_steps(\n                env_name=env_name,\n                base_shots=128,\n                total_steps=total_steps,\n                rollout_steps=rollout_steps,\n                seed=seed,\n                update_epochs=1,\n            )\n        )\n\n    else:\n        raise ValueError(f"Unknown method: {method}")\n\n    # --------------------------------------------------------\n    # Evaluation\n    # --------------------------------------------------------\n    eval_seeds = make_final_eval_seeds(\n        training_seed=seed,\n        n_episodes=eval_episodes,\n    )\n\n    if method == "Classical-PPO":\n        eval_df, eval_summary = evaluate_classical_final(\n            model=model,\n            env_name=env_name,\n            seeds=eval_seeds,\n        )\n    else:\n        eval_df, qeval = evaluate_finite_shot_policy(\n            model=model,\n            env_name=env_name,\n            seeds=eval_seeds,\n            deterministic=True,\n        )\n\n        eval_summary = {\n            "evaluation_episodes": qeval["episodes"],\n            "mean_eval_reward": qeval["mean_reward"],\n            "std_eval_reward": qeval["std_reward"],\n            "median_eval_reward": qeval["median_reward"],\n            "min_eval_reward": qeval["min_reward"],\n            "max_eval_reward": qeval["max_reward"],\n            "eval_ci95_low": qeval["ci95_low"],\n            "eval_ci95_high": qeval["ci95_high"],\n            "mean_eval_steps": qeval["mean_steps"],\n            "evaluation_quantum_shots": (\n                qeval["total_evaluation_shots"]\n            ),\n            "evaluation_runtime_seconds": (\n                qeval["elapsed_seconds"]\n            ),\n        }\n\n    # --------------------------------------------------------\n    # Update-level diagnostics\n    # --------------------------------------------------------\n    mean_kl = (\n        float(updates_df["approx_kl"].mean())\n        if (\n            len(updates_df) > 0\n            and "approx_kl" in updates_df.columns\n        )\n        else np.nan\n    )\n\n    mean_clip_fraction = (\n        float(updates_df["clip_fraction"].mean())\n        if (\n            len(updates_df) > 0\n            and "clip_fraction" in updates_df.columns\n        )\n        else np.nan\n    )\n\n    if method == "QTrust-PPO-128":\n        mean_unresolved = float(\n            updates_df["unresolved_fraction"].mean()\n        )\n        class_values = (\n            updates_df["classification_accuracy"]\n            .dropna()\n        )\n        classification_accuracy = (\n            float(class_values.mean())\n            if len(class_values) > 0\n            else np.nan\n        )\n        mean_adaptive_shots = float(\n            updates_df["mean_qtrust_shots"].mean()\n        )\n        accepted_updates = int(\n            train_summary["accepted_updates"]\n        )\n        rolled_back_updates = int(\n            train_summary["rolled_back_updates"]\n        )\n        action_shots = int(\n            train_summary.get("action_shots", 0)\n        )\n        gradient_shots = int(\n            train_summary.get("gradient_shots", 0)\n        )\n        certification_shots = int(\n            train_summary.get("certification_shots", 0)\n        )\n    else:\n        mean_unresolved = np.nan\n        classification_accuracy = np.nan\n        mean_adaptive_shots = np.nan\n        accepted_updates = int(\n            train_summary["ppo_updates"]\n        )\n        rolled_back_updates = 0\n        action_shots = int(\n            train_summary.get("action_shots", 0)\n        )\n        gradient_shots = int(\n            train_summary.get("gradient_shots", 0)\n        )\n        certification_shots = 0\n\n    training_quantum_shots = int(\n        train_summary.get("total_quantum_shots", 0)\n    )\n    evaluation_quantum_shots = int(\n        eval_summary["evaluation_quantum_shots"]\n    )\n\n    final_summary = {\n        "status": "COMPLETE",\n        "protocol_hash": PROTOCOL_HASH,\n        "run_scope": RUN_SCOPE,\n        "run_id": run_id,\n        "environment": env_name,\n        "method": method,\n        "seed": seed,\n        "training_steps": total_steps,\n        "rollout_steps": rollout_steps,\n        "ppo_updates": int(train_summary["ppo_updates"]),\n        "evaluation_episodes": eval_episodes,\n\n        "mean_eval_reward": float(\n            eval_summary["mean_eval_reward"]\n        ),\n        "std_eval_reward": float(\n            eval_summary["std_eval_reward"]\n        ),\n        "median_eval_reward": float(\n            eval_summary["median_eval_reward"]\n        ),\n        "min_eval_reward": float(\n            eval_summary["min_eval_reward"]\n        ),\n        "max_eval_reward": float(\n            eval_summary["max_eval_reward"]\n        ),\n        "eval_ci95_low": float(\n            eval_summary["eval_ci95_low"]\n        ),\n        "eval_ci95_high": float(\n            eval_summary["eval_ci95_high"]\n        ),\n        "mean_eval_steps": float(\n            eval_summary["mean_eval_steps"]\n        ),\n\n        "mean_approx_kl": mean_kl,\n        "mean_clip_fraction": mean_clip_fraction,\n\n        "accepted_updates": accepted_updates,\n        "rolled_back_updates": rolled_back_updates,\n        "mean_unresolved_fraction": mean_unresolved,\n        "classification_accuracy": classification_accuracy,\n        "mean_adaptive_shots": mean_adaptive_shots,\n\n        "action_shots": action_shots,\n        "gradient_shots": gradient_shots,\n        "certification_shots": certification_shots,\n        "training_quantum_shots": training_quantum_shots,\n        "evaluation_quantum_shots": evaluation_quantum_shots,\n        "overall_quantum_shots": (\n            training_quantum_shots\n            + evaluation_quantum_shots\n        ),\n\n        "training_runtime_seconds": float(\n            train_summary["elapsed_seconds"]\n        ),\n        "evaluation_runtime_seconds": float(\n            eval_summary["evaluation_runtime_seconds"]\n        ),\n        "wall_runtime_seconds": float(\n            time.perf_counter() - run_start\n        ),\n    }\n\n    # Save raw traces first.\n    updates_df.to_csv(update_path, index=False)\n    eval_df.to_csv(eval_path, index=False)\n    save_model_state(\n        model=model,\n        method=method,\n        filepath=model_path,\n    )\n\n    # Atomic completion marker last.\n    temp_path = SUMMARY_DIR / f"{run_id}.tmp"\n    with open(temp_path, "w") as f:\n        json.dump(\n            _json_safe(final_summary),\n            f,\n            indent=2,\n            allow_nan=True,\n        )\n    os.replace(temp_path, summary_path)\n\n    print(\n        f"COMPLETE | reward="\n        f"{final_summary[\'mean_eval_reward\']:.3f} | "\n        f"training shots="\n        f"{training_quantum_shots:,}"\n    )\n\n    return final_summary, False\n\n# ============================================================\n# DUAL-T4 WORKER ENTRYPOINT\n# Execution-only wrapper. Scientific protocol is unchanged.\n# ============================================================\n\nimport argparse\nimport traceback\nfrom pathlib import Path\n\nparser = argparse.ArgumentParser()\nparser.add_argument("--tasks", required=True)\nparser.add_argument("--worker-id", type=int, required=True)\nparser.add_argument("--protocol-hash", required=True)\nargs = parser.parse_args()\n\nif PROTOCOL_HASH != args.protocol_hash:\n    raise RuntimeError(\n        f"Protocol hash mismatch: worker={PROTOCOL_HASH}, parent={args.protocol_hash}"\n    )\n\n# The parent launches each worker with one physical GPU exposed.\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA is not available inside GPU worker.")\n\nif torch.cuda.device_count() != 1:\n    raise RuntimeError(\n        f"GPU worker must see exactly one CUDA device; saw {torch.cuda.device_count()}."\n    )\n\nif ANALYTIC_QDEVICE_NAME != "lightning.gpu":\n    raise RuntimeError(\n        f"Expected lightning.gpu, got {ANALYTIC_QDEVICE_NAME!r}."\n    )\n\nprint("=" * 78, flush=True)\nprint(\n    f"WORKER {args.worker_id} | visible GPU: "\n    f"{torch.cuda.get_device_name(0)} | protocol={PROTOCOL_HASH}",\n    flush=True,\n)\nprint("=" * 78, flush=True)\n\n# Finite-shot parameter-shift preflight on this specific GPU.\nset_global_seed(88000 + args.worker_id)\n\n_smoke_actor = FixedShotQuantumActor(\n    state_dim=4,\n    action_dim=2,\n    shots=32,\n)\n\n_smoke_probs = _smoke_actor.single_forward(\n    np.zeros(4, dtype=np.float32)\n)\n\n_smoke_loss = -torch.log(\n    _smoke_probs[0] + 1e-12\n)\n\n_smoke_loss.backward()\n\nif _smoke_actor.quantum_weights.grad is None:\n    raise RuntimeError("Finite-shot parameter-shift gradient is missing.")\n\nif not torch.isfinite(\n    _smoke_actor.quantum_weights.grad\n).all():\n    raise RuntimeError("Finite-shot parameter-shift gradient is non-finite.")\n\ndel _smoke_actor\ndel _smoke_probs\ndel _smoke_loss\n\ntry:\n    torch.cuda.empty_cache()\nexcept Exception:\n    pass\n\nprint(\n    f"WORKER {args.worker_id} PREFLIGHT : PASS",\n    flush=True,\n)\n\nwith open(args.tasks, "r") as f:\n    task_records = json.load(f)\n\ntasks_df = pd.DataFrame(task_records)\n\nfor _, row in tasks_df.iterrows():\n    run_id = safe_run_id(\n        row["environment"],\n        row["method"],\n        row["seed"],\n    )\n\n    try:\n        execute_final_run(row)\n\n    except Exception:\n        ERROR_DIR.mkdir(\n            parents=True,\n            exist_ok=True,\n        )\n\n        with open(\n            ERROR_DIR / f"{run_id}.worker{args.worker_id}.txt",\n            "w",\n        ) as f:\n            f.write(traceback.format_exc())\n\n        print(\n            f"WORKER {args.worker_id} FAILED | {run_id}",\n            flush=True,\n        )\n\n        traceback.print_exc()\n        raise\n\nprint(\n    f"WORKER {args.worker_id} COMPLETE | tasks={len(tasks_df)}",\n    flush=True,\n)\n'

worker_path = (
    WORK_ROOT
    / "qtrust_dual_t4_worker.py"
)

temporary_worker = (
    WORK_ROOT
    / "_qtrust_dual_t4_worker.tmp"
)

with open(
    temporary_worker,
    "w",
    encoding="utf-8",
) as f:
    f.write(
        WORKER_SOURCE
    )

os.replace(
    temporary_worker,
    worker_path,
)

# Syntax-check the exact worker file before spending GPU hours.
with open(
    worker_path,
    "r",
    encoding="utf-8",
) as f:
    compile(
        f.read(),
        str(worker_path),
        "exec",
    )

print(
    "Worker script syntax        : PASS"
)



# Task Balancing


def get_pending_manifest():
    completed = (
        load_completed_summaries()
    )

    completed_ids = (
        set(
            completed["run_id"]
        )
        if len(completed) > 0
        else set()
    )

    manifest = (
        FINAL_EXPERIMENT_MANIFEST.copy()
    )

    manifest[
        "run_id"
    ] = manifest.apply(
        lambda row:
            safe_run_id(
                row["environment"],
                row["method"],
                row["seed"],
            ),
        axis=1,
    )

    return manifest[
        ~manifest[
            "run_id"
        ].isin(
            completed_ids
        )
    ].copy()


METHOD_EXECUTION_WEIGHT = {
    # Classical PPO is CPU-only and much lighter.
    "Classical-PPO":
        0.05,

    
    "Fixed-QPPO-128":
        1.00,

    "Fixed-QPPO-256":
        1.00,

    "QTrust-PPO-128":
        1.08,
}


def split_pending_across_two_gpus(
    pending,
):
    """
    Greedy execution-only balancing.

    The weight changes ONLY scheduling order. It does not alter
    any experiment, seed, environment, training horizon, shot
    count, or algorithm.
    """

    records = (
        pending
        .to_dict(
            orient="records"
        )
    )

    for record in records:

        record[
            "_execution_weight"
        ] = (
            float(
                record[
                    "training_steps"
                ]
            )
            *
            METHOD_EXECUTION_WEIGHT[
                record[
                    "method"
                ]
            ]
        )

    records = sorted(
        records,
        key=lambda r:
            r[
                "_execution_weight"
            ],
        reverse=True,
    )

    assignments = [
        [],
        [],
    ]

    loads = [
        0.0,
        0.0,
    ]

    for record in records:

        gpu_id = int(
            np.argmin(
                loads
            )
        )

        loads[
            gpu_id
        ] += record.pop(
            "_execution_weight"
        )

        assignments[
            gpu_id
        ].append(
            record
        )

    return (
        assignments,
        loads,
    )


def tail_file(
    path,
    n_lines=25,
):
    if not path.exists():
        return ""

    try:
        with open(
            path,
            "r",
            errors="replace",
        ) as f:
            lines = f.readlines()

        return "".join(
            lines[
                -n_lines:
            ]
        )

    except Exception:
        return ""



# Run Pending Jobs With Fresh-Process Recovery


total_runs = len(
    FINAL_EXPERIMENT_MANIFEST
)

completed_start = len(
    load_completed_summaries()
)

print(
    f"Completed before dual run : "
    f"{completed_start} / {total_runs}"
)

print(
    "Resume policy              : "
    "completed runs are skipped; only an interrupted active run restarts"
)

restart_round = 0
last_checkpoint_count = (
    completed_start
)

while True:

    pending = (
        get_pending_manifest()
    )

    if len(
        pending
    ) == 0:
        break

    if (
        restart_round
        > MAX_RESTART_ROUNDS
    ):
        numbered_zip, latest_zip = (
            create_portable_checkpoint()
        )

        raise RuntimeError(
            "Dual-GPU workers exceeded the allowed fresh-process "
            "restart rounds. Completed runs are checkpointed at "
            f"{latest_zip}."
        )

    assignments, loads = (
        split_pending_across_two_gpus(
            pending
        )
    )

    print()
    print(
        "=" * 78
    )

    print(
        f"DUAL-T4 ROUND {restart_round + 1}"
    )

    print(
        f"Pending runs : {len(pending)}"
    )

    print(
        f"GPU 0 tasks  : "
        f"{len(assignments[0])}"
    )

    print(
        f"GPU 1 tasks  : "
        f"{len(assignments[1])}"
    )

    print(
        "Balanced execution weights:",
        [
            round(
                value,
                1,
            )
            for value in loads
        ],
    )

    print(
        "=" * 78
    )

    processes = []
    log_handles = []
    log_paths = []

    try:

        for gpu_id in range(
            DUAL_GPU_WORKERS
        ):

            task_path = (
                WORK_ROOT
                / (
                    f"qtrust_tasks_"
                    f"round{restart_round + 1}"
                    f"_gpu{gpu_id}.json"
                )
            )

            with open(
                task_path,
                "w",
            ) as f:
                json.dump(
                    assignments[
                        gpu_id
                    ],
                    f,
                    indent=2,
                    default=str,
                )

            log_path = (
                LOG_DIR
                / (
                    f"round{restart_round + 1}"
                    f"_gpu{gpu_id}.log"
                )
            )

            log_handle = open(
                log_path,
                "w",
                buffering=1,
            )

            env = (
                os.environ.copy()
            )

            # Critical: each worker sees ONE different physical T4
            # as its local cuda:0 before importing Torch/PennyLane.
            env[
                "CUDA_DEVICE_ORDER"
            ] = "PCI_BUS_ID"

            env[
                "CUDA_VISIBLE_DEVICES"
            ] = str(
                gpu_id
            )

            env[
                "PYTHONUNBUFFERED"
            ] = "1"

            env[
                "PYTHONHASHSEED"
            ] = str(
                MASTER_SEED
            )

            # Avoid CPU thread oversubscription between workers.
            for name in [
                "OMP_NUM_THREADS",
                "MKL_NUM_THREADS",
                "OPENBLAS_NUM_THREADS",
                "NUMEXPR_NUM_THREADS",
            ]:
                env[
                    name
                ] = "1"

            process = subprocess.Popen(
                [
                    sys.executable,
                    "-u",
                    str(
                        worker_path
                    ),
                    "--tasks",
                    str(
                        task_path
                    ),
                    "--worker-id",
                    str(
                        gpu_id
                    ),
                    "--protocol-hash",
                    PROTOCOL_HASH,
                ],
                stdout=
                    log_handle,

                stderr=
                    subprocess.STDOUT,

                env=
                    env,

                cwd=
                    str(
                        WORK_ROOT
                    ),
            )

            processes.append(
                process
            )

            log_handles.append(
                log_handle
            )

            log_paths.append(
                log_path
            )

        print(
            "Both isolated GPU workers launched."
        )

        last_reported_count = len(
            load_completed_summaries()
        )

        worker_failed = False

        while True:

            return_codes = [
                process.poll()
                for process
                in processes
            ]

            completed_now = len(
                load_completed_summaries()
            )

            if (
                completed_now
                != last_reported_count
            ):

                percent_now = (
                    100.0
                    * completed_now
                    / max(total_runs, 1)
                )

                print(
                    f"Progress : "
                    f"{completed_now} / {total_runs} "
                    f"frozen runs complete "
                    f"({percent_now:.1f}%)"
                )

                last_reported_count = (
                    completed_now
                )

            if (
                completed_now
                - last_checkpoint_count
                >= CHECKPOINT_EVERY_COMPLETIONS
            ):

                _, latest_zip = (
                    create_portable_checkpoint()
                )

                last_checkpoint_count = (
                    completed_now
                )

                print(
                    "Rolling checkpoint :",
                    latest_zip.name,
                )

                # Explicit live checkpoint status after every completed run.
                show_checkpoint_view(
                    detailed=False
                )

            failed_ids = [
                idx
                for idx, code
                in enumerate(
                    return_codes
                )
                if (
                    code is not None
                    and code != 0
                )
            ]

            if failed_ids:

                worker_failed = True

                print(
                    "Worker failure detected; "
                    "ending this round and relaunching "
                    "remaining runs in fresh processes."
                )

                for process in processes:
                    if (
                        process.poll()
                        is None
                    ):
                        process.terminate()

                break

            if all(
                code is not None
                for code
                in return_codes
            ):
                break

            time.sleep(
                MONITOR_SECONDS
            )

        # Ensure all processes exit.
        for process in processes:

            try:
                process.wait(
                    timeout=30
                )

            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()

    except KeyboardInterrupt:

        for process in processes:

            if (
                process.poll()
                is None
            ):
                process.terminate()

        for process in processes:

            try:
                process.wait(
                    timeout=20
                )

            except Exception:
                try:
                    process.kill()
                except Exception:
                    pass

        _, latest_zip = (
            create_portable_checkpoint()
        )

        print(
            "Manual interruption checkpoint:",
            latest_zip,
        )

        raise

    finally:

        for handle in log_handles:

            try:
                handle.close()
            except Exception:
                pass

    # Always checkpoint the completed work at round boundary.
    _, latest_zip = (
        create_portable_checkpoint()
    )

    completed_after_round = len(
        load_completed_summaries()
    )

    last_checkpoint_count = (
        completed_after_round
    )

    print(
        f"Round completion : "
        f"{completed_after_round} / {total_runs}"
    )

    print(
        "Checkpoint       :",
        latest_zip,
    )

    show_checkpoint_view(
        detailed=True
    )

    # Show diagnostics only if a worker failed.
    round_codes = [
        process.returncode
        for process
        in processes
    ]

    if any(
        code != 0
        for code
        in round_codes
    ):

        for gpu_id, path in enumerate(
            log_paths
        ):

            print()
            print(
                f"--- GPU {gpu_id} log tail ---"
            )

            print(
                tail_file(
                    path
                )
            )

        restart_round += 1

        # Recompute pending runs and redistribute them over both T4s.
        continue

    # Both workers ended cleanly. Recheck pending state.
    if len(
        get_pending_manifest()
    ) == 0:
        break

    # Defensive fresh-process retry if a clean exit somehow left
    # pending runs.
    restart_round += 1






FINAL_RUN_RESULTS = (
    load_completed_summaries()
)

if len(
    FINAL_RUN_RESULTS
) > 0:

    FINAL_RUN_RESULTS = (
        FINAL_RUN_RESULTS
        .sort_values(
            [
                "environment",
                "method",
                "seed",
            ]
        )
        .reset_index(
            drop=True
        )
    )

persist_aggregate_files()

numbered_zip, latest_zip = (
    create_portable_checkpoint()
)

expected_runs = len(
    FINAL_EXPERIMENT_MANIFEST
)

if len(
    FINAL_RUN_RESULTS
) != expected_runs:

    raise RuntimeError(
        "Publication sweep incomplete after dual-GPU execution: "
        f"{len(FINAL_RUN_RESULTS)} / {expected_runs}. "
        f"Checkpoint: {latest_zip}"
    )

group_counts = (
    FINAL_RUN_RESULTS
    .groupby(
        [
            "environment",
            "method",
        ]
    )
    .size()
)

if (
    len(group_counts)
    != (
        len(
            FINAL_ENVIRONMENTS
        )
        * len(
            FINAL_METHODS
        )
    )
    or
    not group_counts.eq(
        len(
            ACTIVE_SEEDS
        )
    ).all()
):

    raise RuntimeError(
        "Frozen manifest integrity check failed."
    )

show_checkpoint_view(
    detailed=True
)

print()
print(
    "=" * 78
)

print(
    "DUAL-T4 PUBLICATION MAIN EXPERIMENT : COMPLETE"
)

print(
    "Frozen runs :",
    len(
        FINAL_RUN_RESULTS
    ),
)

print(
    "Protocol    :",
    PROTOCOL_HASH,
)

print(
    "Checkpoint  :",
    latest_zip,
)

print(
    "=" * 78
)

try:
    from IPython.display import FileLink, display
    display(
        FileLink(
            str(
                latest_zip
            )
        )
    )
except Exception:
    pass


## Seed-level statistical analysis

The analysis uses training seed as the inferential unit. Paired QTrust comparisons are matched by seed and accompanied by bootstrap confidence intervals, Wilcoxon signed-rank tests, paired rank-biserial effect sizes, and Holm correction within the pre-specified comparison family.

In [ ]:
# The 30 evaluation episodes inside each training seed are
# repeated measurements. Inferential n is therefore the number
# of independent training seeds, not 30 × seeds.

# Primary paired comparisons:
#   QTrust-PPO-128 vs Classical-PPO
#   QTrust-PPO-128 vs Fixed-QPPO-128
#   QTrust-PPO-128 vs Fixed-QPPO-256

# Wilcoxon signed-rank tests are paired by training seed.
# Holm correction is applied within each environment.


from scipy.stats import t, wilcoxon, rankdata
import numpy as np
import pandas as pd

FINAL_RUN_RESULTS = load_completed_summaries()

if len(FINAL_RUN_RESULTS) == 0:
    raise RuntimeError(
        "No completed final runs found. Run the batch runner first."
    )

FINAL_RUN_RESULTS = (
    FINAL_RUN_RESULTS[
        FINAL_RUN_RESULTS["run_id"].isin(
            set(
                FINAL_EXPERIMENT_MANIFEST.apply(
                    lambda row: safe_run_id(
                        row["environment"],
                        row["method"],
                        row["seed"],
                    ),
                    axis=1,
                )
            )
        )
    ]
    .sort_values(
        ["environment", "method", "seed"]
    )
    .reset_index(drop=True)
)

expected_runs = len(FINAL_EXPERIMENT_MANIFEST)

print("=" * 78)
print("FINAL STATISTICAL ANALYSIS")
print("=" * 78)
print(
    f"Completed frozen runs: "
    f"{len(FINAL_RUN_RESULTS)} / {expected_runs}"
)

if len(FINAL_RUN_RESULTS) != expected_runs:
    raise RuntimeError(
        "FINAL STATISTICS BLOCKED: the frozen experiment is incomplete. "
        f"Found {len(FINAL_RUN_RESULTS)} / {expected_runs} runs."
    )

expected_per_group = len(ACTIVE_SEEDS)
group_counts = (
    FINAL_RUN_RESULTS
    .groupby(["environment", "method"])
    .size()
)

if len(group_counts) != (
    len(FINAL_ENVIRONMENTS) * len(FINAL_METHODS)
):
    raise RuntimeError(
        "FINAL STATISTICS BLOCKED: environment × method groups are missing."
    )

if not group_counts.eq(expected_per_group).all():
    raise RuntimeError(
        "FINAL STATISTICS BLOCKED: every environment × method group "
        f"must contain exactly {expected_per_group} independent seeds."
    )

print("Frozen manifest completeness : PASS")


def mean_t_ci(values, confidence=0.95):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    n = len(values)

    if n == 0:
        return np.nan, np.nan, np.nan, np.nan

    mean = float(np.mean(values))

    if n == 1:
        return mean, np.nan, np.nan, np.nan

    sd = float(np.std(values, ddof=1))
    se = sd / np.sqrt(n)
    critical = float(
        t.ppf(
            1.0 - (1.0 - confidence) / 2.0,
            df=n - 1,
        )
    )

    return (
        mean,
        sd,
        mean - critical * se,
        mean + critical * se,
    )


def paired_rank_biserial(x, y):
    """
    Positive value means x tends to exceed y.
    Zero differences are omitted.
    """
    d = np.asarray(x, dtype=float) - np.asarray(y, dtype=float)
    d = d[np.isfinite(d)]
    d = d[d != 0]

    if len(d) == 0:
        return 0.0

    ranks = rankdata(np.abs(d), method="average")
    w_plus = float(np.sum(ranks[d > 0]))
    w_minus = float(np.sum(ranks[d < 0]))
    denominator = w_plus + w_minus

    return (
        (w_plus - w_minus) / denominator
        if denominator > 0
        else 0.0
    )


def paired_bootstrap_mean_difference(
    x,
    y,
    n_boot=20000,
    seed=MASTER_SEED,
):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    mask = np.isfinite(x) & np.isfinite(y)
    d = x[mask] - y[mask]

    if len(d) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)

    indices = rng.integers(
        0,
        len(d),
        size=(n_boot, len(d)),
    )

    boot_means = d[indices].mean(axis=1)

    return (
        float(np.mean(d)),
        float(np.quantile(boot_means, 0.025)),
        float(np.quantile(boot_means, 0.975)),
    )


def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=np.float64)
    m = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(m, dtype=np.float64)

    running_max = 0.0

    for rank_index, original_index in enumerate(order):
        multiplier = m - rank_index
        candidate = min(
            1.0,
            multiplier * p_values[original_index],
        )
        running_max = max(running_max, candidate)
        adjusted[original_index] = running_max

    return adjusted



# Seed-level descriptive summary


summary_rows = []

for (environment, method), group in (
    FINAL_RUN_RESULTS.groupby(
        ["environment", "method"],
        sort=False,
    )
):
    reward_mean, reward_sd, reward_low, reward_high = (
        mean_t_ci(group["mean_eval_reward"])
    )

    kl_mean, kl_sd, kl_low, kl_high = (
        mean_t_ci(group["mean_approx_kl"])
    )

    clip_mean, clip_sd, clip_low, clip_high = (
        mean_t_ci(group["mean_clip_fraction"])
    )

    quantum_rows = group[
        group["training_quantum_shots"] > 0
    ]

    if len(quantum_rows) > 0:
        shots_mean, shots_sd, shots_low, shots_high = (
            mean_t_ci(
                quantum_rows["training_quantum_shots"]
            )
        )
    else:
        shots_mean = shots_sd = shots_low = shots_high = 0.0

    summary_rows.append(
        {
            "Environment": environment,
            "Method": method,
            "Training Seeds": int(len(group)),
            "Mean Reward": reward_mean,
            "Reward SD": reward_sd,
            "Reward CI95 Low": reward_low,
            "Reward CI95 High": reward_high,
            "Mean Approx KL": kl_mean,
            "KL SD": kl_sd,
            "Mean Clip Fraction": clip_mean,
            "Clip SD": clip_sd,
            "Mean Training Quantum Shots": shots_mean,
            "Quantum Shots SD": shots_sd,
        }
    )

MAIN_SUMMARY_TABLE = pd.DataFrame(summary_rows)



# Paired QTrust comparisons


comparison_methods = [
    "Classical-PPO",
    "Fixed-QPPO-128",
    "Fixed-QPPO-256",
]

pairwise_rows = []

for environment in FINAL_ENVIRONMENTS:
    env_df = FINAL_RUN_RESULTS[
        FINAL_RUN_RESULTS["environment"] == environment
    ]

    pivot = env_df.pivot(
        index="seed",
        columns="method",
        values="mean_eval_reward",
    )

    env_row_indices = []
    env_p_values = []

    for comparator in comparison_methods:
        required = {"QTrust-PPO-128", comparator}

        if not required.issubset(pivot.columns):
            continue

        paired = pivot[
            ["QTrust-PPO-128", comparator]
        ].dropna()

        if len(paired) < 2:
            continue

        qtrust_values = paired[
            "QTrust-PPO-128"
        ].to_numpy()

        comparator_values = paired[
            comparator
        ].to_numpy()

        try:
            wilcoxon_result = wilcoxon(
                qtrust_values,
                comparator_values,
                alternative="two-sided",
                zero_method="wilcox",
                method="auto",
            )
            p_value = float(wilcoxon_result.pvalue)
            statistic = float(wilcoxon_result.statistic)
        except ValueError:
            statistic = 0.0
            p_value = 1.0

        mean_diff, diff_low, diff_high = (
            paired_bootstrap_mean_difference(
                qtrust_values,
                comparator_values,
                n_boot=20000,
                seed=(
                    MASTER_SEED
                    + len(pairwise_rows) * 1000
                ),
            )
        )

        effect = paired_rank_biserial(
            qtrust_values,
            comparator_values,
        )

        pairwise_rows.append(
            {
                "Environment": environment,
                "Comparison": (
                    f"QTrust-PPO-128 vs {comparator}"
                ),
                "Paired N": int(len(paired)),
                "QTrust Mean Reward": float(
                    np.mean(qtrust_values)
                ),
                "Comparator Mean Reward": float(
                    np.mean(comparator_values)
                ),
                "Mean Paired Difference": mean_diff,
                "Difference CI95 Low": diff_low,
                "Difference CI95 High": diff_high,
                "Wilcoxon Statistic": statistic,
                "Raw p": p_value,
                "Rank-Biserial": effect,
            }
        )

        env_row_indices.append(
            len(pairwise_rows) - 1
        )
        env_p_values.append(p_value)

    if env_p_values:
        adjusted = holm_adjust(env_p_values)

        for row_idx, adjusted_p in zip(
            env_row_indices,
            adjusted,
        ):
            pairwise_rows[row_idx][
                "Holm-adjusted p"
            ] = float(adjusted_p)

PAIRWISE_STATS_TABLE = pd.DataFrame(pairwise_rows)



# QTrust diagnostic table


qtrust_df = FINAL_RUN_RESULTS[
    FINAL_RUN_RESULTS["method"] == "QTrust-PPO-128"
].copy()

qtrust_diag_rows = []

for environment, group in qtrust_df.groupby(
    "environment",
    sort=False,
):
    qtrust_diag_rows.append(
        {
            "Environment": environment,
            "Training Seeds": int(len(group)),
            "Classification Accuracy": float(
                group["classification_accuracy"].mean()
            ),
            "Mean Unresolved Fraction": float(
                group["mean_unresolved_fraction"].mean()
            ),
            "Mean Adaptive Shots / Ratio": float(
                group["mean_adaptive_shots"].mean()
            ),
            "Accepted Updates": int(
                group["accepted_updates"].sum()
            ),
            "Rolled-back Updates": int(
                group["rolled_back_updates"].sum()
            ),
            "Mean Certification Shots": float(
                group["certification_shots"].mean()
            ),
        }
    )

QTRUST_DIAGNOSTICS_TABLE = pd.DataFrame(
    qtrust_diag_rows
)



# Save statistical outputs


STATS_DIR = RESULT_ROOT / "statistics"
STATS_DIR.mkdir(parents=True, exist_ok=True)

MAIN_SUMMARY_TABLE.to_csv(
    STATS_DIR / "main_summary.csv",
    index=False,
)

PAIRWISE_STATS_TABLE.to_csv(
    STATS_DIR / "paired_statistics.csv",
    index=False,
)

QTRUST_DIAGNOSTICS_TABLE.to_csv(
    STATS_DIR / "qtrust_diagnostics.csv",
    index=False,
)

with open(
    STATS_DIR / "main_summary.tex",
    "w",
) as f:
    f.write(
        MAIN_SUMMARY_TABLE.to_latex(
            index=False,
            float_format="%.4f",
        )
    )

with open(
    STATS_DIR / "paired_statistics.tex",
    "w",
) as f:
    f.write(
        PAIRWISE_STATS_TABLE.to_latex(
            index=False,
            float_format="%.4f",
        )
    )


print()
print("MAIN SUMMARY")
display(MAIN_SUMMARY_TABLE)

print()
print("PAIRED STATISTICS")
display(PAIRWISE_STATS_TABLE)

print()
print("QTRUST DIAGNOSTICS")
display(QTRUST_DIAGNOSTICS_TABLE)

print("=" * 78)
print(
    "Inferential unit: independent training seed."
)
print(
    "Final-paper target: 10 seeds for every environment × method."
)
print("=" * 78)


In [ ]:
# 600-DPI raster + vector PDF.
# No decorative 3-D effects are used because they would distort
# quantitative interpretation; depth is limited to subtle
# marker shadows and layered confidence/error-bar elements.


import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from pathlib import Path

PAPER_DIR = RESULT_ROOT / "publication_outputs"
PAPER_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "mathtext.fontset": "stix",
        "font.size": 11,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "axes.titleweight": "bold",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 140,
        "savefig.dpi": 600,
        "legend.frameon": False,
    }
)

METHOD_ORDER = [
    "Classical-PPO",
    "Fixed-QPPO-128",
    "Fixed-QPPO-256",
    "QTrust-PPO-128",
]

METHOD_COLORS = {
    "Classical-PPO": "#3B4CC0",
    "Fixed-QPPO-128": "#2A9D8F",
    "Fixed-QPPO-256": "#E9C46A",
    "QTrust-PPO-128": "#D1495B",
}

METHOD_MARKERS = {
    "Classical-PPO": "o",
    "Fixed-QPPO-128": "s",
    "Fixed-QPPO-256": "^",
    "QTrust-PPO-128": "D",
}


def save_publication_figure(fig, stem):
    png_path = PAPER_DIR / f"{stem}.png"
    pdf_path = PAPER_DIR / f"{stem}.pdf"

    fig.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white",
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(fig)

    return png_path, pdf_path



# Figure 1: seed-level reward with 95% t-CI
# One clean figure per environment.


for environment in FINAL_ENVIRONMENTS:

    plot_df = MAIN_SUMMARY_TABLE[
        MAIN_SUMMARY_TABLE["Environment"] == environment
    ].copy()

    plot_df["Method"] = pd.Categorical(
        plot_df["Method"],
        categories=METHOD_ORDER,
        ordered=True,
    )

    plot_df = plot_df.sort_values(
        "Method"
    )

    fig, ax = plt.subplots(
        figsize=(7.4, 4.8)
    )

    x = np.arange(
        len(plot_df)
    )

    means = plot_df[
        "Mean Reward"
    ].to_numpy()

    lower = (
        means
        - plot_df[
            "Reward CI95 Low"
        ].to_numpy()
    )

    upper = (
        plot_df[
            "Reward CI95 High"
        ].to_numpy()
        - means
    )

    for i, (_, row) in enumerate(
        plot_df.reset_index(drop=True).iterrows()
    ):
        method = str(
            row["Method"]
        )

        mean_value = float(
            row["Mean Reward"]
        )
        lower_error = float(
            mean_value
            - row["Reward CI95 Low"]
        )
        upper_error = float(
            row["Reward CI95 High"]
            - mean_value
        )

        container = ax.errorbar(
            i,
            mean_value,
            yerr=np.array(
                [
                    [lower_error],
                    [upper_error],
                ]
            ),
            fmt=METHOD_MARKERS[
                method
            ],
            markersize=9,
            linewidth=2.0,
            capsize=5,
            color=METHOD_COLORS[
                method
            ],
            markeredgecolor="white",
            markeredgewidth=0.9,
        )

        # Subtle depth without altering data geometry.
        try:
            container[0].set_path_effects(
                [
                    pe.SimpleLineShadow(
                        offset=(1.0, -1.0),
                        alpha=0.18,
                    ),
                    pe.Normal(),
                ]
            )
        except Exception:
            pass

    ax.set_xticks(
        x,
        [
            str(m)
            for m in plot_df[
                "Method"
            ]
        ],
        rotation=18,
        ha="right",
    )

    ax.set_ylabel(
        "Evaluation return"
    )

    ax.set_title(
        f"{environment}: seed-level performance"
    )

    ax.grid(
        axis="y",
        alpha=0.18,
        linewidth=0.8,
    )

    save_publication_figure(
        fig,
        "Fig_reward_"
        + environment.replace(
            "-",
            "_"
        ),
    )



# Figure 2: resource–performance plane for quantum methods


quantum_methods = [
    "Fixed-QPPO-128",
    "Fixed-QPPO-256",
    "QTrust-PPO-128",
]

for environment in FINAL_ENVIRONMENTS:

    env_df = FINAL_RUN_RESULTS[
        (
            FINAL_RUN_RESULTS[
                "environment"
            ]
            == environment
        )
        &
        (
            FINAL_RUN_RESULTS[
                "method"
            ].isin(
                quantum_methods
            )
        )
    ]

    fig, ax = plt.subplots(
        figsize=(7.4, 4.8)
    )

    for method in quantum_methods:

        group = env_df[
            env_df[
                "method"
            ]
            == method
        ]

        if len(group) == 0:
            continue

        x = group[
            "training_quantum_shots"
        ].to_numpy(
            dtype=float
        )

        y = group[
            "mean_eval_reward"
        ].to_numpy(
            dtype=float
        )

        ax.scatter(
            x,
            y,
            s=65,
            alpha=0.50,
            marker=METHOD_MARKERS[
                method
            ],
            color=METHOD_COLORS[
                method
            ],
            edgecolor="white",
            linewidth=0.7,
        )

        ax.scatter(
            [np.mean(x)],
            [np.mean(y)],
            s=145,
            marker=METHOD_MARKERS[
                method
            ],
            color=METHOD_COLORS[
                method
            ],
            edgecolor="black",
            linewidth=0.8,
            label=method,
            zorder=4,
        )

    ax.set_xscale(
        "log"
    )

    ax.set_xlabel(
        "Training quantum shots (log scale)"
    )

    ax.set_ylabel(
        "Evaluation return"
    )

    ax.set_title(
        f"{environment}: quantum resource–performance plane"
    )

    ax.grid(
        alpha=0.18,
        linewidth=0.8,
    )

    ax.legend(
        loc="best"
    )

    save_publication_figure(
        fig,
        "Fig_resource_"
        + environment.replace(
            "-",
            "_"
        ),
    )



# Figure 3: QTrust reliability across environments


if len(
    QTRUST_DIAGNOSTICS_TABLE
) > 0:

    fig, ax = plt.subplots(
        figsize=(7.4, 4.8)
    )

    x = np.arange(
        len(
            QTRUST_DIAGNOSTICS_TABLE
        )
    )

    accuracy = (
        QTRUST_DIAGNOSTICS_TABLE[
            "Classification Accuracy"
        ].to_numpy(
            dtype=float
        )
    )

    unresolved = (
        QTRUST_DIAGNOSTICS_TABLE[
            "Mean Unresolved Fraction"
        ].to_numpy(
            dtype=float
        )
    )

    width = 0.34

    ax.bar(
        x - width / 2,
        accuracy,
        width,
        label="Resolved classification accuracy",
        color="#355070",
        alpha=0.92,
    )

    ax.bar(
        x + width / 2,
        unresolved,
        width,
        label="Unresolved fraction",
        color="#EAAC8B",
        alpha=0.92,
    )

    ax.set_xticks(
        x,
        QTRUST_DIAGNOSTICS_TABLE[
            "Environment"
        ],
        rotation=12,
    )

    ax.set_ylim(
        0.0,
        1.05,
    )

    ax.set_ylabel(
        "Fraction"
    )

    ax.set_title(
        "QTrust-PPO certification reliability"
    )

    ax.grid(
        axis="y",
        alpha=0.18,
    )

    ax.legend(
        loc="best"
    )

    save_publication_figure(
        fig,
        "Fig_qtrust_reliability",
    )



# Figure 4: controlled boundary benchmark


boundary_path = (
    WORK_ROOT
    / "qtrust_boundary_reliability.csv"
)

if boundary_path.exists():

    boundary_df = pd.read_csv(
        boundary_path
    )

    fig, ax = plt.subplots(
        figsize=(7.4, 4.8)
    )

    fixed = boundary_df[
        boundary_df[
            "Method"
        ].str.startswith(
            "Fixed-"
        )
    ].copy()

    qtrust = boundary_df[
        boundary_df[
            "Method"
        ]
        == "QTrust"
    ].copy()

    ax.plot(
        fixed[
            "Mean Shots / Ratio"
        ],
        fixed[
            "Resolved Accuracy"
        ],
        marker="o",
        linewidth=2.0,
        label="Fixed-shot plug-in",
        color="#2A9D8F",
    )

    if len(qtrust) == 1:

        ax.scatter(
            qtrust[
                "Mean Shots / Ratio"
            ],
            qtrust[
                "Resolved Accuracy"
            ],
            s=150,
            marker="D",
            label=(
                "QTrust resolved accuracy"
            ),
            color="#D1495B",
            edgecolor="black",
            linewidth=0.8,
            zorder=4,
        )

        ax.scatter(
            qtrust[
                "Mean Shots / Ratio"
            ],
            qtrust[
                "Resolved Coverage"
            ],
            s=115,
            marker="X",
            label=(
                "QTrust resolved coverage"
            ),
            color="#6D597A",
            edgecolor="black",
            linewidth=0.7,
            zorder=4,
        )

    ax.set_xscale(
        "log",
        base=2,
    )

    ax.set_ylim(
        0.0,
        1.03,
    )

    ax.set_xlabel(
        "Mean probability-measurement shots / ratio"
    )

    ax.set_ylabel(
        "Fraction"
    )

    ax.set_title(
        "Near-boundary clipping reliability"
    )

    ax.grid(
        alpha=0.18,
    )

    ax.legend(
        loc="best"
    )

    save_publication_figure(
        fig,
        "Fig_boundary_reliability",
    )






MAIN_SUMMARY_TABLE.to_csv(
    PAPER_DIR / "Table_main_summary.csv",
    index=False,
)

PAIRWISE_STATS_TABLE.to_csv(
    PAPER_DIR / "Table_pairwise_statistics.csv",
    index=False,
)

QTRUST_DIAGNOSTICS_TABLE.to_csv(
    PAPER_DIR / "Table_qtrust_diagnostics.csv",
    index=False,
)


print("=" * 78)
print("PUBLICATION OUTPUTS CREATED")
print("=" * 78)
print(PAPER_DIR)
print(
    "Figures: 600-DPI PNG + vector PDF"
)
print(
    "Tables: CSV + LaTeX statistics exports"
)
print("=" * 78)


In [ ]:
import zipfile

FINAL_BUNDLE = (
    WORK_ROOT
    / "QTrust_PPO_Final_Paper_Artifacts.zip"
)

with zipfile.ZipFile(
    FINAL_BUNDLE,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    
    for file in RESULT_ROOT.rglob("*"):
        if (
            file.is_file()
            and "models" not in file.parts
        ):
            zf.write(
                file,
                arcname=str(
                    file.relative_to(
                        WORK_ROOT
                    )
                ),
            )

    boundary_file = (
        WORK_ROOT
        / "qtrust_boundary_reliability.csv"
    )

    if boundary_file.exists():
        zf.write(
            boundary_file,
            arcname=
                boundary_file.name,
        )

print("=" * 78)
print("FINAL PAPER ARTIFACT BUNDLE")
print("=" * 78)
print(FINAL_BUNDLE)
print("=" * 78)






assert len(FINAL_RUN_RESULTS) == len(FINAL_EXPERIMENT_MANIFEST)

print()
print("=" * 78)
print("QTRUST-PPO PUBLICATION PIPELINE : COMPLETE")
print("Main RL runs                    :", len(FINAL_RUN_RESULTS))
print("Final artifact bundle           :", FINAL_BUNDLE)
print("=" * 78)

try:
    from IPython.display import FileLink, display
    display(FileLink(str(FINAL_BUNDLE)))
except Exception:
    pass
